# Modelo de segmentación según modalidad laboral

## *0. Configuración y parámetros del experimento*

Este cuaderno implementa la rama conductual-individualizada del pipeline
metodológico propuesto en la tesis: activa los cinco módulos y recorre la
cadena completa desde trazas de telefonía móvil hasta segmentos de modalidad
laboral.

**Fuentes.** El pipeline no exige un esquema de origen particular: exige que la
fuente sea transformable a la representación canónica declarada en la Tabla 3.1
del manuscrito (`user_id_anon`, `event_ts`, `lat_obs`, `lon_obs`). Cualquier
fuente de telefonía móvil que contenga identificador seudonimizado, marca
temporal y coordenadas puede alimentar el procedimiento declarando sus nombres
de campo en `ExperimentConfig`. La fuente utilizada en el proyecto es
confidencial y no forma parte de esta entrega.

**Criterio de publicación.** Se publica el procedimiento completo; no se
publican los artefactos ni las salidas derivadas de la fuente telco. Por el
mismo criterio, el cuaderno se entrega sin resultados de ejecución: los valores
de la corrida documentada figuran en el manuscrito de tesis.

In [ ]:
# ============================================================
# 0 — Setup y control experimental
# ============================================================
from __future__ import annotations

# --- Standard library ---
import gc
import json
import os
import platform
import random
import re
import sys
import uuid
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, time, timezone
from itertools import combinations
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple

# --- Third-party ---
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pandas.api.types import CategoricalDtype
from pyogrio import list_layers, read_info
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp
from shapely.ops import nearest_points
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

In [ ]:
# ============================================================
# 0.0 — Fuentes externas y directorio de salidas
# ============================================================
# Las fuentes externas (telco, DPA 2023, OSM) se ubican fuera del árbol
# versionado, en un directorio que el usuario crea en su propio entorno.
# Su ubicación puede definirse mediante la variable de entorno
# MDSPML_DATA_DIR o editando el valor por defecto.
EXTERNAL_DATA_DIR = Path(
    os.environ.get("MDSPML_DATA_DIR", Path.cwd() / "external_data")
).expanduser().resolve()

# Directorio de artefactos de la corrida. Todo lo que el cuaderno escribe
# —parquets intermedios, figuras y logs .jsonl— deriva de la fuente telco y no
# forma parte de esta entrega: el repositorio se publicó sin ese directorio.
OUTPUTS_DIR = (Path.cwd() / "artifacts").resolve()

print(f"EXTERNAL_DATA_DIR = {EXTERNAL_DATA_DIR}")
print(f"OUTPUTS_DIR       = {OUTPUTS_DIR}")

In [ ]:
# ============================================================
# 0.1 — Utils: tiempo, memoria, logging, seeds
# ============================================================
def now_utc() -> datetime:
    return datetime.now(timezone.utc)


def mem_mb_df(df: Optional[pd.DataFrame]) -> float:
    if df is None:
        return 0.0
    try:
        return float(df.memory_usage(deep=True).sum()) / (1024.0**2)
    except Exception:
        return float("nan")


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def log_jsonl(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")


def ensure_run_dirs(outputs_dir: Path, run_id: str) -> Dict[str, Path]:
    run_dir = outputs_dir / run_id
    paths = {
        "run_dir": run_dir,
        "logs_dir": run_dir / "logs",
        "tables_dir": run_dir / "tables",
        "figures_dir": run_dir / "figures",
        "cache_dir": run_dir / "cache",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths

In [ ]:
# ============================================================
# 0.2 — Configuración del experimento (Baseline vs Augmented)
# ============================================================
Variant = Literal["baseline", "augmented"]


def make_run_id(prefix: str = "segm") -> str:
    ts = now_utc().strftime("%Y%m%d_%H%M%S")
    short = uuid.uuid4().hex[:8]
    return f"{prefix}_{ts}_{short}"


@dataclass(frozen=True)
class ExperimentConfig:
    # Identidad corrida
    variant: Variant
    run_id: str
    seed: int

    # Paths
    telco_parquet_dir: Path
    dpa_comunal_path: Path
    outputs_dir: Path

    # Augmented
    osm_pbf_path: Optional[Path] = None

    # --- Esquema canónico del pipeline (Módulo I: estandarización) ---
    # Nombres de campo declarados en la Tabla 3.1 del manuscrito. El pipeline
    # no exige un esquema de origen particular: exige que la fuente sea
    # transformable a esta representación. Si su fuente emplea otros nombres,
    # reemplácelos aquí.
    col_user: str = "user_id_anon"
    col_ts: str = "event_ts"
    col_lat: str = "lat_obs"
    col_lon: str = "lon_obs"

    # (RAM) Columnas a leer desde los .parquet. None => leer todo.
    # Puede acotarse a los nombres de campo efectivos de la fuente propia.
    telco_read_columns: Optional[Tuple[str, ...]] = None

    # Identificadores excluidos por volumen y estructura incompatibles con
    # movilidad humana ordinaria. La REGLA pertenece al pipeline; los VALORES
    # dependen de cada fuente, de su período de observación y de la política de
    # captura del proveedor. Esta lista se entrega vacía: complétela a partir
    # del diagnóstico de la distribución de registros por usuario de su propia
    # fuente, y documente la decisión.
    anomalous_user_ids: Tuple[str, ...] = ()

    # Horario confiable
    valid_time_start: time = time(7, 0, 0)
    valid_time_end: time = time(23, 0, 0)  # exclusivo

    # Evidencia observacional basal mínima por usuario.
    # Un usuario con una sola observación no permite construir permanencias
    # ni transiciones: queda fuera del registro depurado (Módulo I).
    min_records_per_user: int = 2

    # H3
    h3_res_main: int = 9
    h3_res_parent: int = 8

    # Stays / Moves
    stay_min_minutes: int = 15
    stay_min_pings: int = 2
    gap_max_minutes: int = 60
    move_max_minutes: int = 60

    # Gatekeeper
    gk_min_days_with_stays: int = 5
    gk_min_weekdays_with_stays: int = 3
    gk_min_total_hours: float = 10.0
    gk_min_span_days: int = 7

    # Home proxy
    home_win_primary: Tuple[time, time] = (time(20, 0, 0), time(23, 0, 0))
    home_win_secondary: Tuple[time, time] = (time(7, 0, 0), time(9, 0, 0))
    home_alpha: float = 0.7
    home_min_days: int = 2
    home_min_ratio: float = 0.30

    # Worksite
    work_win: Tuple[time, time] = (time(9, 0, 0), time(17, 0, 0))
    work_min_days: int = 3
    work_min_ratio: float = 0.20

    # Segmentación
    tau_remote: float = 0.60
    mobile_tau_unique: int = 15
    mobile_percentile_p: float = 75.0

    # --- Clustering (Sección 9) ---
    # Número de conglomerados de la configuración base. El diagnóstico del
    # rango [k_min_diag, k_max_diag] se reporta pero no selecciona k: la
    # elección es metodológica y corresponde a las tres modalidades laborales
    # que el prototipo busca distinguir.
    k_baseline: int = 3
    k_min_diag: int = 2
    k_max_diag: int = 8

    # Umbral por sobre el cual el enlace jerárquico completo deja de ser
    # tratable y se recurre a agrupamiento con restricción de conectividad kNN.
    ward_full_max_n: int = 8000
    ward_sample_n_for_diag: int = 5000
    ward_knn_neighbors: int = 15

    # Tamaño máximo de la submuestra empleada para el coeficiente de silueta.
    silhouette_sample_n: int = 4000

    # --- Robustez y sensibilidad (Sección 13) ---
    # Número de réplicas de submuestreo para evaluar estabilidad del
    # agrupamiento, y fracción retenida en cada una.
    rob_n_seeds: int = 10
    rob_subsample_frac: float = 0.80

    # QA / Snapping / Handover
    snap_max_distance_m: float = 200.0
    handover_max_distance_m: float = 300.0

    # Control de carga (debug)
    max_parquet_files: Optional[int] = None
    max_rows: Optional[int] = None

    def validate(self) -> None:
        if self.variant == "augmented":
            if self.osm_pbf_path is None:
                raise ValueError("variant='augmented' requiere osm_pbf_path.")
            if not self.osm_pbf_path.exists():
                raise FileNotFoundError(
                    "No se encontró el extracto OSM de Chile (chile.osm.pbf).\n"
                    f"  Ruta esperada: {self.osm_pbf_path}\n"
                    "  Es una fuente pública. Descárguela y ubíquela en la ruta "
                    "indicada, o ajuste MDSPML_DATA_DIR."
                )

        if not self.telco_parquet_dir.exists():
            raise FileNotFoundError(
                "No se encontró el directorio de la fuente telco.\n"
                f"  Ruta esperada: {self.telco_parquet_dir}\n"
                "  Es una fuente de telefonía móvil provista por el usuario y no "
                "forma parte de esta entrega. El procedimiento opera sobre "
                "cualquier fuente que contenga identificador seudonimizado, "
                "marca temporal y coordenadas; declare sus nombres de campo en "
                "col_user / col_ts / col_lat / col_lon."
            )

        if not self.dpa_comunal_path.exists():
            raise FileNotFoundError(
                "No se encontró la cartografía DPA 2023 (capa comunal).\n"
                f"  Ruta esperada: {self.dpa_comunal_path}\n"
                "  Es una fuente pública. Descárguela y ubíquela en la ruta "
                "indicada, o ajuste MDSPML_DATA_DIR."
            )

        # Columnas mínimas, solo si se acotó la lectura
        if self.telco_read_columns is not None:
            req = {self.col_user, self.col_ts, self.col_lat, self.col_lon}
            miss = req - set(self.telco_read_columns)
            if miss:
                raise ValueError(
                    "telco_read_columns debe incluir las columnas mínimas del "
                    f"esquema canónico: faltan {miss}"
                )

    def describe(self) -> str:
        """Resumen legible de la configuración activa y de la disponibilidad
        de cada fuente."""
        osm_state = "n/a"
        if self.osm_pbf_path is not None:
            osm_state = "sí" if self.osm_pbf_path.exists() else "no"

        lineas = [
            "MdSpML — configuración activa",
            f"  run_id   : {self.run_id}",
            f"  variant  : {self.variant}",
            f"  seed     : {self.seed}",
            "",
            "  Disponibilidad de fuentes:",
            f"    Telco (usuario)  : {'sí' if self.telco_parquet_dir.exists() else 'no'}",
            f"    DPA 2023         : {'sí' if self.dpa_comunal_path.exists() else 'no'}",
            f"    OSM .pbf         : {osm_state}",
            "",
            "  Esquema canónico   : "
            f"{self.col_user}, {self.col_ts}, {self.col_lat}, {self.col_lon}",
            f"  Identificadores excluidos : {len(self.anomalous_user_ids)}",
            "",
            f"  Ventana horaria    : [{self.valid_time_start.strftime('%H:%M')}, "
            f"{self.valid_time_end.strftime('%H:%M')})",
            f"  Registros mínimos/usuario : {self.min_records_per_user}",
            f"  H3 principal / auxiliar   : {self.h3_res_main} / {self.h3_res_parent}",
            f"  Umbral de colisiones      : {self.handover_max_distance_m:.0f} m",
            f"  Tolerancia de snap        : {self.snap_max_distance_m:.0f} m",
            f"  Gatekeeper                : días>={self.gk_min_days_with_stays}, "
            f"hábiles>={self.gk_min_weekdays_with_stays}, "
            f"horas>={self.gk_min_total_hours:.0f}, "
            f"span>={self.gk_min_span_days}",
            f"  tau_remote                : {self.tau_remote}",
            f"  Clustering                : k={self.k_baseline}, "
            f"diagnóstico k∈[{self.k_min_diag}, {self.k_max_diag}]",
            f"  Robustez                  : {self.rob_n_seeds} réplicas, "
            f"submuestreo {self.rob_subsample_frac:.0%}"
        ]
        return "\n".join(lineas)


# ===== Instanciación =====
# Las rutas derivan de EXTERNAL_DATA_DIR (ver 0.0). Ajuste los subdirectorios
# si su organización de fuentes difiere.
cfg = ExperimentConfig(
    variant="augmented",
    run_id=make_run_id("segm"),
    seed=42,
    telco_parquet_dir=EXTERNAL_DATA_DIR / "telco",
    dpa_comunal_path=EXTERNAL_DATA_DIR / "dpa2023" / "COMUNAS" / "COMUNAS_v1.shp",
    outputs_dir=OUTPUTS_DIR,
    osm_pbf_path=EXTERNAL_DATA_DIR / "osm" / "chile.osm.pbf",
    telco_read_columns=None,
    max_parquet_files=None,
    max_rows=None,
)

cfg.validate()
set_global_seed(cfg.seed)
paths = ensure_run_dirs(cfg.outputs_dir, cfg.run_id)
print("Artifacts run_dir:", paths["run_dir"])

# logging inicial: config + entorno
run_config_path = paths["logs_dir"] / "run_config.jsonl"
env_path = paths["logs_dir"] / "env.jsonl"

log_jsonl(run_config_path, {"event": "config", "ts_utc": now_utc(), **asdict(cfg)})
log_jsonl(
    env_path,
    {
        "event": "env",
        "ts_utc": now_utc(),
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
)

In [ ]:
# ============================================================
# 0.3 — Lectura TELCO (parquet) con control de carga (RAM-first)
# ============================================================
def list_parquet_files(parquet_dir: Path, max_files: Optional[int] = None) -> List[Path]:
    # intento directo: *.parquet
    files = sorted(parquet_dir.glob("*.parquet"))
    # fallback recursivo (si hay subcarpetas)
    if not files:
        files = sorted(parquet_dir.rglob("*.parquet"))
    if max_files is not None:
        files = files[:max_files]
    if not files:
        raise FileNotFoundError(f"No se encontraron .parquet en: {parquet_dir}")
    return files


def read_telco_parquets(cfg: ExperimentConfig, paths: Dict[str, Path]) -> pd.DataFrame:
    io_log = paths["logs_dir"] / "io.jsonl"
    files = list_parquet_files(cfg.telco_parquet_dir, cfg.max_parquet_files)
    log_jsonl(io_log, {"event": "parquet_list", "ts_utc": now_utc(), "n_files": len(files)})

    required = {cfg.col_user, cfg.col_ts, cfg.col_lat, cfg.col_lon}
    dfs: List[pd.DataFrame] = []
    rows_acc = 0
    cols = list(cfg.telco_read_columns) if cfg.telco_read_columns else None

    for i, fp in enumerate(files, start=1):
        df = pd.read_parquet(fp, engine="pyarrow", columns=cols)

        # Verificación del esquema canónico (por archivo; falla temprano)
        miss = required - set(df.columns)
        if miss:
            raise ValueError(
                f"Archivo {fp.name}: faltan columnas del esquema canónico: {miss}.\n"
                f"  Columnas disponibles: {list(df.columns)}\n"
                "  Declare los nombres de campo de su fuente en "
                "cfg.col_user / col_ts / col_lat / col_lon."
            )

        dfs.append(df)
        rows_acc += len(df)

        if (i % 5 == 0) or (i == len(files)):
            log_jsonl(
                io_log,
                {
                    "event": "parquet_progress",
                    "ts_utc": now_utc(),
                    "files_read": i,
                    "rows_acc": int(rows_acc),
                    "last_file": fp.name,
                    "mem_mb_last_df": mem_mb_df(df),
                },
            )

        # stop temprano si max_rows se alcanzó (RAM + tiempo)
        if cfg.max_rows is not None and rows_acc >= cfg.max_rows:
            break

    df_total = pd.concat(dfs, ignore_index=True)
    del dfs
    gc.collect()

    if cfg.max_rows is not None and len(df_total) > cfg.max_rows:
        df_total = df_total.iloc[: cfg.max_rows]

    log_jsonl(
        io_log,
        {
            "event": "telco_loaded",
            "ts_utc": now_utc(),
            "rows": int(df_total.shape[0]),
            "cols": int(df_total.shape[1]),
            "mem_mb_df_total": mem_mb_df(df_total),
            "n_files_used": len(files),
        },
    )
    return df_total


df_total = read_telco_parquets(cfg, paths)
print("df_total:", df_total.shape, f"| mem={mem_mb_df(df_total):.1f} MB")
print("Columnas leídas:", list(df_total.columns))

In [ ]:
# ============================================================
# 0.4 — Resumen de configuración y disponibilidad de fuentes
# ============================================================
print(cfg.describe())

## *1. Limpieza de trazas móviles*

### **1.1 Tratamiento de duplicados**

In [ ]:
# ============================================================
# 1.1 — Utils: Distancia geodésica WGS84 (vectorizada, metros)
# ============================================================
from pyproj import Geod

# Objeto reutilizable para resolver el problema geodésico inverso
# sobre el elipsoide WGS84.
GEOD_WGS84 = Geod(ellps="WGS84")


def geodesic_wgs84_m(lat1, lon1, lat2, lon2) -> np.ndarray:
    """
    Calcula la distancia geodésica elipsoidal WGS84 entre pares
    de coordenadas geográficas.

    Parámetros
    ----------
    lat1, lon1, lat2, lon2:
        Escalares o arreglos de coordenadas expresadas en grados
        decimales. Pueden ser float32 o float64.

    Retorna
    -------
    np.ndarray
        Distancias geodésicas en metros.

    Notas
    -----
    pyproj.Geod.inv recibe las coordenadas en orden:
    longitud, latitud, longitud, latitud.
    Internamente, las coordenadas se convierten a float64.
    """
    lat1 = np.asarray(lat1, dtype=np.float64)
    lon1 = np.asarray(lon1, dtype=np.float64)
    lat2 = np.asarray(lat2, dtype=np.float64)
    lon2 = np.asarray(lon2, dtype=np.float64)

    _, _, distance_m = GEOD_WGS84.inv(
        lon1,
        lat1,
        lon2,
        lat2,
    )
    return np.abs(
        np.asarray(distance_m, dtype=np.float64)
    )

# ============================================================
# 1.1 — Deduplicación RAM-safe: exactos + anómalos + (u,t)
# ============================================================
def preprocess_deduplicates(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Limpieza de duplicados (Módulo I):

      1) Elimina duplicados exactos.
      2) Excluye los identificadores declarados en cfg.anomalous_user_ids.
      3) Resuelve colisiones en el par (usuario, timestamp):
           - grupos con size != 2: se eliminan completamente;
           - grupos con size == 2:
               * distancia geodésica WGS84 <= umbral:
                 se fusionan mediante promedio de latitud y longitud;
               * distancia geodésica WGS84 > umbral:
                 se eliminan ambos registros.

    Además, registra el número de usuarios únicos antes y después
    del tratamiento.

    La exclusión de identificadores anómalos es una REGLA del pipeline;
    los VALORES concretos dependen de cada fuente y deben justificarse a
    partir de su distribución empírica de registros por usuario
    (ver cfg.anomalous_user_ids).

    La distancia corresponde a la geodésica elipsoidal WGS84,
    equivalente a la definición utilizada en Deep Gravity Chile.
    """
    user_col = cfg.col_user
    ts_col = cfg.col_ts
    lat_col = cfg.col_lat
    lon_col = cfg.col_lon

    thr_m = float(cfg.handover_max_distance_m)
    key_cols = [user_col, ts_col]

    anomalous_ids = tuple(getattr(cfg, "anomalous_user_ids", ()) or ())

    required = {
        user_col,
        ts_col,
        lat_col,
        lon_col,
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Faltan columnas requeridas: {missing}"
        )

    log_path = paths["logs_dir"] / "cleaning.jsonl"

    # Usuarios únicos antes de cualquier tratamiento
    users_before = int(
        df[user_col].nunique(dropna=True)
    )

    report: Dict[str, Any] = {
        "event": "dedup",
        "ts_utc": now_utc(),
        "user_column": user_col,
        "thr_handover_m": thr_m,
        # Especificación reproducible de la distancia
        "distance_method": "ellipsoidal_geodesic",
        "distance_implementation": "pyproj.Geod.inv",
        "distance_ellipsoid": "WGS84",
        "coordinate_crs": "EPSG:4326",
        "threshold_operator_merge": "<=",
        "fusion_method": "arithmetic_mean_lat_lon",
        # Estado inicial
        "rows_before": int(len(df)),
        "users_before": users_before,
        "mem_mb_before": mem_mb_df(df),
    }

    def update_final_counts(
        current_df: pd.DataFrame,
    ) -> None:
        """
        Actualiza el reporte con los conteos finales de registros
        y usuarios únicos.
        """
        users_after = int(
            current_df[user_col].nunique(dropna=True)
        )
        users_removed = int(
            users_before - users_after
        )

        report["rows_after"] = int(
            len(current_df)
        )
        report["users_after"] = users_after
        report["users_removed"] = users_removed
        report["users_removed_pct"] = (
            float(users_removed / users_before * 100.0)
            if users_before > 0
            else 0.0
        )
        report["mem_mb_after"] = mem_mb_df(
            current_df
        )

    out = df

    # --------------------------------------------------------
    # Validación y normalización mínima de tipos
    # --------------------------------------------------------
    if not pd.api.types.is_datetime64_any_dtype(
        out[ts_col]
    ):
        out = out.copy()
        out[ts_col] = pd.to_datetime(
            out[ts_col],
            errors="coerce",
        )

    if out[ts_col].isna().any():
        raise ValueError(
            f"Timestamps inválidos tras coerción "
            f"en '{ts_col}'."
        )

    if (
        not pd.api.types.is_numeric_dtype(
            out[lat_col]
        )
        or not pd.api.types.is_numeric_dtype(
            out[lon_col]
        )
    ):
        out = out.copy()
        out[lat_col] = pd.to_numeric(
            out[lat_col],
            errors="coerce",
        )
        out[lon_col] = pd.to_numeric(
            out[lon_col],
            errors="coerce",
        )

    if (
        out[lat_col].isna().any()
        or out[lon_col].isna().any()
    ):
        raise ValueError(
            "Se encontraron latitudes o longitudes NaN "
            "tras la coerción numérica."
        )

    # Validar dominio de coordenadas geográficas
    invalid_lat = ~out[lat_col].between(
        -90.0,
        90.0,
    )
    invalid_lon = ~out[lon_col].between(
        -180.0,
        180.0,
    )
    if invalid_lat.any() or invalid_lon.any():
        raise ValueError(
            "Se encontraron coordenadas fuera del dominio "
            "geográfico: "
            f"latitudes inválidas="
            f"{int(invalid_lat.sum()):,}, "
            f"longitudes inválidas="
            f"{int(invalid_lon.sum()):,}."
        )

    # --------------------------------------------------------
    # 1) Duplicados exactos
    # --------------------------------------------------------
    n0 = len(out)
    out = out.drop_duplicates()
    report["dropped_exact"] = int(
        n0 - len(out)
    )

    # --------------------------------------------------------
    # 2) Identificadores anómalos declarados
    # --------------------------------------------------------
    report["anomalous_user_ids_n"] = int(len(anomalous_ids))

    if anomalous_ids:
        n1 = len(out)
        out = out[
            ~out[user_col].isin(anomalous_ids)
        ]
        report["dropped_anomalous_rows"] = int(
            n1 - len(out)
        )
    else:
        report["dropped_anomalous_rows"] = 0
        if verbose:
            print(
                "[Dedup] cfg.anomalous_user_ids está vacío: no se excluyó "
                "ningún identificador.\n"
                "        Inspeccione la distribución de registros por usuario "
                "y complete la lista si corresponde."
            )

    # --------------------------------------------------------
    # 3) Duplicados por (usuario, timestamp)
    # --------------------------------------------------------
    dupl_mask = out.duplicated(
        subset=key_cols,
        keep=False,
    )
    n_dupl_rows = int(
        dupl_mask.sum()
    )
    report["rows_flagged_ut_dup"] = (
        n_dupl_rows
    )

    # --------------------------------------------------------
    # Caso sin colisiones usuario-timestamp
    # --------------------------------------------------------
    if n_dupl_rows == 0:
        out = out.reset_index(drop=True)
        report.update(
            merged_pairs=0,
            dropped_ut_invalid_rows=0,
            dropped_ut_pairs_far=0,
            dropped_ut_far_rows=0,
        )
        update_final_counts(out)
        log_jsonl(
            log_path,
            report,
        )

        if verbose:
            print(
                f"[Dedup] dropped_exact="
                f"{report['dropped_exact']:,}"
            )
            print(
                f"[Dedup] dropped_anomalous_rows="
                f"{report['dropped_anomalous_rows']:,}"
            )
            print(
                "[Dedup] "
                "rows_flagged_ut_dup=0"
            )
            print(
                f"[Dedup] users_before="
                f"{report['users_before']:,} | "
                f"users_after="
                f"{report['users_after']:,} | "
                f"users_removed="
                f"{report['users_removed']:,}"
            )
            print(
                f"[Dedup] shape_after={out.shape} | "
                f"mem={report['mem_mb_after']:.1f} MB"
            )

        return out

    # --------------------------------------------------------
    # Trabajar únicamente con el subconjunto duplicado
    # --------------------------------------------------------
    dup_idx = out.index[
        dupl_mask
    ]
    dup_min = out.loc[
        dup_idx,
        key_cols + [lat_col, lon_col],
    ]

    # --------------------------------------------------------
    # 3.a) Eliminar grupos cuya cardinalidad sea distinta de 2
    # --------------------------------------------------------
    grp_size = (
        dup_min
        .groupby(
            key_cols,
            sort=False,
        )[lat_col]
        .transform("size")
    )
    invalid_dup_idx = dup_min.index[
        grp_size != 2
    ]
    report["dropped_ut_invalid_rows"] = int(
        len(invalid_dup_idx)
    )

    if len(invalid_dup_idx) > 0:
        out = out.drop(
            index=invalid_dup_idx
        )

    del invalid_dup_idx
    gc.collect()

    # --------------------------------------------------------
    # Recalcular duplicados tras eliminar grupos inválidos
    # --------------------------------------------------------
    dupl_mask2 = out.duplicated(
        subset=key_cols,
        keep=False,
    )

    if int(dupl_mask2.sum()) == 0:
        out = out.reset_index(drop=True)
        report.update(
            merged_pairs=0,
            dropped_ut_pairs_far=0,
            dropped_ut_far_rows=0,
        )
        update_final_counts(out)
        log_jsonl(
            log_path,
            report,
        )

        if verbose:
            print(
                f"[Dedup] dropped_exact="
                f"{report['dropped_exact']:,}"
            )
            print(
                f"[Dedup] dropped_anomalous_rows="
                f"{report['dropped_anomalous_rows']:,}"
            )
            print(
                "[Dedup] "
                "dropped_ut_invalid_rows(size!=2)="
                f"{report['dropped_ut_invalid_rows']:,}"
            )
            print(
                f"[Dedup] users_before="
                f"{report['users_before']:,} | "
                f"users_after="
                f"{report['users_after']:,} | "
                f"users_removed="
                f"{report['users_removed']:,}"
            )
            print(
                f"[Dedup] ut_dups_after=0 | "
                f"shape_after={out.shape} | "
                f"mem={report['mem_mb_after']:.1f} MB"
            )

        return out

    dup2_idx = out.index[
        dupl_mask2
    ]
    dup2_min = out.loc[
        dup2_idx,
        key_cols + [lat_col, lon_col],
    ]

    # Orden estable por (usuario, timestamp).
    # Como todos los grupos restantes tienen size=2,
    # cada par queda en filas consecutivas.
    dup2_sorted = dup2_min.sort_values(
        key_cols,
        kind="mergesort",
    )

    if len(dup2_sorted) % 2 != 0:
        raise RuntimeError(
            "Inconsistencia: dup2_sorted tiene longitud "
            "impar aunque todos los grupos deberían "
            "tener size=2."
        )

    pair0 = dup2_sorted.iloc[
        0::2
    ]
    pair1 = dup2_sorted.iloc[
        1::2
    ]

    # --------------------------------------------------------
    # Guardrail: comprobar alineación de las claves
    # --------------------------------------------------------
    keys0 = pair0[
        key_cols
    ].to_numpy()
    keys1 = pair1[
        key_cols
    ].to_numpy()

    if not np.array_equal(
        keys0,
        keys1,
    ):
        raise RuntimeError(
            "Inconsistencia: los pares "
            "(usuario, timestamp) no quedaron alineados "
            "tras la ordenación."
        )

    # --------------------------------------------------------
    # Distancia geodésica elipsoidal WGS84
    # --------------------------------------------------------
    dist_m = geodesic_wgs84_m(
        pair0[lat_col].to_numpy(),
        pair0[lon_col].to_numpy(),
        pair1[lat_col].to_numpy(),
        pair1[lon_col].to_numpy(),
    )

    if not np.isfinite(
        dist_m
    ).all():
        raise RuntimeError(
            "Se obtuvieron distancias geodésicas "
            "no finitas."
        )

    # Fusionar si la distancia es menor o igual al umbral.
    # Eliminar el par completo si la distancia supera el umbral.
    merge_sel = (
        dist_m <= thr_m
    )
    merged_pairs = int(
        np.sum(merge_sel)
    )
    dropped_pairs_far = int(
        len(dist_m) - merged_pairs
    )

    # Índices originales de la primera observación de cada par
    idx0 = pair0.index.to_numpy()
    merge_idx0 = idx0[
        merge_sel
    ]

    # --------------------------------------------------------
    # Construcción de registros fusionados
    # --------------------------------------------------------
    merged_df = None
    if merged_pairs > 0:
        merged_df = out.loc[
            merge_idx0
        ].copy()

        pair0_lat = pair0[
            lat_col
        ].to_numpy(
            dtype=np.float64
        )
        pair0_lon = pair0[
            lon_col
        ].to_numpy(
            dtype=np.float64
        )
        pair1_lat = pair1[
            lat_col
        ].to_numpy(
            dtype=np.float64
        )
        pair1_lon = pair1[
            lon_col
        ].to_numpy(
            dtype=np.float64
        )

        merged_lat = (
            pair0_lat[merge_sel]
            + pair1_lat[merge_sel]
        ) / 2.0
        merged_lon = (
            pair0_lon[merge_sel]
            + pair1_lon[merge_sel]
        ) / 2.0

        merged_df.loc[
            :,
            lat_col,
        ] = merged_lat
        merged_df.loc[
            :,
            lon_col,
        ] = merged_lon

    # --------------------------------------------------------
    # Eliminar todos los registros originales de pares size=2
    # --------------------------------------------------------
    out = out.drop(
        index=dup2_idx
    )

    # Reinyectar únicamente los pares fusionados
    if merged_df is not None:
        out = pd.concat(
            [
                out,
                merged_df,
            ],
            ignore_index=True,
        )
        del merged_df
        gc.collect()
    else:
        out = out.reset_index(
            drop=True
        )

    # No convertir latitud y longitud a float32:
    # se conserva la precisión original para mantener equivalencia
    # con el pretratamiento de Deep Gravity Chile.

    # Conversión opcional del identificador para reducir memoria.
    # No altera la información ni las decisiones espaciales.
    try:
        if out[user_col].dtype == "object":
            out[user_col] = out[
                user_col
            ].astype("category")
    except (TypeError, ValueError):
        pass

    # --------------------------------------------------------
    # Reporte final
    # --------------------------------------------------------
    report.update(
        merged_pairs=merged_pairs,
        dropped_ut_pairs_far=dropped_pairs_far,
        dropped_ut_far_rows=(
            2 * dropped_pairs_far
        ),
        min_pair_distance_m=float(
            np.min(dist_m)
        ),
        max_pair_distance_m=float(
            np.max(dist_m)
        ),
    )
    update_final_counts(out)
    log_jsonl(
        log_path,
        report,
    )

    if verbose:
        print(
            f"[Dedup] dropped_exact="
            f"{report['dropped_exact']:,}"
        )
        print(
            f"[Dedup] dropped_anomalous_rows="
            f"{report['dropped_anomalous_rows']:,}"
        )
        print(
            "[Dedup] "
            "dropped_ut_invalid_rows(size!=2)="
            f"{report['dropped_ut_invalid_rows']:,}"
        )
        print(
            "[Dedup] "
            f"merged_pairs(WGS84 <= {thr_m:.0f}m)="
            f"{report['merged_pairs']:,}"
        )
        print(
            "[Dedup] "
            f"dropped_pairs_far(WGS84 > {thr_m:.0f}m)="
            f"{report['dropped_ut_pairs_far']:,}"
        )
        print(
            f"[Dedup] users_before="
            f"{report['users_before']:,} | "
            f"users_after="
            f"{report['users_after']:,} | "
            f"users_removed="
            f"{report['users_removed']:,} "
            f"({report['users_removed_pct']:.4f}%)"
        )
        print(
            f"[Dedup] shape_after={out.shape} | "
            f"mem={report['mem_mb_after']:.1f} MB"
        )

    gc.collect()
    return out

In [ ]:
# ============================================================
# 1.1 — Ejecutar deduplicación
# ============================================================
df_1 = preprocess_deduplicates(
    df=df_total,
    cfg=cfg,
    paths=paths,
    verbose=True,
)

print(
    "df_1:",
    df_1.shape,
    f"| usuarios={df_1[cfg.col_user].nunique(dropna=True):,}",
    f"| mem={mem_mb_df(df_1):.1f} MB",
)

# RAM hygiene: liberar df_total si no se utiliza posteriormente
del df_total
_ = gc.collect()

### **1.2 Selección de rango horario válido**

In [ ]:
# ============================================================
# 1.2 — Filtrado por ventana horaria válida [07:00, 23:00)
# ============================================================
def filter_valid_time_window(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Filtra los registros cuya hora se encuentra dentro de la
    ventana horaria válida [inicio, fin) declarada en la
    configuración (Módulo I: delimitación temporal).

    Además, registra el número de registros y usuarios únicos
    antes y después del tratamiento.
    """
    user_col = cfg.col_user
    ts_col = cfg.col_ts

    required = {
        user_col,
        ts_col,
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Faltan columnas requeridas: {missing}"
        )

    t0 = now_utc()
    out = df

    # --------------------------------------------------------
    # Estado inicial: registros y usuarios
    # --------------------------------------------------------
    rows_before = int(len(out))
    users_before = int(
        out[user_col].nunique(dropna=True)
    )
    mem_before = mem_mb_df(out)

    # --------------------------------------------------------
    # Asegurar tipo datetime
    # --------------------------------------------------------
    if not pd.api.types.is_datetime64_any_dtype(
        out[ts_col]
    ):
        out = out.copy()
        out[ts_col] = pd.to_datetime(
            out[ts_col],
            errors="coerce",
        )

    if out[ts_col].isna().any():
        raise ValueError(
            f"Timestamps inválidos tras coerción "
            f"en '{ts_col}'."
        )

    # --------------------------------------------------------
    # Definir ventana horaria en minutos desde medianoche
    # --------------------------------------------------------
    start_min = (
        cfg.valid_time_start.hour * 60
        + cfg.valid_time_start.minute
    )
    end_min = (
        cfg.valid_time_end.hour * 60
        + cfg.valid_time_end.minute
    )

    # --------------------------------------------------------
    # Construir minuto del día sin crear columna persistente
    # --------------------------------------------------------
    h = out[ts_col].dt.hour.to_numpy(
        dtype=np.int16,
        na_value=0,
    )
    m = out[ts_col].dt.minute.to_numpy(
        dtype=np.int16,
        na_value=0,
    )
    minute_of_day = (
        h * 60 + m
    ).astype(np.int16)

    # Ventana semiabierta:
    # inicio incluido y fin excluido
    mask_keep = (
        (minute_of_day >= start_min)
        & (minute_of_day < end_min)
    )

    # --------------------------------------------------------
    # Aplicar filtro horario
    # --------------------------------------------------------
    out2 = out.loc[
        mask_keep
    ].copy()
    out2.reset_index(
        drop=True,
        inplace=True,
    )

    # --------------------------------------------------------
    # Estado final: registros y usuarios
    # --------------------------------------------------------
    rows_after = int(
        len(out2)
    )
    users_after = int(
        out2[user_col].nunique(dropna=True)
    )

    rows_removed = int(
        rows_before - rows_after
    )
    users_removed = int(
        users_before - users_after
    )

    rows_removed_pct = (
        float(rows_removed / rows_before * 100.0)
        if rows_before > 0
        else 0.0
    )
    users_removed_pct = (
        float(users_removed / users_before * 100.0)
        if users_before > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Reporte y trazabilidad
    # --------------------------------------------------------
    report = {
        "event": "valid_time_filter",
        "ts_utc": now_utc(),
        "user_column": user_col,
        # Ventana aplicada
        "start": cfg.valid_time_start.strftime("%H:%M"),
        "end": cfg.valid_time_end.strftime("%H:%M"),
        "interval_definition": "[start, end)",
        # Registros
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_removed,
        "rows_removed_pct": rows_removed_pct,
        # Usuarios
        "users_before": users_before,
        "users_after": users_after,
        "users_removed": users_removed,
        "users_removed_pct": users_removed_pct,
        # Recursos y tiempo
        "mem_mb_before": mem_before,
        "mem_mb_after": mem_mb_df(out2),
        "elapsed_s": (
            now_utc() - t0
        ).total_seconds(),
    }

    log_jsonl(
        paths["logs_dir"] / "cleaning.jsonl",
        report,
    )

    # --------------------------------------------------------
    # Impresión de resultados
    # --------------------------------------------------------
    if verbose:
        print(
            f"[Horario válido] "
            f"{report['start']}–{report['end']} "
            f"(fin excluido)"
        )
        print(
            f"[Horario válido] "
            f"rows_before={report['rows_before']:,} | "
            f"rows_after={report['rows_after']:,} | "
            f"rows_removed={report['rows_removed']:,} "
            f"({report['rows_removed_pct']:.4f}%)"
        )
        print(
            f"[Horario válido] "
            f"users_before={report['users_before']:,} | "
            f"users_after={report['users_after']:,} | "
            f"users_removed={report['users_removed']:,} "
            f"({report['users_removed_pct']:.4f}%)"
        )
        print(
            f"[Horario válido] "
            f"shape_after={out2.shape} | "
            f"mem={report['mem_mb_after']:.1f} MB"
        )

    # --------------------------------------------------------
    # RAM hygiene
    # --------------------------------------------------------
    del h, m, minute_of_day, mask_keep
    gc.collect()

    return out2

In [ ]:
# ============================================================
# 1.2 — Ejecutar
# ============================================================
df_2 = filter_valid_time_window(
    df=df_1,
    cfg=cfg,
    paths=paths,
    verbose=True,
)

print(
    "df_2:",
    df_2.shape,
    f"| usuarios="
    f"{df_2[cfg.col_user].nunique(dropna=True):,}",
    f"| mem={mem_mb_df(df_2):.1f} MB",
)

# Si df_1 ya no se utiliza posteriormente, liberar temprano
del df_1
_ = gc.collect()

### **1.3 Cruce con DPA 2023 + Snapping**

In [ ]:
# ============================================================
# 1.3 — Cruce con DPA 2023 + Snapping
# ============================================================

# ---------- Helpers ----------
def _normalize_cut(series: pd.Series, width: int) -> pd.Series:
    """
    Normaliza códigos CUT: string sin sufijo .0 y con zfill(width).
    """
    s = series.astype(str)
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.zfill(width)
    return s


def load_dpa_comunas(cfg, paths: Dict[str, Path], target_epsg: int = 4326) -> gpd.GeoDataFrame:
    """
    Carga DPA comunal, reproyecta a EPSG target, y conserva solo columnas necesarias.
    """
    t0 = now_utc()
    gdf = gpd.read_file(cfg.dpa_comunal_path)
    if gdf.crs is None:
        raise ValueError("DPA comunal no tiene CRS definido (.prj).")

    # CRS a 4326 para join con pings lat/lon
    gdf = gdf.to_crs(epsg=target_epsg)

    if "CUT_COM" not in gdf.columns:
        raise ValueError("DPA comunal no trae 'CUT_COM'. Revisa que sea la capa comunas.")

    # Conservar columnas mínimas + geometry (RAM)
    keep_cols = ["CUT_REG", "CUT_PROV", "CUT_COM", "REGION", "PROVINCIA", "COMUNA", "SUPERFICIE", "geometry"]
    keep_cols = [c for c in keep_cols if c in gdf.columns]
    gdf = gdf[keep_cols].copy()

    # Normalización CUT (defensiva)
    if "CUT_REG" in gdf.columns:
        gdf["CUT_REG"] = _normalize_cut(gdf["CUT_REG"], 2)
    if "CUT_PROV" in gdf.columns:
        gdf["CUT_PROV"] = _normalize_cut(gdf["CUT_PROV"], 3)
    gdf["CUT_COM"] = _normalize_cut(gdf["CUT_COM"], 5)

    # SUPERFICIE a float32 si existe (RAM)
    if "SUPERFICIE" in gdf.columns:
        gdf["SUPERFICIE"] = pd.to_numeric(gdf["SUPERFICIE"], errors="coerce").astype("float32")

    log_jsonl(
        paths["logs_dir"] / "cleaning.jsonl",
        {
            "event": "dpa_loaded",
            "ts_utc": now_utc(),
            "rows": int(len(gdf)),
            "epsg": int(gdf.crs.to_epsg()),
            "cols": list(gdf.columns),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )
    return gdf


def assign_dpa_and_snap(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    gdf_comunas: gpd.GeoDataFrame,
    metric_epsg: int = 32719,          # UTM 19S
    drop_source_cols: Tuple[str, ...] = (),
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Asigna CUT_* (y nombres) por join espacial y corrige outliers espaciales vía snapping:
      - Join: gdf_points mínimo (row_id + geometry) vs DPA comunal.
      - Snapping: solo para row_id sin match, usando nearest dentro de max_distance.
      - Retorna DataFrame (sin geometry) para minimizar RAM.
      - Escribe artifact de auditoría de snapping (solo filas corregidas).

    La asignación territorial se resuelve exclusivamente mediante la DPA 2023.
    Si la fuente trae códigos administrativos propios y desea descartarlos
    temprano por RAM, declárelos en drop_source_cols; en caso contrario, la
    selección explícita de columnas de la Sección 1.6 los elimina igualmente.
    """
    t0 = now_utc()
    user_col, ts_col = cfg.col_user, cfg.col_ts
    lat_col, lon_col = cfg.col_lat, cfg.col_lon
    max_snap_m = float(cfg.snap_max_distance_m)

    # Guardrails mínimas
    for c in [user_col, ts_col, lat_col, lon_col]:
        if c not in df.columns:
            raise ValueError(f"Falta columna requerida: {c}")

    # Asegurar RangeIndex (evita merges caros; usamos row_id = índice)
    out = df
    if not isinstance(out.index, pd.RangeIndex):
        out = out.reset_index(drop=True)

    n0 = len(out)

    # --- GeoDataFrame mínimo para join (RAM-safe) ---
    gdf_points = gpd.GeoDataFrame(
        {"row_id": out.index.astype("int64", copy=False)},
        geometry=gpd.points_from_xy(out[lon_col].to_numpy(), out[lat_col].to_numpy()),
        crs="EPSG:4326",
    )

    # Join espacial: puntos (mínimos) + columnas DPA
    admin_cols = [c for c in ["CUT_REG", "CUT_PROV", "CUT_COM", "REGION", "PROVINCIA", "COMUNA", "SUPERFICIE"] if c in gdf_comunas.columns]

    gdf_join = gpd.sjoin(
        gdf_points,
        gdf_comunas[admin_cols + ["geometry"]],
        how="left",
        predicate="intersects",
    ).drop(columns=["index_right"], errors="ignore")

    # Duplicados raros por borde/overlap (si ocurren): quedarse con el primero por row_id
    if gdf_join["row_id"].duplicated().any():
        dup_n = int(gdf_join["row_id"].duplicated().sum())
        gdf_join = gdf_join.drop_duplicates(subset=["row_id"], keep="first")
    else:
        dup_n = 0

    # Construir máscara matched sin crear estructura pesada
    matched = np.zeros(n0, dtype=bool)
    rid = gdf_join["row_id"].to_numpy(dtype=np.int64, copy=False)
    matched[rid] = gdf_join["CUT_COM"].notna().to_numpy()

    n_missing = int((~matched).sum())

    log_jsonl(
        paths["logs_dir"] / "cleaning.jsonl",
        {
            "event": "dpa_join",
            "ts_utc": now_utc(),
            "rows": int(n0),
            "missing_cut_com": n_missing,
            "missing_pct": float(n_missing / n0),
            "rowid_duplicates_in_join": dup_n,
        },
    )

    if verbose:
        print(f"[DPA join] sin comuna: {n_missing:,} ({n_missing/n0:.3%})")

    # Asignación de columnas DPA al dataframe (sin merge grande)
    for c in admin_cols:
        if c not in out.columns:
            out[c] = pd.NA

    join_idx = gdf_join["row_id"].astype("int64").to_numpy()
    for c in admin_cols:
        out.iloc[join_idx, out.columns.get_loc(c)] = gdf_join[c].to_numpy()

    # Descarte opcional de columnas administrativas propias de la fuente (RAM)
    for c in drop_source_cols:
        if c in out.columns:
            out.drop(columns=c, inplace=True)

    # Si no hay missing: liberar y retornar
    if n_missing == 0:
        # Casts RAM-friendly
        for c in ["CUT_REG", "CUT_PROV", "CUT_COM", "REGION", "PROVINCIA", "COMUNA"]:
            if c in out.columns:
                out[c] = out[c].astype("category")
        if "SUPERFICIE" in out.columns:
            out["SUPERFICIE"] = pd.to_numeric(out["SUPERFICIE"], errors="coerce").astype("float32")

        del gdf_points, gdf_join
        gc.collect()

        log_jsonl(
            paths["logs_dir"] / "cleaning.jsonl",
            {
                "event": "dpa_snap_done",
                "ts_utc": now_utc(),
                "fixed": 0,
                "missing_after": 0,
                "elapsed_s": (now_utc() - t0).total_seconds(),
                "mem_mb_after": mem_mb_df(out),
            },
        )
        return out

    # --- Snapping SOLO para los missing ---
    miss_ids = np.where(~matched)[0].astype(np.int64)

    # GeoDF missing (mínimo) en métrico
    gdf_miss = gpd.GeoDataFrame(
        {"row_id": miss_ids},
        geometry=gpd.points_from_xy(out.iloc[miss_ids][lon_col].to_numpy(), out.iloc[miss_ids][lat_col].to_numpy()),
        crs="EPSG:4326",
    ).to_crs(epsg=metric_epsg)

    gdf_com_m = gdf_comunas.to_crs(epsg=metric_epsg)

    gdf_near = gpd.sjoin_nearest(
        gdf_miss,
        gdf_com_m[admin_cols + ["geometry"]],
        how="left",
        max_distance=max_snap_m,
        distance_col="dist_m",
    )

    # Filtrar matches reales (dist_m no nulo y CUT_COM no nulo)
    ok = gdf_near["CUT_COM"].notna() & gdf_near["dist_m"].notna()
    gdf_fix = gdf_near.loc[ok].copy()

    if gdf_fix.empty:
        if verbose:
            print("[Snap] 0 matches dentro del umbral; se mantiene missing.")
        missing_after = n_missing
        log_jsonl(
            paths["logs_dir"] / "cleaning.jsonl",
            {
                "event": "snap",
                "ts_utc": now_utc(),
                "max_snap_m": max_snap_m,
                "fixed": 0,
                "missing_after": int(missing_after),
                "missing_after_pct": float(missing_after / n0),
                "elapsed_s": (now_utc() - t0).total_seconds(),
            },
        )
        del gdf_points, gdf_join, gdf_miss, gdf_com_m, gdf_near, gdf_fix
        gc.collect()
        return out

    # Snap geométrico al polígono (solo para los corregidos; usualmente pocos)
    polys = gdf_com_m.geometry  # índice coincide con gdf_com_m

    def _snap_inside_metric(pt, poly):
        if poly is None or poly.is_empty:
            return pt
        if poly.contains(pt):
            return pt
        snapped = nearest_points(poly, pt)[0]
        if not poly.contains(snapped):
            inner = poly.buffer(-1)  # 1m hacia interior
            if not inner.is_empty:
                snapped = nearest_points(inner, pt)[0]
        return snapped

    snapped_pts = []
    # gdf_fix index es el de gdf_miss (no el row_id); row_id viene como columna
    for _, r in gdf_fix.iterrows():
        poly = polys.loc[r["index_right"]]
        snapped_pts.append(_snap_inside_metric(r.geometry, poly))

    gser_snap = gpd.GeoSeries(snapped_pts, crs=f"EPSG:{metric_epsg}").to_crs(epsg=4326)
    new_lon = gser_snap.x.to_numpy()
    new_lat = gser_snap.y.to_numpy()

    row_ids_fix = gdf_fix["row_id"].to_numpy(dtype=np.int64, copy=False)

    # Artifact de auditoría del snap (solo observaciones corregidas).
    # Deriva de la fuente telco: se escribe en el directorio de artefactos y
    # no forma parte de la entrega pública.
    snap_audit = pd.DataFrame(
        {
            "row_id": row_ids_fix,
            "dist_m": gdf_fix["dist_m"].to_numpy(dtype="float32", copy=False),
            "lat_orig": out.iloc[row_ids_fix][lat_col].to_numpy(dtype="float64", copy=False),
            "lon_orig": out.iloc[row_ids_fix][lon_col].to_numpy(dtype="float64", copy=False),
            "lat_new": new_lat.astype("float64", copy=False),
            "lon_new": new_lon.astype("float64", copy=False),
            "CUT_COM": gdf_fix["CUT_COM"].astype(str).to_numpy(),
        }
    )
    snap_audit_path = paths["tables_dir"] / "snap_audit.parquet"
    snap_audit.to_parquet(snap_audit_path, index=False)

    # Actualizar lat/lon + DPA attrs en el dataframe principal
    out.iloc[row_ids_fix, out.columns.get_loc(lat_col)] = new_lat
    out.iloc[row_ids_fix, out.columns.get_loc(lon_col)] = new_lon

    # Asignar atributos admin desde gdf_fix
    for c in admin_cols:
        out.iloc[row_ids_fix, out.columns.get_loc(c)] = gdf_fix[c].to_numpy()

    # Recalcular missing CUT_COM
    missing_after = int(out["CUT_COM"].isna().sum())

    log_jsonl(
        paths["logs_dir"] / "cleaning.jsonl",
        {
            "event": "snap",
            "ts_utc": now_utc(),
            "max_snap_m": max_snap_m,
            "fixed": int(len(row_ids_fix)),
            "missing_after": int(missing_after),
            "missing_after_pct": float(missing_after / n0),
            "snap_audit_path": str(snap_audit_path),
        },
    )

    if verbose:
        print(f"[Snap] corregidos: {len(row_ids_fix):,} | sin comuna después: {missing_after:,} ({missing_after/n0:.3%})")

    # Casts RAM-friendly (dejar códigos como category)
    for c in ["CUT_REG", "CUT_PROV", "CUT_COM", "REGION", "PROVINCIA", "COMUNA"]:
        if c in out.columns:
            out[c] = out[c].astype("category")
    if "SUPERFICIE" in out.columns:
        out["SUPERFICIE"] = pd.to_numeric(out["SUPERFICIE"], errors="coerce").astype("float32")

    # Limpieza intermedios pesados
    del gdf_points, gdf_join, gdf_miss, gdf_com_m, gdf_near, gdf_fix, gser_snap, snap_audit
    gc.collect()

    log_jsonl(
        paths["logs_dir"] / "cleaning.jsonl",
        {
            "event": "dpa_snap_done",
            "ts_utc": now_utc(),
            "rows": int(len(out)),
            "missing_cut_com_final": int(out["CUT_COM"].isna().sum()),
            "elapsed_s": (now_utc() - t0).total_seconds(),
            "mem_mb_after": mem_mb_df(out),
        },
    )

    return out

In [ ]:
# ============================================================
# 1.3 — Ejecutar
# ============================================================
gdf_comunas = load_dpa_comunas(cfg, paths)

df_3 = assign_dpa_and_snap(
    df_2,
    cfg,
    paths,
    gdf_comunas,
    metric_epsg=32719,
    drop_source_cols=(),
    verbose=True,
)

print("df_3:", df_3.shape, f"| mem={mem_mb_df(df_3):.1f} MB")
print("Missing CUT_COM:", int(df_3["CUT_COM"].isna().sum()))

# RAM hygiene
del df_2, gdf_comunas
_ = gc.collect()

### **1.4 Depuración final de casos residuales sin comuna**

In [ ]:
# ============================================================
# 1.4 — Depuración residual sin comuna
# ============================================================
def finalize_missing_commune(
    df: pd.DataFrame,
    cfg,
    commune_col: str = "CUT_COM",
    strategy: Literal["drop", "label"] = "drop",
    unknown_label: str = "UNKNOWN",
    add_trace_col: bool = False,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Resuelve los registros que permanecen sin comuna después
    del cruce con DPA y del procedimiento de snapping.

    Estrategias
    -----------
    drop:
        Elimina los registros cuyo código comunal es nulo.
    label:
        Conserva los registros y reemplaza el código comunal
        faltante por una etiqueta explícita.

    Además, registra el número de filas y usuarios únicos antes
    y después del tratamiento.
    """
    user_col = cfg.col_user

    required = {
        user_col,
        commune_col,
    }
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"Faltan columnas requeridas: {missing_cols}"
        )

    if strategy not in {"drop", "label"}:
        raise ValueError(
            "strategy inválida. Usa 'drop' o 'label'."
        )

    t0 = now_utc()

    # --------------------------------------------------------
    # Estado inicial
    # --------------------------------------------------------
    rows_before = int(len(df))
    users_before = int(
        df[user_col].nunique(dropna=True)
    )
    mem_before = mem_mb_df(df)

    miss_mask = df[commune_col].isna()
    missing_before = int(
        miss_mask.sum()
    )

    # Usuarios que poseen al menos un registro sin comuna.
    # No necesariamente serán eliminados completamente.
    users_with_missing_before = int(
        df.loc[
            miss_mask,
            user_col,
        ].nunique(dropna=True)
    )

    out = df

    # --------------------------------------------------------
    # Columna opcional de trazabilidad
    # --------------------------------------------------------
    if add_trace_col:
        out = out.copy()
        out["was_missing_commune"] = miss_mask

    # --------------------------------------------------------
    # Aplicar estrategia
    # --------------------------------------------------------
    if strategy == "drop":
        if missing_before == 0:
            out2 = out
        else:
            out2 = out.loc[
                ~miss_mask
            ].copy()
    else:  # strategy == "label"
        out2 = out.copy()
        out2[commune_col] = (
            out2[commune_col]
            .astype("string[python]")
            .fillna(unknown_label)
            .astype("category")
        )

    if out2 is not out:
        out2.reset_index(
            drop=True,
            inplace=True,
        )

    # --------------------------------------------------------
    # Estado final
    # --------------------------------------------------------
    rows_after = int(
        len(out2)
    )
    users_after = int(
        out2[user_col].nunique(dropna=True)
    )

    rows_removed = int(
        rows_before - rows_after
    )
    users_removed = int(
        users_before - users_after
    )

    rows_removed_pct = (
        float(rows_removed / rows_before * 100.0)
        if rows_before > 0
        else 0.0
    )
    users_removed_pct = (
        float(users_removed / users_before * 100.0)
        if users_before > 0
        else 0.0
    )

    missing_after = int(
        out2[commune_col].isna().sum()
    )

    # --------------------------------------------------------
    # Reporte
    # --------------------------------------------------------
    report = {
        "event": "finalize_missing_commune",
        "ts_utc": now_utc(),
        "strategy": strategy,
        "user_column": user_col,
        "commune_column": commune_col,
        # Registros
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_removed,
        "rows_removed_pct": rows_removed_pct,
        # Usuarios
        "users_before": users_before,
        "users_after": users_after,
        "users_removed": users_removed,
        "users_removed_pct": users_removed_pct,
        # Registros y usuarios afectados por missing
        "missing_before": missing_before,
        "missing_after": missing_after,
        "users_with_missing_before": users_with_missing_before,
        # Recursos
        "mem_mb_before": mem_before,
        "mem_mb_after": mem_mb_df(out2),
        "elapsed_s": (
            now_utc() - t0
        ).total_seconds(),
    }

    log_jsonl(
        paths["logs_dir"] / "cleaning.jsonl",
        report,
    )

    # --------------------------------------------------------
    # Impresión
    # --------------------------------------------------------
    if verbose:
        print(
            f"[Depuración comuna] "
            f"strategy={report['strategy']} | "
            f"missing_before={report['missing_before']:,} | "
            f"missing_after={report['missing_after']:,}"
        )
        print(
            f"[Depuración comuna] "
            f"rows_before={report['rows_before']:,} | "
            f"rows_after={report['rows_after']:,} | "
            f"rows_removed={report['rows_removed']:,} "
            f"({report['rows_removed_pct']:.4f}%)"
        )
        print(
            f"[Depuración comuna] "
            f"users_before={report['users_before']:,} | "
            f"users_after={report['users_after']:,} | "
            f"users_removed={report['users_removed']:,} "
            f"({report['users_removed_pct']:.4f}%)"
        )
        print(
            f"[Depuración comuna] "
            f"users_with_missing_before="
            f"{report['users_with_missing_before']:,} | "
            f"mem={report['mem_mb_after']:.1f} MB"
        )

    # --------------------------------------------------------
    # RAM hygiene
    # --------------------------------------------------------
    del miss_mask
    gc.collect()

    return out2

In [ ]:
# ============================================================
# 1.4 — Ejecutar
# ============================================================
df_4 = finalize_missing_commune(
    df=df_3,
    cfg=cfg,
    commune_col="CUT_COM",
    strategy="drop",
    add_trace_col=False,
    verbose=True,
)

print(
    "df_4:",
    df_4.shape,
    f"| usuarios="
    f"{df_4[cfg.col_user].nunique(dropna=True):,}",
    f"| mem={mem_mb_df(df_4):.1f} MB",
)

# Liberar anterior si ya no se utiliza
del df_3
_ = gc.collect()

### **1.5 Eliminación de usuarios con un único registro**

In [ ]:
# ============================================================
# 1.5 — Filtrar usuarios con < min_records pings
# ============================================================
def filter_users_by_min_records(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    min_records: int = 2,
    verbose: bool = True,
) -> pd.DataFrame:
    user_col = cfg.col_user
    if user_col not in df.columns:
        raise ValueError(f"No existe la columna '{user_col}'.")

    t0 = now_utc()
    n0 = int(len(df))
    out = df

    # RAM: user como category si está en object
    try:
        if out[user_col].dtype == "object":
            out[user_col] = out[user_col].astype("category")
    except Exception:
        pass

    # Conteo por usuario (Series: user -> n)
    counts = out.groupby(user_col, observed=True).size()
    users_before = int(counts.shape[0])

    keep_users = counts.index[counts >= min_records]
    users_keep = int(keep_users.shape[0])
    users_low = int(users_before - users_keep)
    pct_low = float(users_low / users_before * 100.0) if users_before else 0.0

    # Filtrar filas por usuario válido (semi-join)
    keep_mask = out[user_col].isin(keep_users)
    out2 = out.loc[keep_mask].copy()
    out2.reset_index(drop=True, inplace=True)

    report = {
        "event": "filter_users_min_records",
        "ts_utc": now_utc(),
        "min_records": int(min_records),
        "rows_before": n0,
        "rows_after": int(len(out2)),
        "users_before": users_before,
        "users_after": int(out2[user_col].nunique()),
        "users_below_min": users_low,
        "users_below_min_pct": pct_low,
        "mem_mb_before": mem_mb_df(out),
        "mem_mb_after": mem_mb_df(out2),
        "elapsed_s": (now_utc() - t0).total_seconds(),
    }
    log_jsonl(paths["logs_dir"] / "cleaning.jsonl", report)

    if verbose:
        print(
            f"[Usuarios] total={report['users_before']:,} | <{min_records} ping(s)={report['users_below_min']:,} "
            f"({report['users_below_min_pct']:.2f}%)"
        )
        print(
            f"[Filtrado] filas: {report['rows_before']:,} -> {report['rows_after']:,} | "
            f"usuarios_after={report['users_after']:,} | mem={report['mem_mb_after']:.1f}MB"
        )

    # RAM hygiene
    del counts, keep_users, keep_mask
    gc.collect()
    return out2

In [ ]:
# ============================================================
# 1.5 — Ejecutar
# ============================================================
df_5 = filter_users_by_min_records(
    df=df_4,
    cfg=cfg,
    paths=paths,
    min_records=cfg.min_records_per_user,
    verbose=True,
)

print("df_5:", df_5.shape, f"| mem={mem_mb_df(df_5):.1f} MB")

# Liberar anterior
del df_4
_ = gc.collect()

### **1.6 Estandarización final de esquema y tipos**

In [ ]:
# ============================================================
# 1.6 — Estandarizar esquema y tipos
# ============================================================
def standardize_trace_schema(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    sort_keys: bool = True,
    drop_aux_cols: bool = True,
    keep_optional_admin: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Estandariza el esquema final de las trazas móviles:
      1) valida y filtra coordenadas inválidas;
      2) normaliza los tipos de datos;
      3) conserva las columnas necesarias;
      4) elimina columnas auxiliares y cualquier columna sobrante
         de la fuente original;
      5) ordena establemente por usuario y timestamp.

    Además, registra el número de registros y usuarios únicos
    antes y después del tratamiento.
    """
    t0 = now_utc()

    user_col = cfg.col_user
    ts_col = cfg.col_ts
    lat_col = cfg.col_lat
    lon_col = cfg.col_lon

    required = {
        user_col,
        ts_col,
        lat_col,
        lon_col,
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Faltan columnas requeridas: {missing}"
        )

    # --------------------------------------------------------
    # Estado inicial
    # --------------------------------------------------------
    rows_before = int(len(df))
    users_before = int(
        df[user_col].nunique(dropna=True)
    )
    mem_before = mem_mb_df(df)

    out = df

    # --------------------------------------------------------
    # Coordenadas: conversión numérica cuando sea necesaria
    # --------------------------------------------------------
    if (
        not pd.api.types.is_numeric_dtype(out[lat_col])
        or not pd.api.types.is_numeric_dtype(out[lon_col])
    ):
        out = out.copy()
        out[lat_col] = pd.to_numeric(
            out[lat_col],
            errors="coerce",
        )
        out[lon_col] = pd.to_numeric(
            out[lon_col],
            errors="coerce",
        )

    # --------------------------------------------------------
    # Filtrar coordenadas inválidas o faltantes
    # --------------------------------------------------------
    mask_valid = (
        out[lat_col].notna()
        & out[lon_col].notna()
        & out[lat_col].between(
            -90.0,
            90.0,
            inclusive="both",
        )
        & out[lon_col].between(
            -180.0,
            180.0,
            inclusive="both",
        )
    )

    dropped_bad_coords = int(
        (~mask_valid).sum()
    )

    if dropped_bad_coords > 0:
        out = out.loc[
            mask_valid
        ].copy()

    # --------------------------------------------------------
    # Tipos base
    # --------------------------------------------------------
    # Identificador de usuario como categoría
    try:
        if out[user_col].dtype == "object":
            out[user_col] = out[
                user_col
            ].astype("category")
    except (TypeError, ValueError):
        pass

    # Timestamp
    if not pd.api.types.is_datetime64_any_dtype(
        out[ts_col]
    ):
        out = out.copy()
        out[ts_col] = pd.to_datetime(
            out[ts_col],
            errors="coerce",
        )

    if out[ts_col].isna().any():
        raise ValueError(
            f"Timestamps inválidos tras coerción "
            f"en '{ts_col}'."
        )

    # Coordenadas a float32 para reducir memoria
    try:
        out[lat_col] = out[
            lat_col
        ].astype("float32")
        out[lon_col] = out[
            lon_col
        ].astype("float32")
    except (TypeError, ValueError):
        pass

    # --------------------------------------------------------
    # Códigos CUT
    # --------------------------------------------------------
    cut_cols = [
        c
        for c in [
            "CUT_REG",
            "CUT_PROV",
            "CUT_COM",
        ]
        if c in out.columns
    ]

    for c in cut_cols:
        try:
            s = (
                out[c]
                .astype("string[python]")
                .str.replace(
                    r"\.0$",
                    "",
                    regex=True,
                )
            )
            if c == "CUT_REG":
                s = s.str.zfill(2)
            elif c == "CUT_PROV":
                s = s.str.zfill(3)
            elif c == "CUT_COM":
                s = s.str.zfill(5)
            out[c] = s.astype("category")
        except (TypeError, ValueError):
            pass

    # --------------------------------------------------------
    # Selección de columnas esenciales
    # --------------------------------------------------------
    cols_keep = [
        user_col,
        ts_col,
        lat_col,
        lon_col,
    ] + cut_cols

    if keep_optional_admin:
        for c in [
            "REGION",
            "PROVINCIA",
            "COMUNA",
            "SUPERFICIE",
        ]:
            if c in out.columns:
                cols_keep.append(c)

    if drop_aux_cols:
        aux_drop = {
            "dist_m",
            "index_right",
            "poly_geom",
            "geometry",
            "geometry_orig",
            "geometry_snap",
            "was_missing_commune",
        }
        cols_keep = [
            c
            for c in cols_keep
            if c not in aux_drop
        ]

    # Evitar columnas inexistentes y duplicadas.
    # Esta selección explícita descarta además cualquier columna sobrante
    # de la fuente original que no forme parte del esquema del pipeline.
    cols_keep = list(
        dict.fromkeys(
            c
            for c in cols_keep
            if c in out.columns
        )
    )

    # Copia consolidada con el esquema final
    out = out.loc[
        :,
        cols_keep,
    ].copy()

    # --------------------------------------------------------
    # Tipos de atributos administrativos opcionales
    # --------------------------------------------------------
    for c in [
        "REGION",
        "PROVINCIA",
        "COMUNA",
    ]:
        if c in out.columns:
            out[c] = out[c].astype(
                "category"
            )

    if "SUPERFICIE" in out.columns:
        out["SUPERFICIE"] = pd.to_numeric(
            out["SUPERFICIE"],
            errors="coerce",
        ).astype("float32")

    # --------------------------------------------------------
    # Orden estable requerido para construir stays
    # --------------------------------------------------------
    if sort_keys:
        out.sort_values(
            [user_col, ts_col],
            kind="mergesort",
            inplace=True,
        )
        out.reset_index(
            drop=True,
            inplace=True,
        )
    elif not isinstance(
        out.index,
        pd.RangeIndex,
    ):
        out.reset_index(
            drop=True,
            inplace=True,
        )

    # --------------------------------------------------------
    # Estado final
    # --------------------------------------------------------
    rows_after = int(
        len(out)
    )
    users_after = int(
        out[user_col].nunique(dropna=True)
    )

    rows_removed = int(
        rows_before - rows_after
    )
    users_removed = int(
        users_before - users_after
    )

    rows_removed_pct = (
        float(
            rows_removed
            / rows_before
            * 100.0
        )
        if rows_before > 0
        else 0.0
    )
    users_removed_pct = (
        float(
            users_removed
            / users_before
            * 100.0
        )
        if users_before > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Reporte
    # --------------------------------------------------------
    report = {
        "event": "standardize_schema",
        "ts_utc": now_utc(),
        # Configuración
        "user_column": user_col,
        "sort_keys": bool(sort_keys),
        "drop_aux_cols": bool(drop_aux_cols),
        "keep_optional_admin": bool(
            keep_optional_admin
        ),
        # Registros
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_removed,
        "rows_removed_pct": rows_removed_pct,
        "dropped_bad_coords": (
            dropped_bad_coords
        ),
        # Usuarios
        "users_before": users_before,
        "users_after": users_after,
        "users_removed": users_removed,
        "users_removed_pct": (
            users_removed_pct
        ),
        # Esquema
        "cols": list(out.columns),
        "dtypes": {
            c: str(out.dtypes[c])
            for c in out.columns
        },
        # Recursos
        "mem_mb_before": mem_before,
        "mem_mb_after": mem_mb_df(out),
        "elapsed_s": (
            now_utc() - t0
        ).total_seconds(),
    }

    log_jsonl(
        paths["logs_dir"] / "cleaning.jsonl",
        report,
    )

    # --------------------------------------------------------
    # Impresión
    # --------------------------------------------------------
    if verbose:
        print(
            f"[Schema] "
            f"rows_before={report['rows_before']:,} | "
            f"rows_after={report['rows_after']:,} | "
            f"rows_removed={report['rows_removed']:,} "
            f"({report['rows_removed_pct']:.4f}%)"
        )
        print(
            f"[Schema] "
            f"dropped_bad_coords="
            f"{report['dropped_bad_coords']:,}"
        )
        print(
            f"[Schema] "
            f"users_before="
            f"{report['users_before']:,} | "
            f"users_after="
            f"{report['users_after']:,} | "
            f"users_removed="
            f"{report['users_removed']:,} "
            f"({report['users_removed_pct']:.4f}%)"
        )
        print(
            f"[Schema] "
            f"mem_before="
            f"{report['mem_mb_before']:.1f} MB | "
            f"mem_after="
            f"{report['mem_mb_after']:.1f} MB"
        )
        print(
            f"[Schema] cols: "
            f"{report['cols']}"
        )

    # --------------------------------------------------------
    # RAM hygiene
    # --------------------------------------------------------
    del mask_valid
    gc.collect()

    return out

In [ ]:
# ============================================================
# 1.6 — Ejecutar
# ============================================================
df_6 = standardize_trace_schema(
    df=df_5,
    cfg=cfg,
    paths=paths,
    sort_keys=True,
    drop_aux_cols=True,
    keep_optional_admin=True,
    verbose=True,
)

print(
    "df_6:",
    df_6.shape,
    f"| usuarios="
    f"{df_6[cfg.col_user].nunique(dropna=True):,}",
    f"| mem={mem_mb_df(df_6):.1f} MB",
)
print("dtypes:", {c: str(t) for c, t in df_6.dtypes.items()})

# RAM hygiene
del df_5
_ = gc.collect()

### **1.7 Asignación de H3**

In [ ]:
# ============================================================
# 1.7 — Asignación H3 (main + parent)
# ============================================================
def _get_h3_funcs():
    """
    Compatibilidad h3-py:
    - nuevas: latlng_to_cell / cell_to_parent
    - antiguas: geo_to_h3 / h3_to_parent
    """
    latlng_to_cell = getattr(h3, "latlng_to_cell", None)
    cell_to_parent = getattr(h3, "cell_to_parent", None)
    if latlng_to_cell is not None and cell_to_parent is not None:
        return latlng_to_cell, cell_to_parent

    geo_to_h3 = getattr(h3, "geo_to_h3", None)
    h3_to_parent = getattr(h3, "h3_to_parent", None)
    if geo_to_h3 is None or h3_to_parent is None:
        raise ImportError("La versión instalada de 'h3' no expone latlng_to_cell/cell_to_parent ni geo_to_h3/h3_to_parent.")
    return geo_to_h3, h3_to_parent


def assign_h3_ids(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    h3_col: str = "h3_id",
    parent_col: str = "h3_parent",
    as_category: bool = True,
    batch_size: int = 500_000,
    verbose: bool = True,
) -> pd.DataFrame:
    t0 = now_utc()
    lat_col, lon_col = cfg.col_lat, cfg.col_lon
    res_main, res_parent = int(cfg.h3_res_main), int(cfg.h3_res_parent)

    for c in (lat_col, lon_col):
        if c not in df.columns:
            raise ValueError(f"Falta columna requerida: '{c}'")

    lat = df[lat_col].to_numpy()
    lon = df[lon_col].to_numpy()

    if np.isnan(lat).any() or np.isnan(lon).any():
        raise ValueError("Existen lat/lon NaN. Revisa estandarización y snapping previo.")

    if (lat.min() < -90) or (lat.max() > 90) or (lon.min() < -180) or (lon.max() > 180):
        raise ValueError("Existen lat/lon fuera de rango geográfico válido.")

    latlng_to_cell, cell_to_parent = _get_h3_funcs()

    n = int(len(df))
    h3_main = np.empty(n, dtype=object)
    h3_parent = np.empty(n, dtype=object)

    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        for i in range(start, end):
            hid = latlng_to_cell(float(lat[i]), float(lon[i]), res_main)
            h3_main[i] = hid
            h3_parent[i] = cell_to_parent(hid, res_parent)
        if verbose:
            print(f"[H3] batch {start:,}-{end:,} / {n:,}")

    out = df.copy(deep=False)
    out[h3_col] = h3_main
    out[parent_col] = h3_parent

    if out[h3_col].isna().any() or out[parent_col].isna().any():
        raise ValueError("Se generaron valores nulos en H3. Revisa coordenadas de entrada.")

    if as_category:
        out[h3_col] = out[h3_col].astype("category")
        out[parent_col] = out[parent_col].astype("category")

    report = {
        "event": "assign_h3",
        "ts_utc": now_utc(),
        "rows": n,
        "res_main": res_main,
        "res_parent": res_parent,
        "as_category": bool(as_category),
        "batch_size": int(batch_size),
        "mem_mb_before": mem_mb_df(df),
        "mem_mb_after": mem_mb_df(out),
        "elapsed_s": (now_utc() - t0).total_seconds(),
    }
    log_jsonl(paths["logs_dir"] / "cleaning.jsonl", report)

    if verbose:
        print(f"[H3] completado | res_main={res_main} res_parent={res_parent} | mem={report['mem_mb_after']:.1f}MB")

    gc.collect()
    return out

In [ ]:
# ============================================================
# 1.7 — Ejecutar asignación H3
# ============================================================
df_7 = assign_h3_ids(
    df=df_6,
    cfg=cfg,
    paths=paths,
    h3_col="h3_id",
    parent_col="h3_parent",
    as_category=True,
    batch_size=500_000,
    verbose=True,
)

print("df_7:", df_7.shape, f"| mem={mem_mb_df(df_7):.1f} MB")
print(
    f"[H3] celdas únicas: "
    f"res{cfg.h3_res_main}={df_7['h3_id'].nunique():,} | "
    f"res{cfg.h3_res_parent}={df_7['h3_parent'].nunique():,}"
)

# RAM hygiene
del df_6
_ = gc.collect()

### **1.8 Ordenamiento y preparación temporal mínima**

In [ ]:
# ============================================================
# 1.8 — Ordenamiento (ya hecho) + preparación temporal mínima
# ============================================================
def finalize_ping_level(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    persist: bool = True,
    persist_name: str = "pings_clean_h3_temporal.parquet",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    - RAM hygiene final tras H3: asegurar que no haya geometry, castear categorías.
    - Crear campos temporales mínimos: event_date, dow, is_weekday, minute_of_day.
    - (Opcional) persistir parquet como cache para iteraciones posteriores.

    El parquet persistido deriva de la fuente telco: se escribe en el
    directorio de artefactos y no forma parte de la entrega pública.
    """
    t0 = now_utc()
    user_col, ts_col = cfg.col_user, cfg.col_ts
    out = df

    # 1) Asegurar ausencia de geometry
    dropped_geometry = False
    if "geometry" in out.columns:
        out = out.drop(columns=["geometry"])
        dropped_geometry = True

    # 2) Casts RAM: CUT_*, H3 y user como category
    for c in ["CUT_REG", "CUT_PROV", "CUT_COM", "REGION", "PROVINCIA", "COMUNA", "h3_id", "h3_parent", user_col]:
        if c in out.columns and out[c].dtype != "category":
            try:
                out[c] = out[c].astype("category")
            except Exception:
                pass

    # 3) Timestamp a datetime64[ns] si fuera necesario (guardrail)
    if not pd.api.types.is_datetime64_any_dtype(out[ts_col]):
        out[ts_col] = pd.to_datetime(out[ts_col], errors="coerce")
        if out[ts_col].isna().any():
            raise ValueError(f"Timestamps inválidos tras coerción en '{ts_col}'.")

    # 4) Campos temporales mínimos
    ts_values = out[ts_col].to_numpy(dtype="datetime64[ns]", copy=False)
    dt = pd.DatetimeIndex(ts_values)

    out["event_date"] = dt.normalize().to_numpy(dtype="datetime64[ns]", copy=False)
    out["dow"] = dt.dayofweek.astype("int8", copy=False)
    out["is_weekday"] = (out["dow"].to_numpy(dtype=np.int8, copy=False) < 5)

    minute_of_day = (dt.hour.astype(np.int16) * 60 + dt.minute.astype(np.int16)).astype(np.int16, copy=False)
    out["minute_of_day"] = minute_of_day

    # 5) Logging
    report = {
        "event": "finalize_ping_level",
        "ts_utc": now_utc(),
        "rows": int(len(out)),
        "cols_added": ["event_date", "dow", "is_weekday", "minute_of_day"],
        "dropped_geometry": bool(dropped_geometry),
        "mem_mb_after": mem_mb_df(out),
        "persist": bool(persist),
        "elapsed_s": (now_utc() - t0).total_seconds(),
    }
    log_jsonl(paths["logs_dir"] / "cleaning.jsonl", report)

    if verbose:
        print(
            f"[1.8] temporal fields OK | rows={report['rows']:,} | "
            f"mem={report['mem_mb_after']:.1f}MB | persist={persist}"
        )

    # 6) Persistencia (cache ping-level, Nivel 2)
    if persist:
        out_path = paths["cache_dir"] / persist_name
        out.to_parquet(out_path, index=False)
        log_jsonl(
            paths["logs_dir"] / "io.jsonl",
            {"event": "cache_saved", "ts_utc": now_utc(), "path": str(out_path), "rows": int(len(out))},
        )
        if verbose:
            print(f"[Cache] guardado: {out_path.name}")

    # RAM hygiene
    del dt, ts_values, minute_of_day
    gc.collect()
    return out

In [ ]:
# ============================================================
# 1.8 — Ejecutar
# ============================================================
df_final = finalize_ping_level(
    df=df_7,
    cfg=cfg,
    paths=paths,
    persist=True,
    persist_name="pings_clean_h3_temporal.parquet",
    verbose=True,
)

print("df_final:", df_final.shape, f"| mem={mem_mb_df(df_final):.1f} MB")
print(
    f"[1.8] rango observado: "
    f"{df_final['event_date'].min().date()} → {df_final['event_date'].max().date()} "
    f"| días distintos={df_final['event_date'].nunique():,} "
    f"| hábiles={int(df_final['is_weekday'].sum()):,} registros"
)

# RAM hygiene
del df_7
_ = gc.collect()

## *2. Construcción de stays y estados de movimiento*

Esta sección implementa las representaciones intermedias del Módulo III:
construye **permanencias** (*stays*) y **desplazamientos** (*moves*) a partir de
la base de eventos depurada e indexada.

Una permanencia es un segmento maximal de observaciones consecutivas del mismo
usuario en la misma celda H3, sin interrupciones mayores al gap máximo, que
satisface una duración mínima y un número mínimo de observaciones. Un
desplazamiento es la transición entre dos permanencias consecutivas del mismo
usuario cuya separación temporal es positiva y no excede el máximo declarado.

Los parámetros efectivos —duración mínima, gap máximo, observaciones mínimas por
permanencia y duración máxima de desplazamiento— se declaran en
`ExperimentConfig` (Sección 0.2). Esta sección no realiza inferencia sobre las
permanencias: la asignación de significado (hogar, lugar de trabajo) ocurre
recién en las Secciones 4 y 5, sobre el universo elegible que define el
*gatekeeper* de la Sección 3.

In [ ]:
# ============================================================
# 2.0 — QA mínimo + parámetros
# ============================================================
USER_COL = cfg.col_user
TS_COL = cfg.col_ts
LAT_COL = cfg.col_lat
LON_COL = cfg.col_lon
H3_COL = "h3_id"
PARENT_COL = "h3_parent"

STAY_MIN_MINUTES = int(cfg.stay_min_minutes)
GAP_MAX_MINUTES = int(cfg.gap_max_minutes)
MIN_PINGS_IN_STAY = int(cfg.stay_min_pings)
MOVE_MAX_MINUTES = int(cfg.move_max_minutes)

# Trabajar sobre una vista de columnas mínimas (no duplica bloques de datos)
cols_min = [USER_COL, TS_COL, LAT_COL, LON_COL, H3_COL, PARENT_COL]
cols_min = [c for c in cols_min if c in df_final.columns]

df_pings = df_final.loc[:, cols_min]

# Casts RAM (solo si hace falta)
if df_pings[USER_COL].dtype != "category":
    df_pings[USER_COL] = df_pings[USER_COL].astype("category")
if df_pings[LAT_COL].dtype != "float32":
    df_pings[LAT_COL] = df_pings[LAT_COL].astype("float32")
if df_pings[LON_COL].dtype != "float32":
    df_pings[LON_COL] = df_pings[LON_COL].astype("float32")
for c in [H3_COL, PARENT_COL]:
    if c in df_pings.columns and df_pings[c].dtype != "category":
        df_pings[c] = df_pings[c].astype("category")

# QA mínimos (sin sort, sin copias masivas)
assert df_pings[USER_COL].notna().all(), "Hay usuarios nulos."
assert pd.api.types.is_datetime64_any_dtype(df_pings[TS_COL]), "TS no es datetime64[ns]."
assert df_pings[[H3_COL, PARENT_COL]].isna().sum().sum() == 0, "Hay H3 nulos."

# Duplicados (u,t) debe ser 0 (viene desde Sección 1.1)
n_dups_ut = int(df_pings.duplicated(subset=[USER_COL, TS_COL], keep=False).sum())
assert n_dups_ut == 0, f"Persisten duplicados (usuario,timestamp): {n_dups_ut:,}"

# QA orden temporal intra-usuario (sin sort)
u_codes = df_pings[USER_COL].cat.codes.to_numpy(np.int32, copy=False)
ts_arr = df_pings[TS_COL].to_numpy(dtype="datetime64[ns]", copy=False)
viol = int(((ts_arr[1:] < ts_arr[:-1]) & (u_codes[1:] == u_codes[:-1])).sum())
assert viol == 0, f"df_pings no está ordenado por (user,ts). Violaciones: {viol:,}"

del u_codes, ts_arr
_ = gc.collect()

log_jsonl(
    paths["logs_dir"] / "stays.jsonl",
    {
        "event": "section2_params",
        "ts_utc": now_utc(),
        "rows_pings": int(len(df_pings)),
        "stay_min_min": STAY_MIN_MINUTES,
        "gap_max_min": GAP_MAX_MINUTES,
        "min_pings_stay": MIN_PINGS_IN_STAY,
        "move_max_min": MOVE_MAX_MINUTES,
        "mem_mb_df_pings": mem_mb_df(df_pings),
        "cols_used": cols_min,
    },
)

print("[2.0] OK | df_pings:", df_pings.shape, f"| mem={mem_mb_df(df_pings):.1f} MB")

In [ ]:
# ============================================================
# 2.1 — Construcción de stays (SIN columnas temporales en df_pings)
# ============================================================
def build_stays_numpy(
    df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    h3_col: str = "h3_id",
    parent_col: str = "h3_parent",
    min_duration_min: int = 15,
    max_gap_min: int = 60,
    min_pings_in_stay: int = 2,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Construye permanencias (stays) a partir de la base de eventos indexada
    (Módulo III: representaciones intermedias).

    Un stay es un segmento maximal de observaciones consecutivas del mismo
    usuario en la misma celda H3, delimitado por cambio de usuario, cambio de
    celda o gap temporal superior a max_gap_min, y que satisface simultáneamente
    duración >= min_duration_min y n_pings >= min_pings_in_stay.

    La implementación opera sobre arrays de códigos categóricos y usa límites de
    segmento en lugar de groupby, para acotar el consumo de memoria sobre bases
    de varios millones de registros. Requiere que df venga ordenado por
    (usuario, timestamp), condición verificada en la Sección 2.0.
    """
    t0 = now_utc()
    user_col, ts_col = cfg.col_user, cfg.col_ts
    lat_col, lon_col = cfg.col_lat, cfg.col_lon

    required = {user_col, ts_col, lat_col, lon_col, h3_col}
    miss = required - set(df.columns)
    if miss:
        raise ValueError(f"Faltan columnas: {miss}")

    if df[user_col].dtype != "category":
        raise ValueError("Se espera user como category (ver 2.0).")
    if not pd.api.types.is_datetime64_any_dtype(df[ts_col]):
        raise ValueError("ts debe ser datetime64[ns].")
    if df[h3_col].dtype != "category":
        df[h3_col] = df[h3_col].astype("category")
    if parent_col in df.columns and df[parent_col].dtype != "category":
        df[parent_col] = df[parent_col].astype("category")

    n = int(len(df))
    if n == 0:
        return pd.DataFrame(columns=[user_col, "stay_id"])

    # Arrays base (livianos)
    u = df[user_col].cat.codes.to_numpy(np.int32, copy=False)
    h = df[h3_col].cat.codes.to_numpy(np.int32, copy=False)
    ts = df[ts_col].to_numpy(dtype="datetime64[ns]", copy=False)
    lat = df[lat_col].to_numpy(dtype=np.float32, copy=False)
    lon = df[lon_col].to_numpy(dtype=np.float32, copy=False)

    # Flags de nuevo usuario
    new_user = np.empty(n, dtype=bool)
    new_user[0] = True
    new_user[1:] = (u[1:] != u[:-1])

    # Gap en minutos (float32); infinito al cambiar usuario
    gap_min = np.empty(n, dtype=np.float32)
    gap_min[0] = np.inf
    gap_min[1:] = ((ts[1:] - ts[:-1]) / np.timedelta64(1, "m")).astype(np.float32)
    gap_min[new_user] = np.inf

    # Cambio de H3 o gap grande => nuevo segmento
    h_change = np.empty(n, dtype=bool)
    h_change[0] = True
    h_change[1:] = (h[1:] != h[:-1])
    h_change[new_user] = True

    new_seg = new_user | h_change | (gap_min > float(max_gap_min))

    # Segment boundaries
    seg_starts = np.flatnonzero(new_seg).astype(np.int64)
    seg_ends = np.empty_like(seg_starts)
    seg_ends[:-1] = seg_starts[1:] - 1
    seg_ends[-1] = n - 1
    seg_len = (seg_ends - seg_starts + 1).astype(np.int32)

    # Agregados por segmento (sin groupby)
    start_ts = ts[seg_starts]
    end_ts = ts[seg_ends]

    # Means por segmento usando reduceat
    lat_sum = np.add.reduceat(lat.astype(np.float64), seg_starts)
    lon_sum = np.add.reduceat(lon.astype(np.float64), seg_starts)
    lat_mean = (lat_sum / seg_len).astype(np.float32)
    lon_mean = (lon_sum / seg_len).astype(np.float32)

    # first h3/parent y user (por código y categorías)
    user_codes_seg = u[seg_starts]
    h3_codes_seg = h[seg_starts]
    parent_codes_seg = None
    if parent_col in df.columns:
        parent_codes_seg = df[parent_col].cat.codes.to_numpy(np.int32, copy=False)[seg_starts]

    # Duración
    dur_min = ((end_ts - start_ts) / np.timedelta64(1, "m")).astype(np.float32)

    # Filtro calidad
    ok = (dur_min >= float(min_duration_min)) & (seg_len >= int(min_pings_in_stay))

    seg_starts = seg_starts[ok]
    seg_ends = seg_ends[ok]
    seg_len = seg_len[ok]
    start_ts = start_ts[ok]
    end_ts = end_ts[ok]
    dur_min = dur_min[ok]
    lat_mean = lat_mean[ok]
    lon_mean = lon_mean[ok]
    user_codes_seg = user_codes_seg[ok]
    h3_codes_seg = h3_codes_seg[ok]
    if parent_codes_seg is not None:
        parent_codes_seg = parent_codes_seg[ok]

    # Reconstrucción a categorías originales
    user_cat = df[user_col].cat.categories
    h3_cat = df[h3_col].cat.categories

    stays = pd.DataFrame(
        {
            user_col: pd.Categorical.from_codes(user_codes_seg, categories=user_cat),
            "start_ts": start_ts,
            "end_ts": end_ts,
            "n_pings": seg_len,
            h3_col: pd.Categorical.from_codes(h3_codes_seg, categories=h3_cat),
            "lat_mean": lat_mean,
            "lon_mean": lon_mean,
            "duration_min": dur_min,
        }
    )

    if parent_codes_seg is not None:
        parent_cat = df[parent_col].cat.categories
        stays[parent_col] = pd.Categorical.from_codes(parent_codes_seg, categories=parent_cat)

    # Orden (stays es pequeño) + stay_id por usuario
    stays.sort_values([user_col, "start_ts"], kind="mergesort", inplace=True)
    stays.reset_index(drop=True, inplace=True)
    stays["stay_id"] = stays.groupby(user_col, sort=False, observed=False).cumcount().astype("int32")

    log_jsonl(
        paths["logs_dir"] / "stays.jsonl",
        {
            "event": "stays_built",
            "ts_utc": now_utc(),
            "rows_stays": int(len(stays)),
            "users_with_stays": int(stays[user_col].nunique()),
            "min_duration_min": int(min_duration_min),
            "min_pings_in_stay": int(min_pings_in_stay),
            "max_gap_min": int(max_gap_min),
            "elapsed_s": (now_utc() - t0).total_seconds(),
            "mem_mb_stays": mem_mb_df(stays),
        },
    )

    if verbose:
        print(f"[Stays] rows={len(stays):,} | users={stays[user_col].nunique():,} | mem={mem_mb_df(stays):.1f}MB")

    # RAM hygiene
    del u, h, ts, lat, lon, new_user, gap_min, h_change, new_seg
    gc.collect()
    return stays

In [ ]:
# ============================================================
# 2.2 — Ejecutar stays + diagnóstico mínimo
# ============================================================
df_stays = build_stays_numpy(
    df=df_pings,
    cfg=cfg,
    paths=paths,
    h3_col=H3_COL,
    parent_col=PARENT_COL,
    min_duration_min=STAY_MIN_MINUTES,
    max_gap_min=GAP_MAX_MINUTES,
    min_pings_in_stay=MIN_PINGS_IN_STAY,
    verbose=True,
)

# Diagnóstico mínimo (sin tablas gigantes)
n_users_total = int(df_pings[USER_COL].nunique())
n_users_stay = int(df_stays[USER_COL].nunique())
total_pings = int(len(df_pings))
pings_in_stays = int(df_stays["n_pings"].sum())

print(f"[Coverage] users_total={n_users_total:,} | users_with_stay={n_users_stay:,} ({(n_users_stay/n_users_total):.2%})")
print(f"[Coverage] pings_total={total_pings:,} | pings_in_stays≈{pings_in_stays:,} ({(pings_in_stays/total_pings):.2%})")
print(df_stays[["duration_min", "n_pings"]].describe())

# Liberación de pings que no se seguirán ocupando
del df_final
_ = gc.collect()

In [ ]:
# ============================================================
# 2.3 — Moves desde stays (vectorizado, sin shift/columnas next_*)
# ============================================================
def build_moves_from_stays(
    stays: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    max_gap_min: int = 60,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Construye desplazamientos (moves) entre permanencias consecutivas del mismo
    usuario (Módulo III: transiciones conductuales).

    Un move se define entre los stays i e i+1 del mismo usuario cuando la
    separación temporal entre el fin del primero y el inicio del segundo es
    estrictamente positiva y no excede max_gap_min. La distancia se calcula
    como geodésica elipsoidal WGS84 entre los centroides de ambas permanencias.

    Nota: la unidad de análisis es el usuario. Estas transiciones no son
    equivalentes a las relaciones territoriales agregadas de la rama
    territorial-relacional, que se construyen directamente sobre la base
    depurada y no requieren permanencias.
    """
    t0 = now_utc()
    user_col = cfg.col_user

    if stays.empty:
        return pd.DataFrame(columns=[user_col, "origin_h3", "dest_h3", "move_duration_min", "move_distance_m"])

    # stays ya viene ordenado por (user, start_ts)
    s = stays
    u = s[user_col].cat.codes.to_numpy(np.int32, copy=False)
    same_user_next = (u[1:] == u[:-1])

    # Datos del par (i -> i+1) solo cuando el usuario es el mismo
    idx = np.where(same_user_next)[0]
    if idx.size == 0:
        return pd.DataFrame(columns=[user_col, "origin_h3", "dest_h3", "move_duration_min", "move_distance_m"])

    end_ts = s["end_ts"].to_numpy(dtype="datetime64[ns]", copy=False)
    start_ts = s["start_ts"].to_numpy(dtype="datetime64[ns]", copy=False)

    move_dur = ((start_ts[idx + 1] - end_ts[idx]) / np.timedelta64(1, "m")).astype(np.float32)

    # Continuidad temporal (marco): 0 < dur <= max_gap
    ok = (move_dur > 0) & (move_dur <= float(max_gap_min))
    idx = idx[ok]
    move_dur = move_dur[ok]

    if idx.size == 0:
        return pd.DataFrame(columns=[user_col, "origin_h3", "dest_h3", "move_duration_min", "move_distance_m"])

    lat0 = s["lat_mean"].to_numpy(dtype=np.float64, copy=False)[idx]
    lon0 = s["lon_mean"].to_numpy(dtype=np.float64, copy=False)[idx]
    lat1 = s["lat_mean"].to_numpy(dtype=np.float64, copy=False)[idx + 1]
    lon1 = s["lon_mean"].to_numpy(dtype=np.float64, copy=False)[idx + 1]

    dist_m = geodesic_wgs84_m(
        lat0,
        lon0,
        lat1,
        lon1,
    )

    moves = pd.DataFrame(
        {
            user_col: s[user_col].iloc[idx].reset_index(drop=True),
            "stay_id": s["stay_id"].to_numpy(np.int32, copy=False)[idx],
            "origin_h3": s["h3_id"].iloc[idx].reset_index(drop=True),
            "dest_h3": s["h3_id"].iloc[idx + 1].reset_index(drop=True),
            "move_start_ts": end_ts[idx],
            "move_end_ts": start_ts[idx + 1],
            "move_duration_min": move_dur,
            "move_distance_m": dist_m,
        }
    )

    log_jsonl(
        paths["logs_dir"] / "stays.jsonl",
        {
            "event": "moves_built",
            "ts_utc": now_utc(),
            "rows_moves": int(len(moves)),
            "max_gap_min": int(max_gap_min),
            "elapsed_s": (now_utc() - t0).total_seconds(),
            "mem_mb_moves": mem_mb_df(moves),
        },
    )

    if verbose:
        print(f"[Moves] rows={len(moves):,} | mem={mem_mb_df(moves):.1f}MB")
        print(moves[["move_duration_min", "move_distance_m"]].describe())

    gc.collect()
    return moves


df_moves = build_moves_from_stays(
    stays=df_stays,
    cfg=cfg,
    paths=paths,
    max_gap_min=MOVE_MAX_MINUTES,
    verbose=True,
)

## *3. Filtro de cobertura mínima (gatekeeper)*

El *gatekeeper* determina el universo elegible: el subconjunto de usuarios cuya
densidad observacional permite sostener las inferencias posteriores. Es la
operacionalización del principio de **elegibilidad antes que exhaustividad**: se
prefiere un universo pequeño con trazabilidad suficiente por sobre uno amplio
con evidencia fragmentaria.

El criterio combina cuatro condiciones sobre las permanencias de cada usuario,
declaradas en `ExperimentConfig`: número mínimo de días calendario con
evidencia, número mínimo de días hábiles con evidencia, horas acumuladas de
permanencia y amplitud temporal del período observado. Un usuario es elegible
solo si satisface las cuatro simultáneamente.

**El filtro es deliberadamente restrictivo y su efecto es sustantivo.** La
retención resultante define el universo sobre el cual operan todas las secciones
siguientes, y condiciona el alcance de cualquier conclusión: los usuarios
elegibles no constituyen una muestra aleatoria, sino precisamente aquellos con
trazas más densas. El análisis de sensibilidad de la Sección 13 cuantifica cuánto
depende el tamaño del universo de cada uno de los cuatro umbrales.

Los artefactos que esta sección persiste derivan de la fuente telco y no forman
parte de la entrega pública.

In [ ]:
# ============================================================
# 3 — Gatekeeper
# ============================================================
def compute_gatekeeper_stats_from_stays(
    stays: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Métricas de cobertura observacional por usuario, calculadas exclusivamente
    sobre permanencias (Módulo III: universo elegible basal).

    Métricas del criterio de elegibilidad:
      - n_days_with_stays     : días calendario únicos con evidencia
      - n_weekdays_with_stays : días hábiles únicos con evidencia
      - total_stay_hours      : horas acumuladas de permanencia
      - span_days             : amplitud temporal (max_date - min_date + 1)

    Métrica adicional, solo de auditoría:
      - n_weekday_stays : número de permanencias iniciadas en día hábil.
        NO participa del criterio. Se distingue de n_weekdays_with_stays porque
        cuenta permanencias y no días únicos: varias permanencias en un mismo
        día hábil incrementan esta métrica pero no aquella.

    Se reporta también n_stays como descriptor del volumen por usuario.

    La implementación remapea los códigos de usuario a un espacio compacto para
    evitar arreglos del tamaño del catálogo global de categorías, y calcula el
    día de la semana aritméticamente sobre días desde la época (1970-01-01 fue
    jueves; con Monday=0, dow = (días + 3) % 7).
    """
    t0 = now_utc()
    user_col = cfg.col_user

    required = {user_col, "start_ts", "end_ts", "duration_min"}
    miss = required - set(stays.columns)
    if miss:
        raise ValueError(f"Faltan columnas en stays: {miss}")

    if stays.empty:
        stats = pd.DataFrame(columns=[
            user_col, "user_code_orig",
            "n_stays", "n_days_with_stays", "n_weekdays_with_stays",
            "n_weekday_stays", "total_stay_hours", "span_days",
        ])
        log_jsonl(paths["logs_dir"] / "gatekeeper.jsonl", {
            "event": "gatekeeper_stats",
            "ts_utc": now_utc(),
            "rows_stays": 0,
            "users": 0,
        })
        return stats

    # user debe ser category para que el remapeo sea eficiente
    if stays[user_col].dtype != "category":
        stays[user_col] = stays[user_col].astype("category")

    # Arrays base
    u_orig = stays[user_col].cat.codes.to_numpy(np.int32, copy=False)
    if (u_orig < 0).any():
        raise ValueError("Hay usuarios NaN en stays (cat.code = -1).")

    # Remap a espacio compacto (evita arrays del tamaño de categorías globales)
    uuniq_orig = np.unique(u_orig)                                  # códigos originales presentes
    u_comp = np.searchsorted(uuniq_orig, u_orig).astype(np.int32)   # 0..m-1
    m = int(uuniq_orig.size)

    # Fecha del stay (día calendario) usando días desde epoch (int)
    start_ts = stays["start_ts"].to_numpy(dtype="datetime64[ns]", copy=False)
    date_int = start_ts.astype("datetime64[D]").astype(np.int32, copy=False)

    # day-of-week aritmético: 1970-01-01 fue jueves => Monday=0 => Thursday=3
    dow = (date_int + 3) % 7
    is_weekday_stay = (dow < 5)

    # n_stays
    n_stays = np.bincount(u_comp, minlength=m).astype(np.int32)

    # total stay hours
    dur_min = stays["duration_min"].to_numpy(dtype=np.float32, copy=False)
    total_min = np.bincount(u_comp, weights=dur_min.astype(np.float64, copy=False), minlength=m)
    total_hours = (total_min / 60.0).astype(np.float32)

    # span_days: min/max date_int por usuario
    min_date = np.full(m, np.iinfo(np.int32).max, dtype=np.int32)
    max_date = np.full(m, np.iinfo(np.int32).min, dtype=np.int32)
    np.minimum.at(min_date, u_comp, date_int)
    np.maximum.at(max_date, u_comp, date_int)
    span_days = (max_date - min_date + 1).astype(np.int32)

    # n_days_with_stays y n_weekdays_with_stays como UNIQUE days por usuario
    order = np.lexsort((date_int, u_comp))   # sort by user then date
    u_s = u_comp[order]
    d_s = date_int[order]
    dow_s = ((d_s + 3) % 7).astype(np.int8, copy=False)
    wk_s = (dow_s < 5)

    is_new_pair = np.ones_like(u_s, dtype=bool)
    is_new_pair[1:] = (u_s[1:] != u_s[:-1]) | (d_s[1:] != d_s[:-1])

    # unique days per user
    u_unique_days = u_s[is_new_pair]
    n_days = np.bincount(u_unique_days, minlength=m).astype(np.int32)

    # unique weekday days per user
    u_unique_wkdays = u_s[is_new_pair & wk_s]
    n_wkdays = np.bincount(u_unique_wkdays, minlength=m).astype(np.int32)

    # audit: número de permanencias en día hábil (no-unique; no es criterio)
    n_weekday_stays = np.bincount(u_comp, weights=is_weekday_stay.astype(np.int32), minlength=m).astype(np.int32)

    # Reconstruir user ids (categoría compacta)
    cats = stays[user_col].cat.categories.to_numpy()
    user_vals = cats[uuniq_orig]
    user_cat = pd.Categorical(user_vals)

    stats = pd.DataFrame({
        user_col: user_cat,
        "user_code_orig": uuniq_orig.astype(np.int32),
        "n_stays": n_stays,
        "n_days_with_stays": n_days,
        "n_weekdays_with_stays": n_wkdays,
        "n_weekday_stays": n_weekday_stays,   # audit
        "total_stay_hours": total_hours,
        "span_days": span_days,
    })

    log_jsonl(
        paths["logs_dir"] / "gatekeeper.jsonl",
        {
            "event": "gatekeeper_stats",
            "ts_utc": now_utc(),
            "rows_stays": int(len(stays)),
            "users": int(m),
            "mem_mb_stats": mem_mb_df(stats),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    if verbose:
        print(f"[Gatekeeper stats] users={m:,} | rows={len(stats):,} | mem={mem_mb_df(stats):.1f}MB")
        print(stats[["n_stays", "n_days_with_stays", "n_weekdays_with_stays", "total_stay_hours", "span_days"]].describe())

    # RAM hygiene
    del u_orig, uuniq_orig, u_comp, start_ts, date_int, dow, is_weekday_stay
    del order, u_s, d_s, dow_s, wk_s, is_new_pair, u_unique_days, u_unique_wkdays
    del n_stays, total_min, total_hours, min_date, max_date, span_days, n_days, n_wkdays, n_weekday_stays
    gc.collect()

    return stats

In [ ]:
def apply_gatekeeper(
    stays: pd.DataFrame,
    moves: pd.DataFrame,
    gk_stats: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    persist: bool = True,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Aplica el criterio de elegibilidad y filtra stays y moves al universo
    resultante.

    Un usuario es elegible si satisface SIMULTÁNEAMENTE los cuatro umbrales
    declarados en la configuración:
      - n_days_with_stays     >= cfg.gk_min_days_with_stays
      - n_weekdays_with_stays >= cfg.gk_min_weekdays_with_stays
      - total_stay_hours      >= cfg.gk_min_total_hours
      - span_days             >= cfg.gk_min_span_days

    El filtrado opera sobre códigos enteros de categoría y luego reduce las
    categorías a las efectivamente usadas, lo que baja sustantivamente la
    memoria del universo resultante.

    La retención es un resultado del procedimiento, no un parámetro: depende de
    la estructura observacional de la fuente. Una retención baja indica
    insuficiencia longitudinal del insumo, no un error de configuración.

    Los artefactos persistidos derivan de la fuente telco y se escriben en el
    directorio de artefactos; no forman parte de la entrega pública.

    Retorna
    -------
    (stays_gk, moves_gk, eligible_users)
    """
    t0 = now_utc()
    user_col = cfg.col_user

    D_min = int(cfg.gk_min_days_with_stays)
    D_wdmin = int(cfg.gk_min_weekdays_with_stays)
    H_min = float(cfg.gk_min_total_hours)
    S_min = int(cfg.gk_min_span_days)

    mask_keep = (
        (gk_stats["n_days_with_stays"] >= D_min) &
        (gk_stats["n_weekdays_with_stays"] >= D_wdmin) &
        (gk_stats["total_stay_hours"] >= H_min) &
        (gk_stats["span_days"] >= S_min)
    )

    users_before = int(gk_stats.shape[0])
    eligible_stats = gk_stats.loc[mask_keep, [user_col, "user_code_orig"]]
    users_after = int(eligible_stats.shape[0])
    retention = float(users_after / users_before) if users_before else float("nan")

    if users_after == 0:
        raise ValueError(
            "El gatekeeper no dejó ningún usuario elegible.\n"
            f"  Umbrales aplicados: días>={D_min}, hábiles>={D_wdmin}, "
            f"horas>={H_min:.1f}, span>={S_min}\n"
            "  Revise la estructura observacional de la fuente y, si "
            "corresponde, recalibre los umbrales en ExperimentConfig."
        )

    # Filtrado por códigos originales de categoría (rápido)
    eligible_codes = eligible_stats["user_code_orig"].to_numpy(dtype=np.int32, copy=False)

    # stays filter
    if stays[user_col].dtype != "category":
        stays[user_col] = stays[user_col].astype("category")
    stay_codes = stays[user_col].cat.codes.to_numpy(np.int32, copy=False)
    keep_stays = np.isin(stay_codes, eligible_codes)
    stays_gk = stays.loc[keep_stays].copy()
    stays_gk.reset_index(drop=True, inplace=True)

    # moves filter
    if moves[user_col].dtype != "category":
        moves[user_col] = moves[user_col].astype("category")
    move_codes = moves[user_col].cat.codes.to_numpy(np.int32, copy=False)
    keep_moves = np.isin(move_codes, eligible_codes)
    moves_gk = moves.loc[keep_moves].copy()
    moves_gk.reset_index(drop=True, inplace=True)

    # Reducir categorías a las usadas (baja RAM fuerte)
    stays_gk[user_col] = stays_gk[user_col].cat.remove_unused_categories()
    moves_gk[user_col] = moves_gk[user_col].cat.remove_unused_categories()

    for c in ["h3_id", "h3_parent"]:
        if c in stays_gk.columns and stays_gk[c].dtype == "category":
            stays_gk[c] = stays_gk[c].cat.remove_unused_categories()
        if c in moves_gk.columns and moves_gk[c].dtype == "category":
            moves_gk[c] = moves_gk[c].cat.remove_unused_categories()

    log_jsonl(
        paths["logs_dir"] / "gatekeeper.jsonl",
        {
            "event": "gatekeeper_apply",
            "ts_utc": now_utc(),
            "users_before": users_before,
            "users_after": users_after,
            "retention_pct": retention,
            "D_min": D_min,
            "D_wdmin": D_wdmin,
            "H_min": H_min,
            "S_min": S_min,
            "stays_before": int(len(stays)),
            "stays_after": int(len(stays_gk)),
            "moves_before": int(len(moves)),
            "moves_after": int(len(moves_gk)),
            "mem_mb_stays_gk": mem_mb_df(stays_gk),
            "mem_mb_moves_gk": mem_mb_df(moves_gk),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    if verbose:
        print(f"[Gatekeeper] umbrales: días>={D_min} | hábiles>={D_wdmin} | horas>={H_min:.1f} | span>={S_min}")
        print(f"[Gatekeeper] users_before={users_before:,} | users_after={users_after:,} | retention={retention:.2%}")
        print(f"[Gatekeeper] stays: {len(stays):,} -> {len(stays_gk):,} | moves: {len(moves):,} -> {len(moves_gk):,}")
        if retention < 0.01:
            print(
                "[Gatekeeper] Retención inferior al 1 %. Es un resultado del "
                "procedimiento, no necesariamente un error: indica que la "
                "fuente es abundante en volumen agregado y escasa en densidad "
                "longitudinal por usuario. La Sección 13 cuantifica la "
                "sensibilidad del universo a cada umbral."
            )

    # Persistencia de artefactos (Nivel 2: derivados de la fuente telco,
    # escritos en el directorio de artefactos y excluidos de la entrega)
    if persist:
        gk_stats.to_parquet(paths["tables_dir"] / "gatekeeper_user_stats.parquet", index=False)
        stays_gk.to_parquet(paths["cache_dir"] / "stays_gk.parquet", index=False)
        moves_gk.to_parquet(paths["cache_dir"] / "moves_gk.parquet", index=False)
        eligible_stats[[user_col]].to_parquet(paths["tables_dir"] / "gatekeeper_eligible_users.parquet", index=False)

        log_jsonl(
            paths["logs_dir"] / "gatekeeper.jsonl",
            {
                "event": "gatekeeper_artifacts_saved",
                "ts_utc": now_utc(),
                "gatekeeper_user_stats": str(paths["tables_dir"] / "gatekeeper_user_stats.parquet"),
                "eligible_users": str(paths["tables_dir"] / "gatekeeper_eligible_users.parquet"),
                "stays_gk": str(paths["cache_dir"] / "stays_gk.parquet"),
                "moves_gk": str(paths["cache_dir"] / "moves_gk.parquet"),
            },
        )

    # eligible users values (compact)
    eligible_users = eligible_stats[user_col].reset_index(drop=True)

    # RAM hygiene
    del stay_codes, move_codes, keep_stays, keep_moves
    gc.collect()

    return stays_gk, moves_gk, eligible_users

In [ ]:
# ============================================================
# 3.1 — Ejecutar gatekeeper
# ============================================================
df_user_gk = compute_gatekeeper_stats_from_stays(df_stays, cfg, paths, verbose=True)

df_stays, df_moves, eligible_users = apply_gatekeeper(
    stays=df_stays,
    moves=df_moves,
    gk_stats=df_user_gk,
    cfg=cfg,
    paths=paths,
    persist=True,
    verbose=True,
)

# Si df_pings sigue vivo, este es un buen punto para soltarlo
del df_pings
_ = gc.collect()

In [ ]:
# ============================================================
# 3.2 — Diagnóstico mínimo post-gatekeeper
# ============================================================
n_users_gk = int(df_stays[cfg.col_user].nunique())

print("[Post-GK] users_stays:", n_users_gk)
print("[Post-GK] stays_rows:", int(len(df_stays)))
print("[Post-GK] moves_rows:", int(len(df_moves)))
print(
    f"[Post-GK] universo elegible: {n_users_gk:,} de "
    f"{int(len(df_user_gk)):,} usuarios con permanencias "
    f"({n_users_gk / max(len(df_user_gk), 1):.2%})"
)

print(df_stays[["duration_min", "n_pings"]].describe())
print(df_moves[["move_duration_min", "move_distance_m"]].describe())

_ = gc.collect()

## *4. Identificación de Hogar (Home Proxy)*

Esta sección infiere la **ancla residencial** de cada usuario elegible: la celda
H3 que concentra su presencia nocturna y matinal durante el período observado.

El procedimiento pondera el solapamiento de cada permanencia con dos ventanas
horarias —una primaria nocturna y una secundaria matinal— mediante el parámetro
`home_alpha`, y selecciona por usuario la celda con mayor tiempo ponderado
acumulado. La asignación se valida contra dos condiciones: número mínimo de días
distintos con evidencia en esa celda y proporción mínima de dominancia respecto
del tiempo ponderado total del usuario. Si alguna falla, el ancla queda como
faltante.

Se registra además `home_conf`, el producto entre la dominancia y un factor de
saturación por días observados. No es una probabilidad: es un descriptor
ordinal de la fuerza de la evidencia sobre la que se apoya la asignación, y
acompaña al producto como condición de uso.

**Advertencia sobre el producto de esta sección.** El ancla residencial es un
atributo inferido, no observado ni declarado por la persona. Junto con el ancla
laboral de la Sección 5 constituye el vector canónico de re-identificación en la
literatura de movilidad: el par residencia–trabajo funciona como
cuasi-identificador incluso sobre registros seudonimizados, y el riesgo se
amplifica cuando la resolución espacial es fina y la cohorte es pequeña. El
manuscrito discute esta cuestión y registra la restricción de uso legítimo
correspondiente. En consecuencia, este cuaderno no imprime asignaciones
individuales: los diagnósticos que emite son agregados, y los artefactos que
persiste no forman parte de la entrega pública.

In [ ]:
# ============================================================
# 4 — Home Proxy + conf^H
# ============================================================
def identify_home_location(
    stays: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    h3_col: str = "h3_id",
    start_col: str = "start_ts",
    end_col: str = "end_ts",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Infiere el ancla residencial por usuario (Módulo III: anclas operacionales).

    Procedimiento
    -------------
    1. Para cada permanencia se calcula el solapamiento, en segundos, con la
       ventana primaria (cfg.home_win_primary) y con la secundaria
       (cfg.home_win_secondary).
    2. Se pondera: score = alpha * primaria + (1 - alpha) * secundaria,
       con alpha = cfg.home_alpha.
    3. Se agrega por (usuario, celda H3) el score total y el número de días
       calendario distintos con evidencia.
    4. Se selecciona por usuario la celda con mayor score total.
    5. La asignación es válida si days_seen >= cfg.home_min_days y
       dominance_ratio >= cfg.home_min_ratio. En caso contrario, home_missing=1.

    Salida
    ------
    home_h3       : celda residencial inferida, o NA si la asignación no es válida
    home_missing  : 1 si el ancla no pudo resolverse, 0 en caso contrario
    home_conf     : dominancia * min(days_seen / home_min_days, 1)

    home_conf NO es una probabilidad ni una medida calibrada: es un descriptor
    ordinal de la fuerza de la evidencia, destinado a acompañar al producto como
    condición de uso. Un valor de 1,0 indica dominancia total dentro de la
    ventana observada, lo que sobre trazas escasas puede reflejar tanto
    concentración real como cobertura insuficiente.

    Estatuto del producto
    ---------------------
    El ancla residencial es un atributo inferido. Su combinación con el ancla
    laboral constituye un cuasi-identificador. No debe publicarse, persistirse
    ni reutilizarse a nivel individual fuera del entorno autorizado.
    """
    t0 = now_utc()
    user_col = cfg.col_user

    (ev_start, ev_end) = cfg.home_win_primary
    (mo_start, mo_end) = cfg.home_win_secondary
    alpha = float(cfg.home_alpha)
    min_days = int(cfg.home_min_days)
    min_ratio = float(cfg.home_min_ratio)

    required = {user_col, h3_col, start_col, end_col}
    miss = required - set(stays.columns)
    if miss:
        raise ValueError(f"Faltan columnas en stays: {miss}")

    if stays.empty:
        out = pd.DataFrame(
            {
                user_col: pd.Categorical([], categories=[]),
                "home_h3": pd.Categorical([], categories=[]),
                "home_missing": np.array([], dtype=np.int8),
                "home_conf": np.array([], dtype=np.float32),
            }
        )
        return out

    # Subset mínimo (sin copy masivo)
    s = stays[[user_col, h3_col, start_col, end_col]].copy()

    # Reducir categorías (baja RAM fuerte)
    if s[user_col].dtype == "category":
        s[user_col] = s[user_col].cat.remove_unused_categories()
    if s[h3_col].dtype == "category":
        s[h3_col] = s[h3_col].cat.remove_unused_categories()

    # Guardrail: datetime
    if not pd.api.types.is_datetime64_any_dtype(s[start_col]) or not pd.api.types.is_datetime64_any_dtype(s[end_col]):
        raise ValueError("start_ts/end_ts deben ser datetime64[ns].")

    # ---- overlap en segundos (enteros) ----
    start_ts = s[start_col].to_numpy(dtype="datetime64[ns]", copy=False)
    end_ts = s[end_col].to_numpy(dtype="datetime64[ns]", copy=False)

    dt_start = pd.DatetimeIndex(start_ts)
    dt_end = pd.DatetimeIndex(end_ts)

    start_sec = (
        dt_start.hour.astype(np.int32) * 3600
        + dt_start.minute.astype(np.int32) * 60
        + dt_start.second.astype(np.int32)
    ).astype(np.int32, copy=False)

    end_sec = (
        dt_end.hour.astype(np.int32) * 3600
        + dt_end.minute.astype(np.int32) * 60
        + dt_end.second.astype(np.int32)
    ).astype(np.int32, copy=False)

    # defensivo: si end < start, forzar end=start (no deberían cruzar día)
    end_sec = np.maximum(end_sec, start_sec)

    ev0 = ev_start.hour * 3600 + ev_start.minute * 60 + ev_start.second
    ev1 = ev_end.hour * 3600 + ev_end.minute * 60 + ev_end.second
    mo0 = mo_start.hour * 3600 + mo_start.minute * 60 + mo_start.second
    mo1 = mo_end.hour * 3600 + mo_end.minute * 60 + mo_end.second

    # overlap(a,b) = max(0, min(end,b) - max(start,a))
    ev_overlap = np.maximum(0, np.minimum(end_sec, ev1) - np.maximum(start_sec, ev0)).astype(np.float32)
    mo_overlap = np.maximum(0, np.minimum(end_sec, mo1) - np.maximum(start_sec, mo0)).astype(np.float32)

    home_wsec = (alpha * ev_overlap + (1.0 - alpha) * mo_overlap).astype(np.float32)
    mask_signal = home_wsec > 0

    if not mask_signal.any():
        all_users = pd.DataFrame({user_col: stays[user_col].cat.remove_unused_categories().unique()})
        out = all_users.assign(home_h3=pd.NA, home_missing=np.int8(1), home_conf=np.float32(0.0))
        log_jsonl(
            paths["logs_dir"] / "home.jsonl",
            {"event": "home_no_signal", "ts_utc": now_utc(), "users": int(len(out))},
        )
        if verbose:
            print(
                "[Home] Ninguna permanencia solapa con las ventanas declaradas: "
                "no se resolvió ningún ancla residencial."
            )
        return out[[user_col, "home_h3", "home_missing", "home_conf"]]

    s = s.loc[mask_signal].copy()
    s["home_wsec"] = home_wsec[mask_signal]

    # Día calendario por inicio (int days) para nunique rápido
    date_int = (
        s[start_col]
        .to_numpy(dtype="datetime64[ns]", copy=False)
        .astype("datetime64[D]")
        .astype(np.int32, copy=False)
    )
    s["date_int"] = date_int

    # Agregación por (user,h3)
    cand = (
        s.groupby([user_col, h3_col], observed=True, sort=False)
        .agg(
            total_home_wsec=("home_wsec", "sum"),
            days_seen=("date_int", pd.Series.nunique),
        )
        .reset_index()
    )

    # Total por usuario para dominancia (cand es pequeño)
    cand["user_total_wsec"] = cand.groupby(user_col, observed=True, sort=False)["total_home_wsec"].transform("sum")
    cand["dominance_ratio"] = (cand["total_home_wsec"] / cand["user_total_wsec"]).astype(np.float32)

    # Mejor candidato por usuario
    cand.sort_values([user_col, "total_home_wsec"], ascending=[True, False], kind="mergesort", inplace=True)
    best = cand.drop_duplicates(subset=[user_col], keep="first").copy()

    best["valid_home"] = (best["days_seen"] >= min_days) & (best["dominance_ratio"] >= min_ratio)

    # conf^H: dominancia * saturación por días
    day_factor = np.clip(best["days_seen"].to_numpy(dtype=np.float32) / max(min_days, 1), 0.0, 1.0)
    best["home_conf"] = (best["dominance_ratio"].to_numpy(dtype=np.float32) * day_factor).astype(np.float32)

    # Salida: todos los usuarios post-GK (desde stays)
    all_users = pd.DataFrame({user_col: stays[user_col].cat.remove_unused_categories().unique()})
    out = all_users.merge(best[[user_col, h3_col, "valid_home", "home_conf"]], on=user_col, how="left")
    out.rename(columns={h3_col: "home_h3"}, inplace=True)

    out["home_missing"] = np.where(out["valid_home"] == True, 0, 1).astype(np.int8)
    out.loc[out["home_missing"] == 1, "home_h3"] = pd.NA
    out["home_conf"] = out["home_conf"].fillna(np.float32(0.0)).astype(np.float32)

    # Tipos compactos
    if "home_h3" in out.columns:
        try:
            out["home_h3"] = out["home_h3"].astype("category")
        except Exception:
            pass

    report = {
        "event": "home_built",
        "ts_utc": now_utc(),
        "users_total": int(len(out)),
        "users_home_ok": int((out["home_missing"] == 0).sum()),
        "users_home_missing": int((out["home_missing"] == 1).sum()),
        "alpha": alpha,
        "min_days": min_days,
        "min_ratio": min_ratio,
        "evening": [ev_start.strftime("%H:%M"), ev_end.strftime("%H:%M")],
        "morning": [mo_start.strftime("%H:%M"), mo_end.strftime("%H:%M")],
        "mem_mb_out": mem_mb_df(out),
        "elapsed_s": (now_utc() - t0).total_seconds(),
    }
    log_jsonl(paths["logs_dir"] / "home.jsonl", report)

    if verbose:
        print(
            f"[Home] ventanas: primaria {report['evening'][0]}–{report['evening'][1]} | "
            f"secundaria {report['morning'][0]}–{report['morning'][1]} | alpha={alpha}"
        )
        print(
            f"[Home] criterio: días>={min_days} | dominancia>={min_ratio:.2f}"
        )
        print(
            f"[Home] users={report['users_total']:,} | "
            f"resueltos={report['users_home_ok']:,} "
            f"({report['users_home_ok'] / max(report['users_total'], 1):.2%}) | "
            f"missing={report['users_home_missing']:,}"
        )

    # RAM hygiene
    del s, cand, best, dt_start, dt_end, start_sec, end_sec, ev_overlap, mo_overlap, home_wsec, mask_signal, date_int
    gc.collect()

    return out[[user_col, "home_h3", "home_missing", "home_conf"]]

In [ ]:
# ============================================================
# 4.1 — Ejecutar Home Proxy
# ============================================================
df_homes = identify_home_location(
    stays=df_stays,
    cfg=cfg,
    paths=paths,
    h3_col="h3_id",
    verbose=True,
)

print("[Homes] shape:", df_homes.shape)
assert df_homes.duplicated(subset=[cfg.col_user]).sum() == 0, "Salida homes tiene usuarios duplicados."

# Diagnóstico agregado de la confianza del ancla.
# No se imprimen asignaciones individuales: home_h3 es un cuasi-identificador
# (ver la nota de esta sección y la discusión de privacidad del manuscrito).
_conf_ok = df_homes.loc[df_homes["home_missing"] == 0, "home_conf"]

if len(_conf_ok):
    print("[Home] distribución de home_conf (solo anclas resueltas):")
    print(_conf_ok.describe())
    print(
        f"[Home] casos con dominancia total (home_conf = 1,0): "
        f"{int((_conf_ok >= 1.0).sum()):,} de {len(_conf_ok):,} "
        f"({(_conf_ok >= 1.0).mean():.2%})"
    )
    print(
        "[Home] Nota: home_conf alto indica concentración de la evidencia "
        "disponible, no necesariamente cobertura observacional suficiente."
    )

del _conf_ok
_ = gc.collect()

In [ ]:
# ============================================================
# 4.2 — Guardar artifacts
# ============================================================
# Artefacto de Nivel 2: contiene anclas residenciales inferidas a nivel
# individual. Deriva de la fuente telco, se escribe en el directorio de
# artefactos y no forma parte de la entrega pública. Su uso queda restringido
# al entorno autorizado; no debe publicarse ni reutilizarse a nivel individual.
homes_path = paths["tables_dir"] / "homes.parquet"
df_homes.to_parquet(homes_path, index=False)

log_jsonl(paths["logs_dir"] / "home.jsonl", {"event": "home_saved", "ts_utc": now_utc(), "path": str(homes_path)})
print("[Home] Guardado en tables_dir y listo para Worksite.")

_ = gc.collect()

## *5. Identificación de Worksite Principal*

Esta sección infiere el **ancla laboral principal** de cada usuario con
residencia resuelta: la celda H3 que concentra su presencia en horario laboral
de día hábil, excluyendo explícitamente la celda residencial.

El procedimiento es homólogo al de la Sección 4 —solapamiento con una ventana,
agregación por celda, selección del máximo y validación por días y dominancia—
con tres diferencias: opera sobre una sola ventana (`work_win`), restringe la
evidencia a días hábiles, y descarta la celda ya asignada como residencia. Solo
se evalúan usuarios cuya ancla residencial fue resuelta, dado que sin ella la
exclusión no puede aplicarse.

**Esta es la sección donde se concentra la fragilidad del prototipo.** El
anclaje residencial se apoya en presencia nocturna y matinal, que es el patrón
mejor cubierto por la fuente; el anclaje laboral requiere presencia sostenida en
horario diurno de día hábil fuera del hogar, que es precisamente lo que trazas
poco densas capturan peor. La proporción de anclas laborales no resueltas es un
resultado del procedimiento y condiciona directamente la interpretación de los
segmentos de la Sección 10: la ausencia de lugar de trabajo identificable no
equivale a trabajo remoto. El análisis de sensibilidad de la Sección 13
confirma esta asimetría — el anclaje residencial es robusto a la ponderación
`alpha`, el laboral no.

Junto con el ancla de la Sección 4, el producto de esta sección constituye el
par residencia–trabajo. Rigen sobre él las mismas restricciones de uso legítimo:
no se imprimen asignaciones individuales y los artefactos persistidos no forman
parte de la entrega pública.

In [ ]:
# ============================================================
# 5 — Worksite principal + conf^W
# ============================================================
def identify_work_location(
    stays: pd.DataFrame,
    homes_df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    h3_col: str = "h3_id",
    start_col: str = "start_ts",
    end_col: str = "end_ts",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Infiere el ancla laboral principal por usuario (Módulo III: anclas
    operacionales).

    Procedimiento
    -------------
    1. Se restringe el universo a usuarios con ancla residencial resuelta
       (home_missing = 0): sin ella, la exclusión del hogar no puede aplicarse.
    2. Para cada permanencia se calcula el solapamiento, en segundos, con la
       ventana laboral (cfg.work_win).
    3. Se retienen como candidatas las permanencias que cumplen las tres
       condiciones: iniciadas en día hábil, con solapamiento positivo, y en una
       celda distinta de home_h3.
    4. Se agrega por (usuario, celda H3) el tiempo total y el número de días
       calendario distintos con evidencia.
    5. Se selecciona por usuario la celda con mayor tiempo total.
    6. La asignación es válida si days_seen >= cfg.work_min_days y
       dominance_ratio >= cfg.work_min_ratio. En caso contrario, work_missing=1.

    Salida
    ------
    work_h3       : celda laboral inferida, o NA si la asignación no es válida
    work_missing  : 1 si el ancla no pudo resolverse, 0 en caso contrario
    work_conf     : dominancia * min(days_seen / work_min_days, 1)

    Nota sobre work_conf: se calcula para todo usuario que tuvo al menos un
    candidato laboral, incluidos aquellos cuya asignación no superó la
    validación. Por lo tanto pueden coexistir work_missing = 1 y work_conf > 0:
    el valor describe la evidencia del mejor candidato, no la de una asignación
    válida. Solo debe interpretarse junto con work_missing = 0.

    Asimetría respecto del anclaje residencial
    ------------------------------------------
    El anclaje laboral es estructuralmente más frágil que el residencial. La
    presencia nocturna y matinal es el patrón mejor cubierto por trazas pasivas;
    la presencia diurna sostenida fuera del hogar en día hábil lo es menos. Una
    proporción alta de work_missing indica insuficiencia observacional y NO debe
    interpretarse como evidencia de trabajo remoto.

    Estatuto del producto
    ---------------------
    Junto con el ancla residencial, este producto constituye el par
    residencia–trabajo, cuasi-identificador reconocido en la literatura de
    movilidad. No debe publicarse, persistirse ni reutilizarse a nivel
    individual fuera del entorno autorizado.
    """
    t0 = now_utc()
    user_col = cfg.col_user

    (wk_start, wk_end) = cfg.work_win
    wk0 = wk_start.hour * 3600 + wk_start.minute * 60 + wk_start.second
    wk1 = wk_end.hour * 3600 + wk_end.minute * 60 + wk_end.second

    min_days = int(cfg.work_min_days)
    min_ratio = float(cfg.work_min_ratio)

    # Validaciones mínimas
    req_stays = {user_col, h3_col, start_col, end_col}
    miss_stays = req_stays - set(stays.columns)
    if miss_stays:
        raise ValueError(f"Faltan columnas en stays: {miss_stays}")

    req_homes = {user_col, "home_h3", "home_missing"}
    miss_homes = req_homes - set(homes_df.columns)
    if miss_homes:
        raise ValueError(f"Faltan columnas en homes_df: {miss_homes}")

    if stays.empty:
        out = homes_df[[user_col]].copy()
        out["work_h3"] = pd.NA
        out["work_missing"] = np.int8(1)
        out["work_conf"] = np.float32(0.0)
        return out[[user_col, "work_h3", "work_missing", "work_conf"]]

    # Solo usuarios con hogar válido
    valid_homes = homes_df.loc[homes_df["home_missing"] == 0, [user_col, "home_h3"]].copy()

    if valid_homes.empty:
        out = homes_df[[user_col]].copy()
        out["work_h3"] = pd.NA
        out["work_missing"] = np.int8(1)
        out["work_conf"] = np.float32(0.0)
        log_jsonl(
            paths["logs_dir"] / "work.jsonl",
            {"event": "work_no_valid_homes", "ts_utc": now_utc(), "users": int(len(out))},
        )
        if verbose:
            print(
                "[Work] Ningún usuario tiene ancla residencial resuelta: no es "
                "posible aplicar la exclusión del hogar y no se infiere ancla "
                "laboral."
            )
        return out[[user_col, "work_h3", "work_missing", "work_conf"]]

    # Subset mínimo de stays y merge con home_h3 (inner)
    s = stays[[user_col, h3_col, start_col, end_col]].merge(valid_homes, on=user_col, how="inner")

    # Guardrails datetime
    if not pd.api.types.is_datetime64_any_dtype(s[start_col]) or not pd.api.types.is_datetime64_any_dtype(s[end_col]):
        raise ValueError("start_ts/end_ts deben ser datetime64[ns].")

    # Arrays de tiempo
    start_ts = s[start_col].to_numpy(dtype="datetime64[ns]", copy=False)
    end_ts = s[end_col].to_numpy(dtype="datetime64[ns]", copy=False)
    dt_start = pd.DatetimeIndex(start_ts)
    dt_end = pd.DatetimeIndex(end_ts)

    # weekday por inicio
    dow = dt_start.dayofweek.astype(np.int8, copy=False)
    is_weekday = (dow < 5)

    # seg del día
    start_sec = (dt_start.hour.astype(np.int32) * 3600 +
                 dt_start.minute.astype(np.int32) * 60 +
                 dt_start.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = (dt_end.hour.astype(np.int32) * 3600 +
               dt_end.minute.astype(np.int32) * 60 +
               dt_end.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = np.maximum(end_sec, start_sec)

    wk_overlap = np.maximum(0, np.minimum(end_sec, wk1) - np.maximum(start_sec, wk0)).astype(np.float32)

    # Candidato laboral: weekday AND overlap>0 AND fuera de casa
    mask_cand = is_weekday & (wk_overlap > 0) & (s[h3_col] != s["home_h3"])

    if not mask_cand.any():
        out = homes_df[[user_col]].copy()
        out["work_h3"] = pd.NA
        out["work_missing"] = np.int8(1)
        out["work_conf"] = np.float32(0.0)
        log_jsonl(
            paths["logs_dir"] / "work.jsonl",
            {"event": "work_no_candidates", "ts_utc": now_utc(), "users": int(len(out))},
        )
        if verbose:
            print(
                "[Work] No hay permanencias candidatas (día hábil, dentro de la "
                "ventana laboral y fuera del hogar): no se infiere ancla laboral."
            )
        del s, dt_start, dt_end, dow, is_weekday, start_sec, end_sec, wk_overlap, mask_cand
        gc.collect()
        return out[[user_col, "work_h3", "work_missing", "work_conf"]]

    # Reducir a filas candidatas mínimas
    sc = s.loc[mask_cand, [user_col, h3_col, start_col]].copy()
    sc["work_wsec"] = wk_overlap[mask_cand]

    # days_seen: día calendario por inicio (int)
    sc["date_int"] = (
        sc[start_col]
        .to_numpy(dtype="datetime64[ns]", copy=False)
        .astype("datetime64[D]")
        .astype(np.int32, copy=False)
    )

    # Agregación por (user,h3)
    cand = (
        sc.groupby([user_col, h3_col], observed=True, sort=False)
        .agg(total_work_wsec=("work_wsec", "sum"),
             days_seen=("date_int", pd.Series.nunique))
        .reset_index()
    )

    # total por usuario (tabla pequeña)
    cand["user_total_wsec"] = cand.groupby(user_col, observed=True, sort=False)["total_work_wsec"].transform("sum")
    cand["dominance_ratio"] = (cand["total_work_wsec"] / cand["user_total_wsec"]).astype(np.float32)

    # Elegir mejor por usuario
    cand.sort_values([user_col, "total_work_wsec"], ascending=[True, False], kind="mergesort", inplace=True)
    best = cand.drop_duplicates(subset=[user_col], keep="first").copy()

    best["valid_work"] = (best["days_seen"] >= min_days) & (best["dominance_ratio"] >= min_ratio)

    # conf^W
    day_factor = np.clip(best["days_seen"].to_numpy(dtype=np.float32) / max(min_days, 1), 0.0, 1.0)
    best["work_conf"] = (best["dominance_ratio"].to_numpy(dtype=np.float32) * day_factor).astype(np.float32)

    # Salida sobre universo homes_df
    out = homes_df[[user_col]].copy()
    out = out.merge(best[[user_col, h3_col, "valid_work", "work_conf"]], on=user_col, how="left")
    out.rename(columns={h3_col: "work_h3"}, inplace=True)

    out["work_missing"] = np.where(out["valid_work"] == True, 0, 1).astype(np.int8)
    out.loc[out["work_missing"] == 1, "work_h3"] = pd.NA
    out["work_conf"] = out["work_conf"].fillna(np.float32(0.0)).astype(np.float32)

    # Tipos compactos
    try:
        out["work_h3"] = out["work_h3"].astype("category")
    except Exception:
        pass

    report = {
        "event": "work_built",
        "ts_utc": now_utc(),
        "users_total": int(len(out)),
        "users_work_ok": int((out["work_missing"] == 0).sum()),
        "users_work_missing": int((out["work_missing"] == 1).sum()),
        "users_with_valid_home": int(len(valid_homes)),
        "users_with_any_candidate": int(len(best)),
        "min_days": min_days,
        "min_ratio": min_ratio,
        "work_window": [wk_start.strftime("%H:%M"), wk_end.strftime("%H:%M")],
        "mem_mb_out": mem_mb_df(out),
        "elapsed_s": (now_utc() - t0).total_seconds(),
    }
    log_jsonl(paths["logs_dir"] / "work.jsonl", report)

    if verbose:
        print(
            f"[Work] ventana: {report['work_window'][0]}–{report['work_window'][1]} | "
            f"solo días hábiles | excluye home_h3"
        )
        print(f"[Work] criterio: días>={min_days} | dominancia>={min_ratio:.2f}")
        print(
            f"[Work] users={report['users_total']:,} | "
            f"con hogar válido={report['users_with_valid_home']:,} | "
            f"con algún candidato={report['users_with_any_candidate']:,}"
        )
        print(
            f"[Work] resueltos={report['users_work_ok']:,} "
            f"({report['users_work_ok'] / max(report['users_total'], 1):.2%}) | "
            f"missing={report['users_work_missing']:,} "
            f"({report['users_work_missing'] / max(report['users_total'], 1):.2%})"
        )

    # RAM hygiene
    del s, sc, cand, best, dt_start, dt_end, dow, is_weekday, start_sec, end_sec, wk_overlap, mask_cand
    gc.collect()

    return out[[user_col, "work_h3", "work_missing", "work_conf"]]

In [ ]:
# ============================================================
# 5.1 — Ejecutar Worksite principal
# ============================================================
df_works = identify_work_location(
    stays=df_stays,
    homes_df=df_homes,
    cfg=cfg,
    paths=paths,
    h3_col="h3_id",
    verbose=True,
)

print("[Works] shape:", df_works.shape)
assert df_works.duplicated(subset=[cfg.col_user]).sum() == 0, "Salida works tiene usuarios duplicados."

# Diagnóstico agregado. No se imprimen asignaciones individuales: el par
# (home_h3, work_h3) es un cuasi-identificador (ver la nota de la Sección 4 y
# la discusión de privacidad del manuscrito).
_n_total = int(len(df_works))
_n_missing = int((df_works["work_missing"] == 1).sum())
_conf_ok = df_works.loc[df_works["work_missing"] == 0, "work_conf"]

if _n_missing:
    print(
        f"[Work] {_n_missing:,} de {_n_total:,} usuarios "
        f"({_n_missing / max(_n_total, 1):.2%}) sin ancla laboral identificable.\n"
        "       Esto refleja insuficiencia observacional en horario laboral y "
        "NO debe interpretarse como evidencia de trabajo remoto: la ausencia "
        "de lugar de trabajo identificable y el trabajo desde el hogar son "
        "condiciones distintas que la fuente no permite separar por sí sola."
    )

# Casos con evidencia parcial: hubo candidato laboral pero no superó la
# validación por días o dominancia. Coexisten work_missing=1 y work_conf>0.
_partial = int(((df_works["work_missing"] == 1) & (df_works["work_conf"] > 0)).sum())
if _partial:
    print(
        f"[Work] {_partial:,} usuarios con evidencia laboral parcial "
        f"(candidato presente que no superó el criterio de validación)."
    )

if len(_conf_ok):
    print("[Work] distribución de work_conf (solo anclas resueltas):")
    print(_conf_ok.describe())

del _n_total, _n_missing, _conf_ok, _partial
_ = gc.collect()

In [ ]:
# ============================================================
# 5.2 — Guardar artifacts
# ============================================================
# Artefacto de Nivel 2: contiene anclas laborales inferidas a nivel individual.
# Combinado con homes.parquet completa el par residencia–trabajo, que constituye
# un cuasi-identificador. Deriva de la fuente telco, se escribe en el directorio
# de artefactos y no forma parte de la entrega pública. Su uso queda restringido
# al entorno autorizado.
works_path = paths["tables_dir"] / "works.parquet"
df_works.to_parquet(works_path, index=False)

log_jsonl(paths["logs_dir"] / "work.jsonl", {"event": "work_saved", "ts_utc": now_utc(), "path": str(works_path)})
print("[Work] Guardado en tables_dir y listo para la siguiente sección.")

_ = gc.collect()

## *6. Detección de Mobile-for-Work (Conductores)*

Esta sección identifica un patrón conductual que las anclas por sí solas no
distinguen: usuarios **sin lugar de trabajo fijo identificable pero con
movilidad amplia y dispersa**, compatible con ocupaciones cuyo desempeño ocurre
en desplazamiento.

El problema que resuelve es de ambigüedad semántica. Tras la Sección 5, un
usuario con `work_missing = 1` puede corresponder a al menos tres situaciones
distintas: trabajo desde el hogar, insuficiencia observacional, o trabajo
móvil. Las dos primeras se caracterizan por concentración espacial; la tercera,
por lo contrario. La regla explota esa diferencia.

Se construyen tres descriptores de movilidad por usuario a partir de sus
permanencias: radio de giro ponderado por duración, entropía espacial de Shannon
sobre el tiempo asignado a cada celda, y número de celdas distintas visitadas.
La marca se activa cuando concurren tres condiciones: ausencia de ancla laboral,
radio de giro por sobre un umbral calibrado sobre esa misma subpoblación, y un
mínimo de celdas distintas.

**El umbral de radio de giro es relativo, no absoluto.** Se calibra como
percentil de la distribución observada entre los usuarios sin ancla laboral, de
modo que identifica a los más móviles *dentro de la cohorte disponible*, no a
quienes superan un criterio externo de movilidad. La marca resultante es un
descriptor conductual y no una categoría ocupacional verificada.

In [ ]:
# ============================================================
# 6 — Mobile-for-Work
# ============================================================
def compute_mobility_features(
    stays: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    h3_col: str = "h3_id",
    duration_col: str = "duration_min",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Construye descriptores de movilidad por usuario a partir de sus
    permanencias (Módulo III: variables conductuales).

    Descriptores
    ------------
    radius_gyration_km
        Radio de giro ponderado por duración. Se calcula el centro de masa de
        las permanencias del usuario, ponderando cada centroide por su duración,
        y luego la raíz de la suma ponderada de distancias cuadradas al centro:
            Rg = sqrt( sum_i (w_i / W) * d(i, CM)^2 )
        Mide la dispersión espacial efectiva de la actividad observada.

    entropy_spatial
        Entropía de Shannon sobre la distribución del tiempo total entre celdas
        H3. Vale 0 cuando toda la permanencia se concentra en una sola celda y
        crece con la diversidad de destinos ponderada por su duración.

    unique_locations
        Número de celdas H3 distintas con al menos una permanencia.

    Los tres se calculan sobre el universo elegible y sobre permanencias con
    duración estrictamente positiva. Rg y entropía son sensibles al volumen
    observado: sobre usuarios con pocas permanencias tienden a valores bajos por
    construcción, lo que debe considerarse al interpretar la regla que se aplica
    en flag_mobile_workers.
    """
    t0 = now_utc()
    user_col = cfg.col_user

    required = {user_col, h3_col, "lat_mean", "lon_mean", duration_col}
    miss = required - set(stays.columns)
    if miss:
        raise ValueError(f"Faltan columnas en stays: {miss}")

    # Subset mínimo (stays es relativamente chico)
    s = stays[[user_col, h3_col, "lat_mean", "lon_mean", duration_col]].copy()

    # Tipos compactos
    if s[user_col].dtype != "category":
        s[user_col] = s[user_col].astype("category")
    s[user_col] = s[user_col].cat.remove_unused_categories()

    if s[h3_col].dtype != "category":
        s[h3_col] = s[h3_col].astype("category")
    s[h3_col] = s[h3_col].cat.remove_unused_categories()

    # Duración/coords
    s[duration_col] = pd.to_numeric(s[duration_col], errors="coerce").astype("float32")
    s["lat_mean"] = pd.to_numeric(s["lat_mean"], errors="coerce").astype("float32")
    s["lon_mean"] = pd.to_numeric(s["lon_mean"], errors="coerce").astype("float32")

    # Filtrar stays degenerados
    s = s.dropna(subset=[duration_col, "lat_mean", "lon_mean"])
    s = s[s[duration_col] > 0].copy()

    # ----------------------------
    # 6.0.1 Centro de masa ponderado por duración (CM)
    # ----------------------------
    g = s.groupby(user_col, observed=True, sort=False)

    s["_lat_w"] = (s["lat_mean"] * s[duration_col]).astype("float32")
    s["_lon_w"] = (s["lon_mean"] * s[duration_col]).astype("float32")

    cm = g.agg(
        dur_total=(duration_col, "sum"),
        lat_w_sum=("_lat_w", "sum"),
        lon_w_sum=("_lon_w", "sum"),
    )

    # evitar div0 (defensivo)
    cm["dur_total"] = cm["dur_total"].replace(0, np.nan)
    cm["lat_cm"] = (cm["lat_w_sum"] / cm["dur_total"]).astype("float32")
    cm["lon_cm"] = (cm["lon_w_sum"] / cm["dur_total"]).astype("float32")

    # mapear CM a filas (stays) sin merge pesado
    s["_lat_cm"] = s[user_col].map(cm["lat_cm"]).astype("float32")
    s["_lon_cm"] = s[user_col].map(cm["lon_cm"]).astype("float32")

    # ----------------------------
    # 6.0.2 Radius of Gyration (Rg)
    #   Rg = sqrt( sum_i (w_i/W) * dist(i,CM)^2 )
    # ----------------------------
    s["_dur_total"] = s[user_col].map(cm["dur_total"]).astype("float32")
    w_norm = (s[duration_col] / s["_dur_total"]).astype("float32")

    dist_m = geodesic_wgs84_m(
        s["lat_mean"].to_numpy(dtype=np.float64, copy=False),
        s["lon_mean"].to_numpy(dtype=np.float64, copy=False),
        s["_lat_cm"].to_numpy(dtype=np.float64, copy=False),
        s["_lon_cm"].to_numpy(dtype=np.float64, copy=False),
    )
    dist_km = (dist_m / 1000.0).astype(np.float32)

    s["_sq_w"] = ((dist_km * dist_km) * w_norm.to_numpy(np.float32, copy=False)).astype("float32")
    rg = np.sqrt(s.groupby(user_col, observed=True, sort=False)["_sq_w"].sum()).astype("float32")
    rg_df = rg.reset_index(name="radius_gyration_km")

    # ----------------------------
    # 6.0.3 Entropía espacial (Shannon) sobre tiempo en H3
    # ----------------------------
    h3_stats = (
        s.groupby([user_col, h3_col], observed=True, sort=False)[duration_col]
        .sum()
        .reset_index(name="dur_h3")
    )
    h3_stats["dur_total"] = h3_stats.groupby(user_col, observed=True, sort=False)["dur_h3"].transform("sum")

    p = (h3_stats["dur_h3"].to_numpy(np.float64, copy=False) / h3_stats["dur_total"].to_numpy(np.float64, copy=False))
    p = np.clip(p, 1e-12, 1.0)
    h3_stats["_plogp"] = (p * np.log(p)).astype("float32")

    ent = (-h3_stats.groupby(user_col, observed=True, sort=False)["_plogp"].sum()).astype("float32")
    ent_df = ent.reset_index(name="entropy_spatial")

    # ----------------------------
    # 6.0.4 Unique locations
    # ----------------------------
    uniq_df = s.groupby(user_col, observed=True, sort=False)[h3_col].nunique().astype("int32").reset_index(name="unique_locations")

    # Consolidación (tablas usuario son pequeñas)
    features = rg_df.merge(ent_df, on=user_col, how="outer").merge(uniq_df, on=user_col, how="outer")

    # Tipos compactos
    features["radius_gyration_km"] = features["radius_gyration_km"].astype("float32")
    features["entropy_spatial"] = features["entropy_spatial"].astype("float32")
    features["unique_locations"] = features["unique_locations"].fillna(0).astype("int32")

    log_jsonl(
        paths["logs_dir"] / "mobile.jsonl",
        {
            "event": "mobility_features_built",
            "ts_utc": now_utc(),
            "users": int(features[user_col].nunique()),
            "rows": int(len(features)),
            "mem_mb_features": mem_mb_df(features),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    if verbose:
        print("[Mobility] shape:", features.shape, f"| mem={mem_mb_df(features):.1f}MB")
        print(features[["radius_gyration_km", "entropy_spatial", "unique_locations"]].describe())

    # RAM hygiene
    del s, cm, w_norm, dist_m, dist_km, h3_stats, p, rg, ent, rg_df, ent_df, uniq_df
    gc.collect()

    return features

In [ ]:
def flag_mobile_workers(
    works_df: pd.DataFrame,
    mobility_df: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    rg_percentile: Optional[float] = None,   # si None: usa cfg.mobile_percentile_p
    min_unique_locs: Optional[int] = None,   # si None: usa cfg.mobile_tau_unique
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Marca usuarios con patrón de trabajo móvil (Módulo III: variables
    conductuales).

    Regla
    -----
    Un usuario se marca cuando concurren las tres condiciones:
      1. work_missing == 1            (sin ancla laboral identificable)
      2. radius_gyration_km >= thr    (movilidad amplia)
      3. unique_locations >= tau_uniq (movilidad dispersa)

    El umbral thr NO es absoluto: se calibra como percentil de la distribución
    de radio de giro observada entre los usuarios sin ancla laboral de esta
    misma corrida. En consecuencia, la regla identifica a los más móviles
    dentro de la cohorte disponible y su valor de corte depende de la fuente y
    del universo elegible resultante. Al reejecutar sobre otro insumo, el
    umbral se recalibra automáticamente y no es comparable con el de la corrida
    documentada.

    Parámetros efectivos
    --------------------
    rg_percentile y min_unique_locs toman por defecto los valores declarados en
    la configuración (cfg.mobile_percentile_p, cfg.mobile_tau_unique). Pueden
    sobrescribirse en el sitio de llamada; cuando eso ocurre, la salida lo
    señala explícitamente y el reporte registra ambos valores.

    Estatuto del producto
    ---------------------
    is_mobile_worker es un descriptor conductual derivado de la estructura
    espacial de las permanencias observadas. No constituye una categoría
    ocupacional verificada ni una inferencia validada contra fuente externa.
    """
    t0 = now_utc()
    user_col = cfg.col_user

    cfg_pct = float(cfg.mobile_percentile_p) / 100.0
    cfg_tau = int(cfg.mobile_tau_unique)

    pct_overridden = rg_percentile is not None
    tau_overridden = min_unique_locs is not None

    if rg_percentile is None:
        rg_percentile = cfg_pct
    if min_unique_locs is None:
        min_unique_locs = cfg_tau

    required_w = {user_col, "work_missing"}
    required_m = {user_col, "radius_gyration_km", "unique_locations", "entropy_spatial"}
    if required_w - set(works_df.columns):
        raise ValueError(f"works_df no tiene columnas: {required_w - set(works_df.columns)}")
    if required_m - set(mobility_df.columns):
        raise ValueError(f"mobility_df no tiene columnas: {required_m - set(mobility_df.columns)}")

    # Merge liviano (por usuario)
    df = works_df[[user_col, "work_h3", "work_missing", "work_conf"]].merge(
        mobility_df[[user_col, "radius_gyration_km", "entropy_spatial", "unique_locations"]],
        on=user_col,
        how="left",
    )

    # Calibración del umbral en work_missing=1
    no_work_rg = df.loc[df["work_missing"] == 1, "radius_gyration_km"].dropna()
    thr = float(no_work_rg.quantile(rg_percentile)) if len(no_work_rg) else float("inf")

    df["rg_threshold"] = np.float32(thr)
    df["is_mobile_worker"] = (
        (df["work_missing"] == 1) &
        (df["radius_gyration_km"] >= thr) &
        (df["unique_locations"] >= int(min_unique_locs))
    ).astype("int8")

    report = {
        "event": "mobile_flagged",
        "ts_utc": now_utc(),
        "rg_percentile": float(rg_percentile),
        "rg_percentile_from_config": float(cfg_pct),
        "rg_percentile_overridden": bool(pct_overridden),
        "min_unique_locs": int(min_unique_locs),
        "min_unique_locs_from_config": int(cfg_tau),
        "min_unique_locs_overridden": bool(tau_overridden),
        "rg_threshold": float(thr),
        "n_calibration_users": int(len(no_work_rg)),
        "n_mobile": int(df["is_mobile_worker"].sum()),
        "mem_mb_flags": mem_mb_df(df),
        "elapsed_s": (now_utc() - t0).total_seconds(),
    }
    log_jsonl(paths["logs_dir"] / "mobile.jsonl", report)

    if verbose:
        print(
            f"[Mobile Flag] umbral Rg calibrado en p{int(rg_percentile * 100)} sobre "
            f"{len(no_work_rg):,} usuarios sin ancla laboral: {thr:.4f} km"
        )
        if pct_overridden:
            print(
                f"[Mobile Flag] rg_percentile sobrescrito en el sitio de llamada: "
                f"{rg_percentile:.2f} (configuración: {cfg_pct:.2f})"
            )
        if tau_overridden:
            print(
                f"[Mobile Flag] min_unique_locs sobrescrito en el sitio de llamada: "
                f"{int(min_unique_locs)} (configuración: {cfg_tau})"
            )
        print(
            f"[Mobile Flag] tau_uniq={int(min_unique_locs)} | "
            f"marcados={int(df['is_mobile_worker'].sum()):,} de {int(len(df)):,}"
        )

    return df

In [ ]:
# ============================================================
# 6.1 — Ejecutar mobility features
# ============================================================
df_mobility = compute_mobility_features(
    stays=df_stays,
    cfg=cfg,
    paths=paths,
    h3_col="h3_id",
    verbose=True,
)

# Artefacto de Nivel 2: descriptores de movilidad a nivel individual, derivados
# de la fuente telco. Se escribe en el directorio de artefactos y no forma parte
# de la entrega pública.
mob_path = paths["tables_dir"] / "mobility_features.parquet"
df_mobility.to_parquet(mob_path, index=False)

log_jsonl(paths["logs_dir"] / "mobile.jsonl", {"event": "mobility_saved", "ts_utc": now_utc(), "path": str(mob_path)})

_ = gc.collect()

In [ ]:
# ============================================================
# 6.2 — Flag Mobile-for-Work
# ============================================================
df_flags = flag_mobile_workers(
    works_df=df_works,
    mobility_df=df_mobility,
    cfg=cfg,
    paths=paths,
    rg_percentile=None,   # usa cfg.mobile_percentile_p
    # Desviación instrumental respecto del valor de marco (cfg.mobile_tau_unique = 15),
    # declarada en el manuscrito. Justificación: con tau_unique >= 10 la regla no
    # identifica ningún caso en ninguno de los percentiles de Rg evaluados, como
    # verifica el análisis de sensibilidad de la Sección 13.2.2. El valor de marco
    # es inaplicable sobre la densidad observacional de esta fuente.
    min_unique_locs=4,
    verbose=True,
)

# Diagnóstico agregado. No se imprimen casos individuales: los descriptores de
# movilidad, combinados con las anclas de las Secciones 4 y 5, incrementan la
# especificidad del perfil por usuario.
_mob = df_flags["is_mobile_worker"] == 1
_calib = df_flags["work_missing"] == 1

print(f"[Mobile] marcados: {int(_mob.sum()):,} de {int(len(df_flags)):,} usuarios")

if int(_mob.sum()):
    print("[Mobile] descriptores de los usuarios marcados:")
    print(df_flags.loc[_mob, ["radius_gyration_km", "entropy_spatial", "unique_locations"]].describe())
    print("[Mobile] descriptores de la subpoblación de calibración (sin ancla laboral):")
    print(df_flags.loc[_calib, ["radius_gyration_km", "entropy_spatial", "unique_locations"]].describe())

del _mob, _calib
_ = gc.collect()

In [ ]:
# ============================================================
# 6.3 — Guardar flags + liberar RAM
# ============================================================
# Artefacto de Nivel 2: marcas conductuales a nivel individual, derivadas de la
# fuente telco. Se escribe en el directorio de artefactos y no forma parte de la
# entrega pública.
flags_path = paths["tables_dir"] / "mobile_flags.parquet"
df_flags.to_parquet(flags_path, index=False)

log_jsonl(paths["logs_dir"] / "mobile.jsonl", {"event": "mobile_flags_saved", "ts_utc": now_utc(), "path": str(flags_path)})

# Si no necesitas df_mobility en RAM, suelta
del df_mobility
_ = gc.collect()

print("[Section 6] OK — flags guardados y memoria liberada.")

## *7. Delimitación del Universo (Workers vs No-Workers)*

Esta sección consolida los productos de las Secciones 4 a 6 y **divide el
universo elegible en dos rutas de asignación**: usuarios que pasan al
clustering de la Sección 9 y usuarios cuyo segmento queda determinado por regla,
sin intervención del clustering.

Se construye primero `home_work_ratio`: la proporción del tiempo en ventana
laboral de día hábil que el usuario pasa en su propia celda residencial. Es el
descriptor que separa presencia domiciliaria diurna de presencia externa, y solo
se calcula donde es informativo — usuarios con hogar resuelto y sin ancla
laboral identificable.

Sobre esa base se asigna una etiqueta previa mediante reglas mutuamente
excluyentes, evaluadas en orden:

| Etiqueta | Condición | Destino |
|---|---|---|
| `MOBILE_WORKER` | marcado como móvil y `home_work_ratio` bajo | descartado del clustering |
| `WORKER_OFFICE` | ancla laboral resuelta | clustering |
| `WORKER_POTENTIAL_REMOTE` | sin ancla laboral y `home_work_ratio >= tau_remote` | clustering |
| `NO_WORKER` | ninguna de las anteriores | descartado del clustering |

La condición de `MOBILE_WORKER` incorpora `home_work_ratio` como
desambiguador: un usuario con movilidad amplia pero alta permanencia
domiciliaria diurna no corresponde al patrón de trabajo móvil, sino a movilidad
no laboral.

**Las dos etiquetas descartadas no son residuales.** `MOBILE_WORKER` y
`NO_WORKER` se excluyen del clustering porque su asignación no depende de la
estructura de las variables conductuales sino de reglas explícitas; sus
usuarios reaparecen en la consolidación final de la Sección 10 como segmentos
propios. El clustering opera exclusivamente sobre trabajadores con evidencia
laboral —resuelta o presunta— porque es entre ellos donde la distinción de
modalidad tiene sentido.

**Advertencia interpretativa.** `WORKER_POTENTIAL_REMOTE` es una etiqueta de
sospecha, no de constatación: agrupa a quienes no tienen lugar de trabajo
identificable y permanecen mayoritariamente en su celda residencial durante el
horario laboral. Dado que el 83,20 % de la cohorte carece de ancla laboral
resuelta (Sección 5), esta etiqueta absorbe tanto teletrabajo efectivo como
insuficiencia observacional. La Sección 10 no resuelve esa ambigüedad; solo la
segmenta.

In [ ]:
# ============================================================
# 7 — Delimitación del Universo (Workers vs No-Workers)
# ============================================================
USER_COL = cfg.col_user
H3_COL = "h3_id"


# ------------------------------------------------------------
# 7.0 — Base universo (merge mínimo) + home_h3 -> código H3
# ------------------------------------------------------------
def build_universe_base(
    df_flags: pd.DataFrame,
    df_homes: pd.DataFrame,
    cfg,
    paths: Dict[str, Path],
    *,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Consolida por usuario los productos de las Secciones 4 a 6 en una tabla
    única: anclas residencial y laboral con sus medidas de confianza, y la marca
    de trabajo móvil.

    Los indicadores de ausencia (home_missing, work_missing) se rellenan con 1
    ante cualquier faltante del merge, de modo que la ausencia de información se
    trate siempre como ancla no resuelta y nunca como ancla válida.
    """
    t0 = now_utc()
    user_col = cfg.col_user

    # Columnas mínimas requeridas
    req_flags = {user_col, "work_h3", "work_missing", "work_conf", "is_mobile_worker"}
    req_homes = {user_col, "home_h3", "home_missing", "home_conf"}

    miss_f = req_flags - set(df_flags.columns)
    miss_h = req_homes - set(df_homes.columns)
    if miss_f:
        raise ValueError(f"df_flags missing cols: {miss_f}")
    if miss_h:
        raise ValueError(f"df_homes missing cols: {miss_h}")

    base = (
        df_flags[[user_col, "work_h3", "work_missing", "work_conf", "is_mobile_worker"]]
        .merge(df_homes[[user_col, "home_h3", "home_missing", "home_conf"]], on=user_col, how="left")
    )

    # user como category (RAM)
    if base[user_col].dtype != "category":
        try:
            base[user_col] = base[user_col].astype("category")
        except Exception:
            pass

    # Tipos compactos. Los indicadores de ausencia se rellenan con 1:
    # ante un faltante, el ancla se considera no resuelta.
    for c in ["work_missing", "is_mobile_worker", "home_missing"]:
        if c in base.columns:
            base[c] = base[c].fillna(1).astype(np.int8)

    for c in ["work_conf", "home_conf"]:
        if c in base.columns:
            base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0).astype(np.float32)

    log_jsonl(
        paths["logs_dir"] / "universe.jsonl",
        {
            "event": "universe_base_built",
            "ts_utc": now_utc(),
            "users": int(base[user_col].nunique()),
            "home_missing_rate": float(base["home_missing"].mean()) if "home_missing" in base.columns else None,
            "work_missing_rate": float(base["work_missing"].mean()) if "work_missing" in base.columns else None,
            "mem_mb_base": mem_mb_df(base),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    if verbose:
        print("[7.0] df_universe_base:", base.shape, f"| mem={mem_mb_df(base):.1f}MB")

    gc.collect()
    return base


df_universe_base = build_universe_base(df_flags, df_homes, cfg, paths, verbose=True)

# Umbral de permanencia domiciliaria en horario laboral.
# Declarado en cfg.tau_remote; se admite override vía cfg.remote_threshold_ratio.
REMOTE_THR = float(getattr(cfg, "remote_threshold_ratio", cfg.tau_remote))
print("[7] tau_remote efectivo:", REMOTE_THR)

# Asegurar stays H3 category (para codes)
if df_stays[H3_COL].dtype != "category":
    df_stays[H3_COL] = df_stays[H3_COL].astype("category")

# Mapeo home_h3 -> código dentro del catálogo de categorías de df_stays[H3_COL].
# La comparación posterior (Sección 7.1) opera sobre códigos enteros y no sobre
# cadenas: comparar directamente una columna 'category' contra otra con distinto
# catálogo produce resultados incorrectos de forma silenciosa.
# Un código -1 indica una celda residencial ausente del catálogo de stays, lo que
# solo puede ocurrir si el ancla no está resuelta.
h3_cats = df_stays[H3_COL].cat.categories
home_vals = df_universe_base.set_index(USER_COL)["home_h3"]
home_codes = h3_cats.get_indexer(home_vals.to_numpy(dtype=object, copy=False)).astype(np.int32)
home_code_map = pd.Series(home_codes, index=home_vals.index, name="home_h3_code")

n_home_unmapped = int((home_code_map.to_numpy() < 0).sum())

del home_vals, home_codes
_ = gc.collect()

log_jsonl(
    paths["logs_dir"] / "universe.jsonl",
    {
        "event": "home_code_map_ready",
        "ts_utc": now_utc(),
        "users_in_map": int(home_code_map.shape[0]),
        "home_codes_missing_neg1": n_home_unmapped,
    },
)

print(
    f"[7.0] mapeo H3 de anclas residenciales: {int(home_code_map.shape[0]):,} usuarios | "
    f"sin correspondencia en el catálogo: {n_home_unmapped:,}"
)

In [ ]:
# ------------------------------------------------------------
# 7.1 — home_work_ratio en ventana laboral (solo cuando se necesita)
# ------------------------------------------------------------
def compute_home_work_ratio(
    df_universe_base: pd.DataFrame,
    df_stays: pd.DataFrame,
    home_code_map: pd.Series,
    cfg,
    paths: Dict[str, Path],
    *,
    h3_col: str = "h3_id",
    verbose: bool = True,
) -> pd.Series:
    """
    Calcula, por usuario, la proporción del tiempo en ventana laboral de día
    hábil que transcurre en su propia celda residencial.

        home_work_ratio = tiempo en home_h3 dentro de la ventana laboral
                          -------------------------------------------------
                          tiempo total en ventana laboral

    Se calcula únicamente para usuarios con hogar resuelto (home_missing = 0) y
    sin ancla laboral identificable (work_missing = 1): es el subconjunto donde
    el descriptor es informativo. Para el resto, el valor por defecto de 0 se
    asigna en el sitio de llamada — para quienes tienen ancla laboral resuelta,
    la etiqueta previa se decide por esa ancla y no por este ratio.

    El denominador se acota inferiormente en 1 segundo para evitar división por
    cero. Dado que las permanencias tienen duración mínima declarada, el caso no
    se materializa en la práctica; la cota es defensiva.

    La comparación con la celda residencial se hace sobre códigos enteros del
    catálogo de categorías de df_stays (ver Sección 7.0), no sobre cadenas.
    """
    t0 = now_utc()
    user_col = cfg.col_user

    (wk_start, wk_end) = cfg.work_win
    wk0 = wk_start.hour * 3600 + wk_start.minute * 60 + wk_start.second
    wk1 = wk_end.hour * 3600 + wk_end.minute * 60 + wk_end.second

    mask_need = (
        (df_universe_base["work_missing"] == 1) &
        (df_universe_base["home_missing"] == 0)
    )
    users_need = df_universe_base.loc[mask_need, user_col]
    n_need = int(users_need.nunique())

    if verbose:
        print(
            f"[7.1] usuarios que requieren home_work_ratio "
            f"(hogar resuelto y sin ancla laboral): {n_need:,}"
        )

    if n_need == 0:
        log_jsonl(
            paths["logs_dir"] / "universe.jsonl",
            {
                "event": "home_work_ratio",
                "ts_utc": now_utc(),
                "users_need_ratio": 0,
                "note": "no users need ratio",
            },
        )
        return pd.Series(dtype=np.float32)

    # Subset stays (mínimo)
    s = df_stays.loc[df_stays[user_col].isin(users_need), [user_col, h3_col, "start_ts", "end_ts"]].copy()

    # Arrays tiempo
    start_ts = s["start_ts"].to_numpy(dtype="datetime64[ns]", copy=False)
    end_ts = s["end_ts"].to_numpy(dtype="datetime64[ns]", copy=False)
    dt_start = pd.DatetimeIndex(start_ts)
    dt_end = pd.DatetimeIndex(end_ts)

    # weekday por día calendario (int) — robusto y rápido
    date_int = start_ts.astype("datetime64[D]").astype(np.int32, copy=False)
    dow = (date_int + 3) % 7
    is_weekday = (dow < 5)

    # seg del día
    start_sec = (dt_start.hour.astype(np.int32) * 3600 +
                 dt_start.minute.astype(np.int32) * 60 +
                 dt_start.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = (dt_end.hour.astype(np.int32) * 3600 +
               dt_end.minute.astype(np.int32) * 60 +
               dt_end.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = np.maximum(end_sec, start_sec)

    wsec = np.maximum(0, np.minimum(end_sec, wk1) - np.maximum(start_sec, wk0)).astype(np.float32)
    mask_lab = is_weekday & (wsec > 0)

    if not mask_lab.any():
        if verbose:
            print("[7.1] Ninguna permanencia solapa la ventana laboral en día hábil.")
        log_jsonl(
            paths["logs_dir"] / "universe.jsonl",
            {
                "event": "home_work_ratio",
                "ts_utc": now_utc(),
                "users_need_ratio": n_need,
                "note": "no labor-overlap stays",
            },
        )
        del s, dt_start, dt_end, date_int, dow, is_weekday, start_sec, end_sec, wsec, mask_lab
        gc.collect()
        return pd.Series(dtype=np.float32)

    s = s.loc[mask_lab, [user_col, h3_col]].copy()
    s["_wsec"] = wsec[mask_lab]

    # Comparación con la celda residencial sobre códigos H3
    h3_codes = s[h3_col].cat.codes.to_numpy(np.int32, copy=False)
    home_codes_row = s[user_col].map(home_code_map).to_numpy(np.int32, copy=False)
    s["_is_home"] = (h3_codes == home_codes_row)

    tot = s.groupby(user_col, observed=True, sort=False)["_wsec"].sum()
    home = s.loc[s["_is_home"]].groupby(user_col, observed=True, sort=False)["_wsec"].sum()

    ratio = (home.reindex(tot.index, fill_value=0.0) / tot.clip(lower=1.0)).astype(np.float32)

    log_jsonl(
        paths["logs_dir"] / "universe.jsonl",
        {
            "event": "home_work_ratio",
            "ts_utc": now_utc(),
            "users_need_ratio": n_need,
            "users_computed": int(ratio.shape[0]),
            "work_window": [wk_start.strftime("%H:%M"), wk_end.strftime("%H:%M")],
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    if verbose:
        print(f"[7.1] home_work_ratio calculado para {int(ratio.shape[0]):,} usuarios")

    # RAM hygiene
    del s, dt_start, dt_end, date_int, dow, is_weekday, start_sec, end_sec, wsec, mask_lab, h3_codes, home_codes_row, tot, home
    gc.collect()

    return ratio


# Inicializar home_work_ratio = 0 y rellenar solo donde exista ratio calculado.
# El valor 0 para usuarios con ancla laboral resuelta no es un dato faltante:
# su etiqueta previa se decide por la presencia del ancla, no por este ratio.
df_universe_base["home_work_ratio"] = np.float32(0.0)

ratio_u = compute_home_work_ratio(
    df_universe_base, df_stays, home_code_map, cfg, paths, h3_col=H3_COL, verbose=True
)

if not ratio_u.empty:
    df_universe_base["home_work_ratio"] = (
        df_universe_base[USER_COL].map(ratio_u).fillna(np.float32(0.0)).astype(np.float32)
    )

In [ ]:
# ------------------------------------------------------------
# 7.2 — Etiquetado PRE-CLUSTERING + split final
# ------------------------------------------------------------
# Reglas mutuamente excluyentes, evaluadas en el orden declarado.
#
# MOBILE_WORKER exige DOS condiciones: la marca conductual de la Sección 6 y una
# permanencia domiciliaria diurna baja. La segunda actúa como desambiguador: un
# usuario con movilidad amplia pero alta presencia en su hogar durante el
# horario laboral no corresponde al patrón de trabajo móvil.
#
# NO_WORKER es la categoría por defecto: recoge a quienes no tienen ancla
# laboral, no fueron marcados como móviles y tampoco alcanzan el umbral de
# permanencia domiciliaria. Es una etiqueta de ausencia de evidencia laboral,
# no de inactividad constatada.
cond_mobile_raw = (df_universe_base["is_mobile_worker"] == 1)
cond_mobile = cond_mobile_raw & (df_universe_base["home_work_ratio"] < REMOTE_THR)
cond_office = (~cond_mobile) & (df_universe_base["work_missing"] == 0)
cond_remote = (~cond_mobile) & (df_universe_base["work_missing"] == 1) & (df_universe_base["home_work_ratio"] >= REMOTE_THR)

df_universe_base["pre_label"] = np.select(
    [cond_mobile, cond_office, cond_remote],
    ["MOBILE_WORKER", "WORKER_OFFICE", "WORKER_POTENTIAL_REMOTE"],
    default="NO_WORKER",
)

# Tipos compactos
df_universe_base["pre_label"] = df_universe_base["pre_label"].astype("category")

# Descartados del clustering: su segmento queda determinado por regla y
# reaparecen en la consolidación final de la Sección 10.
df_discarded = (
    df_universe_base.loc[df_universe_base["pre_label"].isin(["MOBILE_WORKER", "NO_WORKER"]), [USER_COL, "pre_label"]]
    .rename(columns={"pre_label": "final_label"})
    .reset_index(drop=True)
)
df_discarded["final_label"] = df_discarded["final_label"].astype("category")

# Candidatos al clustering: trabajadores con evidencia laboral, resuelta o presunta.
df_clustering_candidates = (
    df_universe_base.loc[df_universe_base["pre_label"].isin(["WORKER_OFFICE", "WORKER_POTENTIAL_REMOTE"])].copy()
    .reset_index(drop=True)
)

valid_clustering_users = df_clustering_candidates[USER_COL]  # Series liviana

_n_total = int(len(df_universe_base))
_counts = df_universe_base["pre_label"].value_counts(dropna=False)

print("[7.2] distribución de etiquetas previas:")
for _lab, _n in _counts.items():
    print(f"       {str(_lab):<24} {int(_n):>6,}  ({_n / max(_n_total, 1):.2%})")

print(
    f"[7.2] al clustering: {len(df_clustering_candidates):,} | "
    f"asignados por regla: {len(df_discarded):,} | "
    f"total: {_n_total:,}"
)

# Nota de lectura: una proporción alta de WORKER_POTENTIAL_REMOTE refleja, en
# primer lugar, la proporción de usuarios sin ancla laboral resuelta. No debe
# leerse como prevalencia de teletrabajo.
if "WORKER_POTENTIAL_REMOTE" in _counts.index:
    _pr = int(_counts["WORKER_POTENTIAL_REMOTE"]) / max(_n_total, 1)
    if _pr > 0.5:
        print(
            f"[7.2] WORKER_POTENTIAL_REMOTE concentra el {_pr:.2%} del universo. "
            "Esta etiqueta agrupa teletrabajo efectivo e insuficiencia "
            "observacional en horario laboral, condiciones que la fuente no "
            "permite separar."
        )

log_jsonl(
    paths["logs_dir"] / "universe.jsonl",
    {
        "event": "universe_labeled",
        "ts_utc": now_utc(),
        "n_total": _n_total,
        "n_mobile": int((df_universe_base["pre_label"] == "MOBILE_WORKER").sum()),
        "n_office": int((df_universe_base["pre_label"] == "WORKER_OFFICE").sum()),
        "n_remote": int((df_universe_base["pre_label"] == "WORKER_POTENTIAL_REMOTE").sum()),
        "n_nowork": int((df_universe_base["pre_label"] == "NO_WORKER").sum()),
        "n_clustering_candidates": int(len(df_clustering_candidates)),
        "n_discarded": int(len(df_discarded)),
        "remote_thr": float(REMOTE_THR),
        "mem_mb_universe": mem_mb_df(df_universe_base),
        "mem_mb_candidates": mem_mb_df(df_clustering_candidates),
    },
)

del _n_total, _counts
_ = gc.collect()

In [ ]:
# ------------------------------------------------------------
# 7.3 — Guardar artifacts + liberar RAM
# ------------------------------------------------------------
# Artefactos de Nivel 2: contienen anclas inferidas y etiquetas conductuales a
# nivel individual, derivadas de la fuente telco. Se escriben en el directorio
# de artefactos y no forman parte de la entrega pública.
full_path = paths["tables_dir"] / "universe_full.parquet"
disc_path = paths["tables_dir"] / "universe_discarded.parquet"
cand_path = paths["tables_dir"] / "universe_clustering_candidates.parquet"

df_universe_base.to_parquet(full_path, index=False)
df_discarded.to_parquet(disc_path, index=False)
df_clustering_candidates.to_parquet(cand_path, index=False)

log_jsonl(
    paths["logs_dir"] / "universe.jsonl",
    {
        "event": "universe_saved",
        "ts_utc": now_utc(),
        "full": str(full_path),
        "discarded": str(disc_path),
        "candidates": str(cand_path),
    },
)

# RAM hygiene: soltar df_flags si ya no se usa
del df_flags
_ = gc.collect()

print("[Section 7] OK — artifacts guardados y RAM liberada.")

## *8. Extracción de Features para Segmentación*

Esta sección construye el vector de variables conductuales sobre el que operará
el clustering de la Sección 9. Trabaja exclusivamente sobre los candidatos
definidos en la Sección 7.

Las variables se organizan en dos conjuntos:

**BASE5** — descriptores construidos en las secciones previas de este cuaderno:
`home_work_ratio`, `radius_gyration_km`, `entropy_spatial`, `n_moves` y
`unique_locations`.

**TELEWORK_PAPER** — ocho variables adaptadas de la literatura sobre patrones de
movilidad de teletrabajadores: duración diaria en el hogar y en el lugar
primario (`DHD`, `DPD`), tiempo de viaje en día hábil y en fin de semana
(`WDTT`, `WETT`), y recuentos de ocurrencia en el hogar, el lugar primario, los
días con viaje y el lugar secundario (`HO`, `PO`, `TO`, `SO`).

`FEATURE_SET` selecciona cuál de los dos conjuntos —o su unión, `COMBINED`—
alimenta el clustering. Las variables no seleccionadas se conservan como
variables de auditoría: se calculan y persisten, pero no participan del
entrenamiento. Esta separación es la que permite el análisis de ablación
declarado en el manuscrito.

**Nota sobre la adaptación.** Las ocho variables de TELEWORK_PAPER son
adaptaciones, no réplicas: se construyen sobre permanencias derivadas de trazas
pasivas y no sobre las representaciones del trabajo original. Los nombres se
conservan por trazabilidad conceptual con la fuente; su semántica operacional es
la que define este cuaderno.

**Nota sobre los ceros.** Varias variables se calculan sobre subconjuntos: los
descriptores de desplazamiento solo existen para usuarios con al menos un
*move*, y la ocurrencia secundaria solo para quienes registran permanencias
fuera de sus anclas. Al consolidar, esos faltantes se rellenan con cero. El cero
resultante indica ausencia de evidencia observada, no ausencia constatada del
fenómeno.

In [ ]:
# ============================================================
# 8.0 — Setup + mappings home/work H3 -> códigos + feature_set
# ============================================================
USER_COL = cfg.col_user
H3_COL = "h3_id"

# --- Control experimental (feature ablation) ---
# Conjunto de variables que alimenta el clustering de la Sección 9.
#   "BASE5"          : descriptores construidos en las Secciones 4 a 7
#   "TELEWORK_PAPER" : ocho variables adaptadas de la literatura de teletrabajo
#   "COMBINED"       : unión de ambos (13 variables)
# Las variables no seleccionadas se conservan como variables de auditoría.
# La configuración documentada en el manuscrito emplea COMBINED.
FEATURE_SET = "COMBINED"

# Guardrail: columnas mínimas en df_clustering_candidates
required_cand = {USER_COL, "pre_label", "home_h3", "work_h3", "home_conf", "work_conf", "home_work_ratio"}
missing_cand = required_cand - set(df_clustering_candidates.columns)
if missing_cand:
    raise ValueError(f"df_clustering_candidates missing required cols: {missing_cand}")

users_clust = df_clustering_candidates[USER_COL]
n_users_clust = int(users_clust.nunique())

print("[8.0] users_clust:", n_users_clust)
print("[8.0] FEATURE_SET:", FEATURE_SET)

# Asegurar categorías para eficiencia
if df_stays[USER_COL].dtype != "category":
    df_stays[USER_COL] = df_stays[USER_COL].astype("category")
if df_stays[H3_COL].dtype != "category":
    df_stays[H3_COL] = df_stays[H3_COL].astype("category")

h3_cats = df_stays[H3_COL].cat.categories

# Mapeo de anclas a códigos del catálogo de categorías de df_stays[H3_COL].
# Como en la Sección 7.0, la comparación posterior opera sobre códigos enteros
# y no sobre cadenas. Un código -1 indica ancla no resuelta: las máscaras de la
# Sección 8.1 lo excluyen explícitamente.
cand_idx = df_clustering_candidates.set_index(USER_COL, drop=False)
home_vals = cand_idx["home_h3"].to_numpy(dtype=object, copy=False)
work_vals = cand_idx["work_h3"].to_numpy(dtype=object, copy=False)

home_code_map = pd.Series(h3_cats.get_indexer(home_vals).astype(np.int32), index=cand_idx.index, name="home_code")
work_code_map = pd.Series(h3_cats.get_indexer(work_vals).astype(np.int32), index=cand_idx.index, name="work_code")

n_home_neg1 = int((home_code_map.to_numpy() < 0).sum())
n_work_neg1 = int((work_code_map.to_numpy() < 0).sum())

# Ventana laboral (segundos)
wk_start, wk_end = cfg.work_win
wk0 = wk_start.hour * 3600 + wk_start.minute * 60 + wk_start.second
wk1 = wk_end.hour * 3600 + wk_end.minute * 60 + wk_end.second

print(
    f"[8.0] anclas sin correspondencia en el catálogo H3: "
    f"home={n_home_neg1:,} | work={n_work_neg1:,} (corresponden a anclas no resueltas)"
)

log_jsonl(
    paths["logs_dir"] / "features.jsonl",
    {
        "event": "section8_setup",
        "ts_utc": now_utc(),
        "users_clust": n_users_clust,
        "feature_set": FEATURE_SET,
        "work_window": [wk_start.strftime("%H:%M"), wk_end.strftime("%H:%M")],
        "home_codes_neg1": n_home_neg1,
        "work_codes_neg1": n_work_neg1,
    },
)

del home_vals, work_vals
_ = gc.collect()

In [ ]:
# ============================================================
# 8.1 — Time allocation laboral (weekdays, ventana 09–17)
# Produce:
#   - df_timealloc: totales (min) y ratios (home/work/other) en ventana laboral
#   - s_lab: stays laborales filtrados con _lab_wsec y flags (_is_home/_is_work/_is_other)
#
# Cada permanencia en ventana laboral se asigna a exactamente una de tres
# categorías: hogar, lugar de trabajo, u otro lugar. La partición es exhaustiva
# y mutuamente excluyente por construcción, de modo que los tres ratios suman 1.
# Las permanencias de usuarios sin ancla resuelta caen en "other": el código -1
# del mapeo nunca coincide con un código H3 válido.
#
# s_lab se conserva vivo: la Sección 8.2 lo consume.
# ============================================================

# Subset stays mínimo
s = df_stays.loc[df_stays[USER_COL].isin(users_clust), [USER_COL, H3_COL, "start_ts", "end_ts"]].copy()

start_ts = s["start_ts"].to_numpy(dtype="datetime64[ns]", copy=False)
end_ts = s["end_ts"].to_numpy(dtype="datetime64[ns]", copy=False)
dt_start = pd.DatetimeIndex(start_ts)
dt_end = pd.DatetimeIndex(end_ts)

# weekday por día calendario (robusto)
date_int = start_ts.astype("datetime64[D]").astype(np.int32, copy=False)
dow = (date_int + 3) % 7
is_weekday = (dow < 5)

# seg del día
start_sec = (dt_start.hour.astype(np.int32) * 3600 +
             dt_start.minute.astype(np.int32) * 60 +
             dt_start.second.astype(np.int32)).astype(np.int32, copy=False)
end_sec = (dt_end.hour.astype(np.int32) * 3600 +
           dt_end.minute.astype(np.int32) * 60 +
           dt_end.second.astype(np.int32)).astype(np.int32, copy=False)
end_sec = np.maximum(end_sec, start_sec)

lab_wsec = np.maximum(0, np.minimum(end_sec, wk1) - np.maximum(start_sec, wk0)).astype(np.float32)
mask_lab = is_weekday & (lab_wsec > 0)

s_lab = s.loc[mask_lab, [USER_COL, H3_COL, "start_ts"]].copy()
s_lab["_lab_wsec"] = lab_wsec[mask_lab]

# Códigos H3 fila y home/work por usuario.
# La condición (codes != -1) impide que un ancla no resuelta se empareje con
# alguna celda por coincidencia de código.
h3_codes = s_lab[H3_COL].cat.codes.to_numpy(np.int32, copy=False)
home_codes_row = s_lab[USER_COL].map(home_code_map).to_numpy(np.int32, copy=False)
work_codes_row = s_lab[USER_COL].map(work_code_map).to_numpy(np.int32, copy=False)

s_lab["_is_home"] = (h3_codes == home_codes_row) & (home_codes_row != -1)
s_lab["_is_work"] = (h3_codes == work_codes_row) & (work_codes_row != -1)
s_lab["_is_other"] = ~(s_lab["_is_home"] | s_lab["_is_work"])

g = s_lab.groupby(USER_COL, observed=True, sort=False)
tot = g["_lab_wsec"].sum().astype(np.float32)
home = s_lab.loc[s_lab["_is_home"]].groupby(USER_COL, observed=True, sort=False)["_lab_wsec"].sum().astype(np.float32)
work = s_lab.loc[s_lab["_is_work"]].groupby(USER_COL, observed=True, sort=False)["_lab_wsec"].sum().astype(np.float32)
other = s_lab.loc[s_lab["_is_other"]].groupby(USER_COL, observed=True, sort=False)["_lab_wsec"].sum().astype(np.float32)

df_timealloc = pd.DataFrame({USER_COL: tot.index})
df_timealloc["total_lab_wmin"] = (tot.values / 60.0).astype(np.float32)
df_timealloc["home_lab_wmin"] = (home.reindex(tot.index, fill_value=0.0).values / 60.0).astype(np.float32)
df_timealloc["work_lab_wmin"] = (work.reindex(tot.index, fill_value=0.0).values / 60.0).astype(np.float32)
df_timealloc["other_lab_wmin"] = (other.reindex(tot.index, fill_value=0.0).values / 60.0).astype(np.float32)

den = np.clip(df_timealloc["total_lab_wmin"].to_numpy(np.float32, copy=False), 1e-6, None)
df_timealloc["home_lab_ratio"] = (df_timealloc["home_lab_wmin"].to_numpy(np.float32, copy=False) / den).astype(np.float32)
df_timealloc["work_lab_ratio"] = (df_timealloc["work_lab_wmin"].to_numpy(np.float32, copy=False) / den).astype(np.float32)
df_timealloc["other_lab_ratio"] = (df_timealloc["other_lab_wmin"].to_numpy(np.float32, copy=False) / den).astype(np.float32)

print(f"[8.1] df_timealloc: {df_timealloc.shape} | de {n_users_clust:,} candidatos")
print(df_timealloc[["total_lab_wmin", "home_lab_ratio", "work_lab_ratio", "other_lab_ratio"]].describe())

# cleanup de arrays intermedios (mantener s_lab)
del s, dt_start, dt_end, date_int, dow, is_weekday, start_sec, end_sec, lab_wsec, mask_lab
del h3_codes, home_codes_row, work_codes_row, g, tot, home, work, other
_ = gc.collect()

In [ ]:
# ============================================================
# 8.2 — Features por día laboral
# Produce:
#   - df_daily: n_workdays_obs, n_days_home_lab (HO), n_days_work_lab (PO),
#               tasas de presencia, promedio y desviación de minutos por día
#   - df_secondary: SO (Secondary Occurrence) por usuario
#
# HO y PO cuentan DÍAS con evidencia, no permanencias: varias permanencias en la
# misma celda el mismo día contribuyen una sola vez. Las tasas de presencia
# normalizan por el número de días laborales efectivamente observados del
# usuario, no por el período completo.
# ============================================================

# work_date robusto: cast vía astype("datetime64[D]")
work_date = (
    s_lab["start_ts"]
    .to_numpy(dtype="datetime64[ns]", copy=False)
    .astype("datetime64[D]")
)
s_lab["work_date"] = work_date

g_ud = s_lab.groupby([USER_COL, "work_date"], observed=True, sort=False)
day_tot = g_ud["_lab_wsec"].sum().astype(np.float32).rename("lab_wsec_day")
day_home = s_lab.loc[s_lab["_is_home"]].groupby([USER_COL, "work_date"], observed=True, sort=False)["_lab_wsec"].sum().astype(np.float32).rename("home_wsec_day")
day_work = s_lab.loc[s_lab["_is_work"]].groupby([USER_COL, "work_date"], observed=True, sort=False)["_lab_wsec"].sum().astype(np.float32).rename("work_wsec_day")

df_day = day_tot.reset_index()
df_day = df_day.merge(day_home.reset_index(), on=[USER_COL, "work_date"], how="left")
df_day = df_day.merge(day_work.reset_index(), on=[USER_COL, "work_date"], how="left")
df_day[["home_wsec_day", "work_wsec_day"]] = df_day[["home_wsec_day", "work_wsec_day"]].fillna(0.0).astype(np.float32)

df_day["has_home_lab"] = (df_day["home_wsec_day"] > 0).astype(np.int8)
df_day["has_work_lab"] = (df_day["work_wsec_day"] > 0).astype(np.int8)

g_u = df_day.groupby(USER_COL, observed=True, sort=False)
df_daily = g_u.agg(
    n_workdays_obs=("work_date", "nunique"),
    n_days_home_lab=("has_home_lab", "sum"),   # HO proxy
    n_days_work_lab=("has_work_lab", "sum"),   # PO proxy
    avg_lab_wmin_per_day=("lab_wsec_day", lambda x: float(np.mean(x) / 60.0)),
    std_lab_wmin_per_day=("lab_wsec_day", lambda x: float(np.std(x, ddof=0) / 60.0)),
).reset_index()

den_days = np.clip(df_daily["n_workdays_obs"].to_numpy(np.float32, copy=False), 1.0, None)
df_daily["workday_presence_rate"] = (df_daily["n_days_work_lab"].to_numpy(np.float32, copy=False) / den_days).astype(np.float32)
df_daily["home_workday_presence_rate"] = (df_daily["n_days_home_lab"].to_numpy(np.float32, copy=False) / den_days).astype(np.float32)
df_daily["avg_lab_wmin_per_day"] = df_daily["avg_lab_wmin_per_day"].astype(np.float32)
df_daily["std_lab_wmin_per_day"] = df_daily["std_lab_wmin_per_day"].astype(np.float32)

print("[8.2] df_daily:", df_daily.shape)
print(df_daily[["n_workdays_obs", "workday_presence_rate", "home_workday_presence_rate", "avg_lab_wmin_per_day", "std_lab_wmin_per_day"]].describe())

# --- Secondary Occurrence (SO) ---
# secondary_h3: la celda no-hogar y no-trabajo con mayor tiempo acumulado en
# ventana laboral de día hábil. SO cuenta los días distintos con evidencia en
# esa celda. Solo existe para usuarios con al menos una permanencia laboral
# fuera de sus anclas; para el resto se rellena con 0 en la consolidación.
s_other = s_lab.loc[s_lab["_is_other"], [USER_COL, H3_COL, "work_date", "_lab_wsec"]].copy()

if s_other.empty:
    df_secondary = pd.DataFrame({USER_COL: df_daily[USER_COL].copy()})
    df_secondary["secondary_h3"] = pd.NA
    df_secondary["secondary_occurrence"] = np.int32(0)
else:
    # total por (user, h3) para elegir secondary
    agg_other = (
        s_other.groupby([USER_COL, H3_COL], observed=True, sort=False)["_lab_wsec"]
        .sum()
        .reset_index(name="other_wsec_total")
    )
    agg_other.sort_values([USER_COL, "other_wsec_total"], ascending=[True, False], kind="mergesort", inplace=True)
    best_other = agg_other.drop_duplicates(subset=[USER_COL], keep="first").copy()
    best_other.rename(columns={H3_COL: "secondary_h3"}, inplace=True)

    # SO: #días con evidencia en secondary_h3
    s_other = s_other.merge(best_other[[USER_COL, "secondary_h3"]], on=USER_COL, how="inner")
    s_other["_is_secondary"] = (s_other[H3_COL] == s_other["secondary_h3"])

    so_days = (
        s_other.loc[s_other["_is_secondary"]]
        .groupby(USER_COL, observed=True, sort=False)["work_date"]
        .nunique()
        .astype(np.int32)
        .reset_index(name="secondary_occurrence")
    )

    df_secondary = best_other[[USER_COL, "secondary_h3"]].merge(so_days, on=USER_COL, how="left")
    df_secondary["secondary_occurrence"] = df_secondary["secondary_occurrence"].fillna(0).astype(np.int32)

print(
    f"[8.2] df_secondary: {df_secondary.shape} | "
    f"{len(df_secondary):,} de {n_users_clust:,} candidatos registran lugar secundario"
)
print(df_secondary["secondary_occurrence"].describe())

del df_day, day_tot, day_home, day_work, g_ud, g_u, work_date, s_other
_ = gc.collect()

In [ ]:
# ============================================================
# 8.3 — Mobility features (desde parquet)
# Se releen desde el artefacto de la Sección 6.1 en lugar de recalcularse, de
# modo que el vector de entrenamiento use exactamente los mismos valores que
# alimentaron la regla de trabajo móvil.
# ============================================================
mob_path = paths["tables_dir"] / "mobility_features.parquet"
df_mob = pd.read_parquet(mob_path, columns=[USER_COL, "radius_gyration_km", "entropy_spatial", "unique_locations"])
df_mob = df_mob.loc[df_mob[USER_COL].isin(users_clust)].copy()

df_mob["radius_gyration_km"] = df_mob["radius_gyration_km"].astype(np.float32)
df_mob["entropy_spatial"] = df_mob["entropy_spatial"].astype(np.float32)
df_mob["unique_locations"] = df_mob["unique_locations"].astype(np.int32)

print("[8.3] df_mob:", df_mob.shape)
print(df_mob[["radius_gyration_km", "entropy_spatial", "unique_locations"]].describe())

_ = gc.collect()

In [ ]:
# ============================================================
# 8.4 — Moves features + Telework-paper travel features (WDTT/WETT/TO)
# Produce:
#   - df_moves_feat: n_moves + estadísticos de distancia y duración
#   - df_travel: TO (días con viaje), WDTT y WETT (minutos de viaje en día
#     hábil y en fin de semana)
#
# Ambos se calculan solo sobre usuarios con al menos un desplazamiento válido
# según el criterio de la Sección 2.3. La consolidación de la Sección 8.5
# rellena con 0 a los restantes.
# ============================================================

# Moves subset (usuarios clust)
m = df_moves.loc[df_moves[USER_COL].isin(users_clust), [USER_COL, "move_distance_m", "move_duration_min", "move_start_ts"]].copy()
m["move_distance_km"] = (pd.to_numeric(m["move_distance_m"], errors="coerce") / 1000.0).astype(np.float32)
m["move_duration_min"] = pd.to_numeric(m["move_duration_min"], errors="coerce").astype(np.float32)

g = m.groupby(USER_COL, observed=True, sort=False)
df_moves_feat = g.agg(
    n_moves=("move_distance_km", "size"),
    mean_move_distance_km=("move_distance_km", "mean"),
    mean_move_duration_min=("move_duration_min", "mean"),
).reset_index()

p90 = g["move_distance_km"].quantile(0.90).astype(np.float32).reset_index(name="p90_move_distance_km")
df_moves_feat = df_moves_feat.merge(p90, on=USER_COL, how="left")

df_moves_feat["n_moves"] = df_moves_feat["n_moves"].astype(np.int32)
for c in ["mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min"]:
    df_moves_feat[c] = df_moves_feat[c].fillna(0.0).astype(np.float32)

print(
    f"[8.4] df_moves_feat: {df_moves_feat.shape} | "
    f"{len(df_moves_feat):,} de {n_users_clust:,} candidatos registran al menos un desplazamiento"
)
print(df_moves_feat[["n_moves", "mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min"]].describe())

# --- Telework-paper travel metrics ---
#   TO   : días calendario distintos con al menos un desplazamiento observado
#   WDTT : minutos totales de desplazamiento iniciados en día hábil
#   WETT : minutos totales de desplazamiento iniciados en fin de semana
# La clasificación usa el día de inicio del desplazamiento.
if m.empty:
    df_travel = pd.DataFrame({USER_COL: df_moves_feat[USER_COL].copy()})
    df_travel["TO"] = np.int32(0)
    df_travel["WDTT"] = np.float32(0.0)
    df_travel["WETT"] = np.float32(0.0)
else:
    move_start = m["move_start_ts"].to_numpy(dtype="datetime64[ns]", copy=False)
    move_date_int = move_start.astype("datetime64[D]").astype(np.int32, copy=False)
    move_dow = (move_date_int + 3) % 7
    is_wd = (move_dow < 5)

    m["_is_weekday"] = is_wd.astype(np.int8, copy=False)
    m["_move_date"] = move_start.astype("datetime64[D]")

    to_days = (
        m.groupby(USER_COL, observed=True, sort=False)["_move_date"]
        .nunique()
        .astype(np.int32)
        .reset_index(name="TO")
    )

    wdtt = (
        m.loc[m["_is_weekday"] == 1]
        .groupby(USER_COL, observed=True, sort=False)["move_duration_min"]
        .sum()
        .astype(np.float32)
        .reset_index(name="WDTT")
    )
    wett = (
        m.loc[m["_is_weekday"] == 0]
        .groupby(USER_COL, observed=True, sort=False)["move_duration_min"]
        .sum()
        .astype(np.float32)
        .reset_index(name="WETT")
    )

    df_travel = to_days.merge(wdtt, on=USER_COL, how="left").merge(wett, on=USER_COL, how="left")
    df_travel["WDTT"] = df_travel["WDTT"].fillna(0.0).astype(np.float32)
    df_travel["WETT"] = df_travel["WETT"].fillna(0.0).astype(np.float32)

print("[8.4] df_travel:", df_travel.shape)
print(df_travel[["TO", "WDTT", "WETT"]].describe())

del m, g, p90
if "move_start" in dir():
    del move_start, move_date_int, move_dow, is_wd, to_days, wdtt, wett
_ = gc.collect()

In [ ]:
# ============================================================
# 8.5 — Consolidación + training/audit + gráficos
# Incluye TELEWORK_PAPER features: DHD, DPD, HO, PO, WDTT, WETT, TO, SO
# ============================================================
base_cols = [
    USER_COL,
    "pre_label",
    "home_h3", "work_h3",
    "home_conf", "work_conf",
    "home_work_ratio",
]

df_base = df_clustering_candidates[base_cols].copy()

df_features = (
    df_base
    .merge(df_timealloc, on=USER_COL, how="left")
    .merge(df_daily, on=USER_COL, how="left")
    .merge(df_secondary[[USER_COL, "secondary_h3", "secondary_occurrence"]], on=USER_COL, how="left")
    .merge(df_mob, on=USER_COL, how="left")
    .merge(df_moves_feat, on=USER_COL, how="left")
    .merge(df_travel, on=USER_COL, how="left")
)

# --- Alias TELEWORK_PAPER ---
# Nombres conservados por trazabilidad conceptual con la literatura de origen.
# Su semántica operacional es la definida en las Secciones 8.1, 8.2 y 8.4:
#   DHD : minutos en el hogar dentro de la ventana laboral (home_lab_wmin)
#   DPD : minutos en el lugar primario dentro de la ventana laboral (work_lab_wmin)
#   HO  : días distintos con presencia en el hogar en ventana laboral
#   PO  : días distintos con presencia en el lugar primario en ventana laboral
#   SO  : días distintos con presencia en el lugar secundario
df_features["DHD"] = pd.to_numeric(df_features.get("home_lab_wmin", 0.0), errors="coerce").fillna(0.0).astype(np.float32)
df_features["DPD"] = pd.to_numeric(df_features.get("work_lab_wmin", 0.0), errors="coerce").fillna(0.0).astype(np.float32)
df_features["HO"] = pd.to_numeric(df_features.get("n_days_home_lab", 0), errors="coerce").fillna(0).astype(np.int32)
df_features["PO"] = pd.to_numeric(df_features.get("n_days_work_lab", 0), errors="coerce").fillna(0).astype(np.int32)
df_features["SO"] = pd.to_numeric(df_features.get("secondary_occurrence", 0), errors="coerce").fillna(0).astype(np.int32)

# Relleno de faltantes con 0.
# Los merges anteriores son left joins sobre subconjuntos: no todos los
# candidatos tienen desplazamientos ni lugar secundario. El 0 resultante indica
# ausencia de evidencia observada, no ausencia constatada del fenómeno.
num_cols = [
    "total_lab_wmin", "home_lab_wmin", "work_lab_wmin", "other_lab_wmin",
    "home_lab_ratio", "work_lab_ratio", "other_lab_ratio",
    "n_workdays_obs", "n_days_home_lab", "n_days_work_lab",
    "workday_presence_rate", "home_workday_presence_rate",
    "avg_lab_wmin_per_day", "std_lab_wmin_per_day",
    "radius_gyration_km", "entropy_spatial", "unique_locations",
    "n_moves", "mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min",
    "TO", "WDTT", "WETT",
    "secondary_occurrence",
    "DHD", "DPD",
]
for c in num_cols:
    if c in df_features.columns:
        df_features[c] = pd.to_numeric(df_features[c], errors="coerce").fillna(0)

# Types compactos
float32_cols = [
    "home_conf", "work_conf", "home_work_ratio",
    "total_lab_wmin", "home_lab_wmin", "work_lab_wmin", "other_lab_wmin",
    "home_lab_ratio", "work_lab_ratio", "other_lab_ratio",
    "workday_presence_rate", "home_workday_presence_rate",
    "avg_lab_wmin_per_day", "std_lab_wmin_per_day",
    "radius_gyration_km", "entropy_spatial",
    "mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min",
    "WDTT", "WETT",
    "DHD", "DPD",
]
int32_cols = [
    "n_workdays_obs", "n_days_home_lab", "n_days_work_lab",
    "unique_locations", "n_moves", "TO",
    "HO", "PO", "SO", "secondary_occurrence",
]

for c in float32_cols:
    if c in df_features.columns:
        df_features[c] = df_features[c].astype(np.float32)
for c in int32_cols:
    if c in df_features.columns:
        df_features[c] = df_features[c].astype(np.int32)

# --- Separación training vs audit (feature-set ablation) ---
BASE5 = ["home_work_ratio", "radius_gyration_km", "entropy_spatial", "n_moves", "unique_locations"]
TELEWORK_PAPER = ["DHD", "DPD", "WDTT", "WETT", "HO", "PO", "TO", "SO"]

if FEATURE_SET == "BASE5":
    training_cols = BASE5
elif FEATURE_SET == "TELEWORK_PAPER":
    training_cols = TELEWORK_PAPER
elif FEATURE_SET == "COMBINED":
    training_cols = BASE5 + TELEWORK_PAPER
else:
    raise ValueError("FEATURE_SET inválido. Usa 'BASE5', 'TELEWORK_PAPER' o 'COMBINED'.")

# Variables de auditoría: se calculan y persisten, pero no alimentan el
# clustering. Permiten caracterizar los segmentos resultantes sin haber
# participado de su construcción.
audit_cols = [
    "total_lab_wmin", "home_lab_ratio", "work_lab_ratio", "other_lab_ratio",
    "workday_presence_rate", "home_workday_presence_rate",
    "avg_lab_wmin_per_day", "std_lab_wmin_per_day",
    "mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min",
    "home_conf", "work_conf",
]

# Guardrail: que existan training_cols
missing_train = [c for c in training_cols if c not in df_features.columns]
if missing_train:
    raise ValueError(f"Missing training feature cols: {missing_train}")

# Artefacto de Nivel 2: contiene anclas inferidas y el vector conductual
# completo a nivel individual. Deriva de la fuente telco, se escribe en el
# directorio de artefactos y no forma parte de la entrega pública.
feat_path = paths["tables_dir"] / "features_raw.parquet"
df_features.to_parquet(feat_path, index=False)

log_jsonl(
    paths["logs_dir"] / "features.jsonl",
    {
        "event": "features_saved",
        "ts_utc": now_utc(),
        "path": str(feat_path),
        "feature_set": FEATURE_SET,
        "users": int(df_features[USER_COL].nunique()),
        "n_cols": int(df_features.shape[1]),
        "training_cols": training_cols,
        "audit_cols": [c for c in audit_cols if c in df_features.columns],
        "mem_mb_features": mem_mb_df(df_features),
    },
)

print("[8.5] df_features:", df_features.shape, f"| mem={mem_mb_df(df_features):.1f}MB")
print(f"[8.5] training_cols ({len(training_cols)}):", training_cols)
print(f"[8.5] audit_cols ({len([c for c in audit_cols if c in df_features.columns])}):",
      [c for c in audit_cols if c in df_features.columns])

# Resumen agregado del vector de entrenamiento. No se imprimen filas
# individuales: df_features combina identificador, ambas anclas y el perfil
# conductual completo por usuario.
print("[8.5] resumen del vector de entrenamiento:")
print(df_features[training_cols].describe().T[["mean", "std", "min", "50%", "max"]])

# Cleanup de dataframes intermedios (deja df_features vivo)
del df_timealloc, df_daily, df_secondary, df_mob, df_moves_feat, df_travel
_ = gc.collect()

In [ ]:
# ============================================================
# 8.6 — Gráficos QA
#   - Correlación entre variables de entrenamiento
#   - Distribución de home_work_ratio por etiqueta previa
#   - Dispersión home_work_ratio vs n_moves por etiqueta previa
#
# Las figuras derivan de la fuente telco y se escriben en el directorio de
# artefactos; no forman parte de la entrega pública.
# ============================================================

# 1) Correlación training features (con valores en celdas)
corr = df_features[training_cols].corr(numeric_only=True)

fig = plt.figure(figsize=(max(6, 0.6 * len(training_cols)), max(5, 0.6 * len(training_cols))))
ax = plt.gca()
im = ax.imshow(corr.values)
ax.set_xticks(range(len(training_cols)))
ax.set_xticklabels(training_cols, rotation=45, ha="right")
ax.set_yticks(range(len(training_cols)))
ax.set_yticklabels(training_cols)
ax.set_title(f"Correlation matrix (training features) — {FEATURE_SET}")
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
corr_path = paths["figures_dir"] / f"corr_training_features_{FEATURE_SET}.png"
fig.savefig(corr_path, dpi=200)
plt.show()
plt.close(fig)

# 2) Distribución de home_work_ratio por pre_label
labels = ["WORKER_OFFICE", "WORKER_POTENTIAL_REMOTE"]
data = [df_features.loc[df_features["pre_label"] == lab, "home_work_ratio"].to_numpy() for lab in labels]

fig = plt.figure(figsize=(7, 4))
ax = plt.gca()
ax.boxplot(data, tick_labels=labels, showfliers=False)
ax.set_title("home_work_ratio by pre_label")
fig.tight_layout()
box_path = paths["figures_dir"] / "home_work_ratio_by_prelabel.png"
fig.savefig(box_path, dpi=200)
plt.show()
plt.close(fig)

# 3) Scatter home_work_ratio vs n_moves
x_name = "home_work_ratio"
y_name = "n_moves" if "n_moves" in df_features.columns else "unique_locations"

fig = plt.figure(figsize=(7, 4))
ax = plt.gca()
for lab in labels:
    sub = df_features.loc[df_features["pre_label"] == lab, [x_name, y_name]].dropna()
    ax.scatter(sub[x_name].to_numpy(), sub[y_name].to_numpy(), s=8, alpha=0.6, label=lab)
ax.set_xlabel(x_name)
ax.set_ylabel(y_name)
ax.set_title(f"{x_name} vs {y_name} by pre_label")
ax.legend()
fig.tight_layout()
scat_path = paths["figures_dir"] / f"scatter_{x_name}_vs_{y_name}_by_prelabel.png"
fig.savefig(scat_path, dpi=200)
plt.show()
plt.close(fig)

log_jsonl(
    paths["logs_dir"] / "features.jsonl",
    {
        "event": "features_figures_saved",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "corr_path": str(corr_path),
        "box_path": str(box_path),
        "scatter_path": str(scat_path),
    },
)

print("[8.6] Figures saved:", corr_path.name, "|", box_path.name, "|", scat_path.name)

# RAM hygiene final (mantener df_features para Sección 9)
del corr, data, labels
_ = gc.collect()

## *9. Clustering Jerárquico (Ward) + Selección de k*

Esta sección agrupa a los candidatos mediante **enlace jerárquico de Ward**
sobre las variables de entrenamiento estandarizadas.

El número de conglomerados es una decisión metodológica, no un resultado del
diagnóstico. Se fija en `k_baseline` porque el prototipo busca distinguir tres
modalidades laborales —presencial, híbrida y remota—, y esa correspondencia
semántica es la que hace interpretable el resultado. El barrido de métricas
sobre el rango `[k_min_diag, k_max_diag]` se calcula y persiste como
diagnóstico, pero no interviene en la elección.

La estandarización se aplica sobre la totalidad de los candidatos, de modo que
cada variable contribuye con varianza unitaria a la distancia euclidiana. Sin
ella, las variables en minutos —`DHD`, `DPD`, `WDTT`, `WETT`— dominarían por
completo sobre las razones y los recuentos.

**Sobre el modo de agrupamiento.** Bajo `ward_full_max_n` se calcula el enlace
completo, lo que permite construir el dendrograma y obtener particiones
anidadas para cualquier k. Por sobre ese umbral el enlace completo deja de ser
tratable y se recurre a agrupamiento aglomerativo con restricción de
conectividad kNN, que es una aproximación: los conglomerados resultantes no
coinciden necesariamente con los del enlace completo. El modo efectivamente
empleado queda registrado en `cluster_method` y en el reporte de auditoría.

**Advertencia sobre los diagnósticos de k.** Silueta, Calinski-Harabasz y
Davies-Bouldin miden separación geométrica en el espacio estandarizado. Un valor
favorable indica que la partición es compacta, no que sea sustantivamente
correcta ni que los conglomerados correspondan a modalidades laborales reales.
La validación de esa correspondencia es externa a esta sección y el manuscrito
declara que no fue posible realizarla.

In [ ]:
# ============================================================
# 9 — Clustering Jerárquico (Ward) + Diagnóstico k
#   - Usa training_cols dinámico (de Sección 8)
#   - Registra FEATURE_SET y TRAINING_COLS efectivos
#   - Artefactos sufijados por FEATURE_SET (evita colisiones entre corridas
#     con distinta configuración de variables)
# ============================================================
USER_COL = cfg.col_user

# ------------------------------------------------------------
# 9.0 — Preparación X + parámetros
# ------------------------------------------------------------
# Requiere que FEATURE_SET y training_cols existan desde la Sección 8.
try:
    _ = FEATURE_SET
    _ = training_cols
except NameError as e:
    raise NameError(
        "Se requiere que la Sección 8 haya definido FEATURE_SET y training_cols "
        "antes de ejecutar la Sección 9."
    ) from e

TRAINING_COLS = list(training_cols)  # copia defensiva

missing = [c for c in TRAINING_COLS if c not in df_features.columns]
if missing:
    raise ValueError(f"Faltan training features en df_features para FEATURE_SET={FEATURE_SET}: {missing}")

# Parámetros de la sección, declarados en ExperimentConfig (Sección 0.2).
K_BASELINE = int(cfg.k_baseline)
K_MIN = int(cfg.k_min_diag)
K_MAX = int(cfg.k_max_diag)
FULL_WARD_MAX_N = int(cfg.ward_full_max_n)
SAMPLE_N_FOR_DIAG = int(cfg.ward_sample_n_for_diag)
SIL_SAMPLE_N = int(cfg.silhouette_sample_n)
KNN_NEIGHBORS = int(cfg.ward_knn_neighbors)

# Construir X (float32) y users
dfX = df_features[[USER_COL] + TRAINING_COLS].copy()
for c in TRAINING_COLS:
    dfX[c] = pd.to_numeric(dfX[c], errors="coerce").fillna(0)

X = dfX[TRAINING_COLS].to_numpy(dtype=np.float32, copy=True)
users = dfX[USER_COL].to_numpy(copy=False)

n, p = X.shape

print(f"[9.0] FEATURE_SET={FEATURE_SET} | X shape=({n:,},{p}) | k_baseline={K_BASELINE}")
print(f"[9.0] modo de enlace: {'completo' if n <= FULL_WARD_MAX_N else f'kNN (n > {FULL_WARD_MAX_N:,})'}")

log_jsonl(
    paths["logs_dir"] / "clustering.jsonl",
    {
        "event": "section9_setup",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "n_users": int(n),
        "n_features": int(p),
        "training_cols": TRAINING_COLS,
        "k_baseline": int(K_BASELINE),
        "diag_k_range": [int(K_MIN), int(K_MAX)],
        "full_ward_max_n": int(FULL_WARD_MAX_N),
        "knn_neighbors": int(KNN_NEIGHBORS),
    },
)

del dfX
_ = gc.collect()

In [ ]:
# ------------------------------------------------------------
# 9.1 — Estandarización
# ------------------------------------------------------------
# Necesaria porque las variables de entrenamiento tienen escalas heterogéneas:
# minutos (DHD, DPD, WDTT, WETT), razones acotadas en [0,1] (home_work_ratio),
# kilómetros (radius_gyration_km) y recuentos (n_moves, HO, PO, TO, SO). Sin
# estandarizar, la distancia euclidiana estaría dominada por las primeras.
#
# Los parámetros del escalador se registran en el log: permiten reconstruir la
# transformación aplicada sin reejecutar el ajuste.
scaler = StandardScaler(with_mean=True, with_std=True)
Xz = scaler.fit_transform(X)  # float64
Xz = Xz.astype(np.float32, copy=False)

log_jsonl(
    paths["logs_dir"] / "clustering.jsonl",
    {
        "event": "scaler_fitted",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "feature_cols": TRAINING_COLS,
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
    },
)

del X
_ = gc.collect()

print("[9.1] Xz dtype:", Xz.dtype, "| shape:", Xz.shape)

In [ ]:
# ------------------------------------------------------------
# 9.2 — Diagnóstico k (NO selecciona k)
# ------------------------------------------------------------
# Las tres métricas miden separación geométrica en el espacio estandarizado:
#   silhouette        : cohesión vs. separación por punto (mayor es mejor)
#   calinski_harabasz : razón de dispersión entre/intra grupos (mayor es mejor)
#   davies_bouldin    : similitud media con el grupo más próximo (menor es mejor)
#
# Se calculan y persisten como diagnóstico. NO determinan k: la elección
# corresponde a la correspondencia semántica con las tres modalidades laborales
# que el prototipo busca distinguir.
rng = np.random.default_rng(int(cfg.seed))


def eval_k_metrics(labels: np.ndarray, X_eval: np.ndarray, seed: int, sil_sample_n: int) -> dict:
    n_eval = X_eval.shape[0]
    sample_n = min(int(sil_sample_n), n_eval)
    if sample_n < n_eval:
        idx = np.random.default_rng(seed).choice(n_eval, size=sample_n, replace=False)
        X_s = X_eval[idx]
        lab_s = labels[idx]
    else:
        X_s = X_eval
        lab_s = labels

    out = {}
    try:
        out["silhouette"] = float(silhouette_score(X_s, lab_s, metric="euclidean"))
    except Exception:
        out["silhouette"] = np.nan
    try:
        out["calinski_harabasz"] = float(calinski_harabasz_score(X_eval, labels))
    except Exception:
        out["calinski_harabasz"] = np.nan
    try:
        out["davies_bouldin"] = float(davies_bouldin_score(X_eval, labels))
    except Exception:
        out["davies_bouldin"] = np.nan
    return out


KS = list(range(K_MIN, K_MAX + 1))
df_kdiag = None
Z_diag = None

if n <= FULL_WARD_MAX_N:
    # Enlace completo: permite dendrograma y particiones anidadas para todo k
    print("[9.2] Linkage completo (diagnóstico y dendrograma)…")
    Z_diag = linkage(Xz, method="ward")

    rows = []
    for k in KS:
        lab = fcluster(Z_diag, t=int(k), criterion="maxclust").astype(np.int32)
        m = eval_k_metrics(lab, Xz, seed=int(cfg.seed), sil_sample_n=SIL_SAMPLE_N)
        rows.append({"k": int(k), **m})
    df_kdiag = pd.DataFrame(rows)

    # dendrograma (solo si full)
    fig = plt.figure(figsize=(9, 4))
    dendrogram(Z_diag, no_labels=True, color_threshold=None)
    plt.title(f"Ward dendrogram (full linkage) — {FEATURE_SET}")
    plt.tight_layout()
    dendro_path = paths["figures_dir"] / f"dendrogram_ward_full_{FEATURE_SET}.png"
    fig.savefig(dendro_path, dpi=200)
    plt.show()
    plt.close(fig)

    log_jsonl(
        paths["logs_dir"] / "clustering.jsonl",
        {"event": "dendrogram_saved", "ts_utc": now_utc(), "feature_set": FEATURE_SET, "path": str(dendro_path)},
    )
else:
    # Submuestra para diagnóstico de k. El dendrograma no se construye:
    # correspondería a la submuestra y no al universo completo.
    n_s = min(SAMPLE_N_FOR_DIAG, n)
    idx = rng.choice(n, size=n_s, replace=False)
    X_s = Xz[idx]
    print(f"[9.2] n grande => diagnóstico k sobre submuestra: n_s={n_s:,}")

    Z_s = linkage(X_s, method="ward")
    rows = []
    for k in KS:
        lab = fcluster(Z_s, t=int(k), criterion="maxclust").astype(np.int32)
        m = eval_k_metrics(lab, X_s, seed=int(cfg.seed), sil_sample_n=min(SIL_SAMPLE_N, n_s))
        rows.append({"k": int(k), **m})
    df_kdiag = pd.DataFrame(rows)

    del X_s, Z_s
    gc.collect()

# Persistir diagnóstico (sufijado por feature_set)
kdiag_path = paths["tables_dir"] / f"k_diagnostic_metrics_{FEATURE_SET}.parquet"
df_kdiag.to_parquet(kdiag_path, index=False)

log_jsonl(
    paths["logs_dir"] / "clustering.jsonl",
    {
        "event": "k_diagnostic_saved",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "path": str(kdiag_path),
        "k_range": [K_MIN, K_MAX],
        "metrics": df_kdiag.to_dict(orient="records"),
    },
)

print("[9.2] diagnóstico de k:")
print(df_kdiag.sort_values("k").to_string(index=False))

# Contraste entre el k de la configuración y el óptimo por silueta.
_k_sil = int(df_kdiag.loc[df_kdiag["silhouette"].idxmax(), "k"]) if df_kdiag["silhouette"].notna().any() else None
if _k_sil is not None and _k_sil != K_BASELINE:
    _s_base = df_kdiag.loc[df_kdiag["k"] == K_BASELINE, "silhouette"]
    _s_best = df_kdiag["silhouette"].max()
    print(
        f"[9.2] La silueta favorece k={_k_sil} ({_s_best:.4f}) sobre "
        f"k={K_BASELINE} ({float(_s_base.iloc[0]):.4f} si está en el rango).\n"
        "      El k de la configuración se conserva: responde a la "
        "correspondencia semántica con las modalidades laborales, no a la "
        "optimización de separación geométrica. La discrepancia queda "
        "registrada como diagnóstico."
    )
del _k_sil

In [ ]:
# ------------------------------------------------------------
# 9.3 — Clustering final (k de la configuración)
# ------------------------------------------------------------
# Ambas ramas producen etiquetas en el rango 1..k. fcluster ya las entrega en
# esa convención; AgglomerativeClustering devuelve 0..k-1 y se desplaza en 1.
# La homogeneidad importa porque las Secciones 10 y 13 asumen etiquetas 1..k.
k = K_BASELINE

if n <= FULL_WARD_MAX_N:
    # Reutilizar el enlace del diagnóstico si existe; si no, construirlo
    if Z_diag is None:
        Z_full = linkage(Xz, method="ward")
    else:
        Z_full = Z_diag

    labels = fcluster(Z_full, t=int(k), criterion="maxclust").astype(np.int32)

    # Guardar linkage para auditoría (sufijado)
    link_path = paths["cache_dir"] / f"ward_linkage_full_{FEATURE_SET}.npy"
    np.save(link_path, Z_full.astype(np.float32, copy=False))
    log_jsonl(
        paths["logs_dir"] / "clustering.jsonl",
        {"event": "linkage_saved", "ts_utc": now_utc(), "feature_set": FEATURE_SET, "path": str(link_path)},
    )

    if Z_diag is None:
        del Z_full
        gc.collect()

    clustering_mode = "ward_full"
else:
    # Restricción de conectividad kNN: aproximación al enlace completo, no
    # equivalente. Los conglomerados pueden diferir de los que produciría el
    # enlace jerárquico exacto.
    print(f"[9.3] Ward + kNN connectivity (n_neighbors={KNN_NEIGHBORS})…")
    conn = kneighbors_graph(Xz, n_neighbors=KNN_NEIGHBORS, mode="connectivity", include_self=False)
    model = AgglomerativeClustering(
        n_clusters=int(k),
        linkage="ward",
        connectivity=conn,
        compute_full_tree=False,
    )
    labels = model.fit_predict(Xz).astype(np.int32) + 1  # 1..k

    del conn, model
    gc.collect()

    clustering_mode = "ward_connectivity_knn"

_uniq, _cnts = np.unique(labels, return_counts=True)
print(f"[9.3] modo={clustering_mode} | k={k}")
print("[9.3] tamaños:", {int(u): int(c) for u, c in zip(_uniq, _cnts)})

log_jsonl(
    paths["logs_dir"] / "clustering.jsonl",
    {
        "event": "clustering_done",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "mode": clustering_mode,
        "k": int(k),
        "n_users": int(n),
        "cluster_sizes": {int(u): int(c) for u, c in zip(_uniq, _cnts)},
    },
)

del _uniq, _cnts

In [ ]:
# ------------------------------------------------------------
# 9.4 — Integrar labels + gráficos básicos
# ------------------------------------------------------------
df_features = df_features.copy()
df_features["cluster_id"] = labels.astype(np.int32)
df_features["cluster_k"] = np.int16(k)
df_features["cluster_method"] = clustering_mode
df_features["feature_set"] = FEATURE_SET

# Distribución de conglomerados
counts = df_features["cluster_id"].value_counts().sort_index()

fig = plt.figure(figsize=(6, 3.5))
plt.bar(counts.index.astype(str), counts.values)
plt.title(f"Cluster counts (k={k}, Ward) — {FEATURE_SET}")
plt.xlabel("cluster_id")
plt.ylabel("n users")
plt.tight_layout()
bar_path = paths["figures_dir"] / f"cluster_counts_k{k}_{FEATURE_SET}.png"
fig.savefig(bar_path, dpi=200)
plt.show()
plt.close(fig)

# Artefacto de Nivel 2: contiene anclas inferidas, vector conductual y
# asignación de conglomerado a nivel individual. Deriva de la fuente telco, se
# escribe en el directorio de artefactos y no forma parte de la entrega pública.
out_path = paths["tables_dir"] / f"features_with_clusters_{FEATURE_SET}.parquet"
df_features.to_parquet(out_path, index=False)

log_jsonl(
    paths["logs_dir"] / "clustering.jsonl",
    {
        "event": "clusters_saved",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "path": str(out_path),
        "k": int(k),
        "mode": clustering_mode,
        "fig_cluster_counts": str(bar_path),
    },
)

print("[9.4] distribución de conglomerados:")
for _cid, _n in counts.items():
    print(f"       cluster {int(_cid)}: {int(_n):>6,}  ({_n / max(int(n), 1):.2%})")

# Nota de lectura: una partición muy desbalanceada suele reflejar la estructura
# del universo de entrada más que la del espacio de variables. En este prototipo,
# la proporción de usuarios sin ancla laboral resuelta condiciona directamente el
# tamaño del conglomerado mayoritario.
_max_share = float(counts.max()) / max(int(n), 1)
if _max_share > 0.6:
    print(
        f"[9.4] El conglomerado mayoritario concentra el {_max_share:.2%} de los "
        "candidatos. Contrastar con la proporción de anclas laborales no "
        "resueltas reportada en la Sección 5 antes de interpretar."
    )

del _max_share

In [ ]:
# ------------------------------------------------------------
# 9.5 — Limpieza RAM
# ------------------------------------------------------------
del labels, Xz, scaler
_ = gc.collect()

print(f"[Section 9] OK — clustering Ward terminado (k={k}), resultados guardados.")

## *10. Consolidación y Etiquetado Final*

Esta sección traduce los conglomerados de la Sección 9 en segmentos de
modalidad laboral y consolida el universo completo: los candidatos
clusterizados y los usuarios asignados por regla en la Sección 7.

**El paso crítico es el mapeo de conglomerado a modalidad.** El clustering
produce grupos numerados sin significado propio; asignarles las etiquetas
ONSITE, HYBRID y REMOTE requiere un criterio externo. Aquí ese criterio es el
ordenamiento por la mediana de una variable de contraste —por defecto
`home_work_ratio`— bajo el supuesto de que mayor permanencia domiciliaria en
horario laboral corresponde a mayor grado de trabajo remoto.

El mapeo es determinista y reproducible, pero **es una interpretación
impuesta sobre la partición, no un resultado de ella**. Su validez descansa en
dos condiciones: que el supuesto de ordenamiento sea correcto, y que la variable
de contraste discrimine efectivamente entre los conglomerados. La segunda
condición se verifica en tiempo de ejecución y se reporta.

La consolidación final produce cinco segmentos en una única columna:
`final_segment`. Tres provienen del clustering; dos —`MOBILE_FOR_WORK` y
`NO_WORKER`— provienen de las reglas de la Sección 7 y no participaron del
agrupamiento.

**Alcance del producto.** Los segmentos describen patrones de presencia
espaciotemporal observados, no condiciones ocupacionales verificadas. No fueron
validados contra fuente externa: el manuscrito declara esa validación como no
habilitada, por incompatibilidad de universos, escalas y definiciones entre las
fuentes disponibles. La prevalencia resultante no es extrapolable a la población.

In [ ]:
# ============================================================
# 10 — Consolidación y Etiquetado Final
#   - Compatible con FEATURE_SET (BASE5 / TELEWORK_PAPER / COMBINED)
#   - Artefactos sufijados por FEATURE_SET
#   - Requisitos mínimos de esquema (no asume columnas de un feature set)
#   - Perfilado extendido con variables de auditoría
#   - Mapeo determinista cluster -> {ONSITE, HYBRID, REMOTE}
#   - Consolidación final: cinco segmentos en una sola columna
# ============================================================
USER_COL = cfg.col_user

# ------------------------------------------------------------
# 10.0 — Insumos, carga y checks (RAM-safe)
# ------------------------------------------------------------
try:
    _ = FEATURE_SET
except NameError as e:
    raise NameError("Se requiere FEATURE_SET definido (Secciones 8/9).") from e

# Descartados (asignados por regla en la Sección 7)
discard_path = paths["tables_dir"] / "universe_discarded.parquet"
df_discarded = pd.read_parquet(discard_path)

# Features con clusters. Si no está en RAM, se carga desde el parquet sufijado.
if "df_features" not in globals():
    feat_path = paths["tables_dir"] / f"features_with_clusters_{FEATURE_SET}.parquet"
    df_features = pd.read_parquet(feat_path)
else:
    # Guardrail de consistencia: si df_features en memoria proviene de una
    # corrida con otro conjunto de variables, se recarga el que corresponde a
    # FEATURE_SET. Evita consolidar segmentos con un vector de entrenamiento
    # distinto del declarado.
    if "feature_set" in df_features.columns:
        fs_vals = df_features["feature_set"].astype(str).unique().tolist()
        if len(fs_vals) == 1 and fs_vals[0] != FEATURE_SET:
            print(
                f"[10.0] df_features en memoria corresponde a FEATURE_SET="
                f"{fs_vals[0]}; se recarga el de {FEATURE_SET}."
            )
            feat_path = paths["tables_dir"] / f"features_with_clusters_{FEATURE_SET}.parquet"
            df_features = pd.read_parquet(feat_path)

# Guardrails mínimos de esquema (núcleo para mapping + consolidación)
req_feat_min = {
    USER_COL, "cluster_id", "cluster_k", "cluster_method",
    "pre_label", "home_h3", "work_h3", "home_conf", "work_conf",
}
miss_feat_min = req_feat_min - set(df_features.columns)
if miss_feat_min:
    raise ValueError(f"df_features missing minimal cols required for Section 10: {miss_feat_min}")

req_disc = {USER_COL, "final_label"}
miss_disc = req_disc - set(df_discarded.columns)
if miss_disc:
    raise ValueError(f"df_discarded missing cols: {miss_disc}")

assert df_features.duplicated(subset=[USER_COL]).sum() == 0, "df_features tiene usuarios duplicados."
assert df_discarded.duplicated(subset=[USER_COL]).sum() == 0, "df_discarded tiene usuarios duplicados."

print("[10.0] FEATURE_SET:", FEATURE_SET)
print("[10.0] df_features:", df_features.shape, "| df_discarded:", df_discarded.shape)

log_jsonl(
    paths["logs_dir"] / "final.jsonl",
    {
        "event": "section10_start",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "n_features": int(len(df_features)),
        "n_discarded": int(len(df_discarded)),
        "discard_path": str(discard_path),
    },
)

_ = gc.collect()

In [ ]:
# ------------------------------------------------------------
# 10.1 — Perfilado por conglomerado (auditoría + soporte del mapeo)
# ------------------------------------------------------------
# Se emplean medianas y no medias: los conglomerados son de tamaño muy
# desigual y varias variables tienen distribuciones asimétricas con valores
# extremos. La mediana da un descriptor central robusto para el ordenamiento.
#
# El perfilado incluye tanto variables de entrenamiento como de auditoría. Las
# segundas no participaron de la construcción de los conglomerados, de modo que
# su comportamiento por grupo es evidencia independiente sobre la naturaleza de
# la partición.

# Variable de contraste para ordenar los conglomerados de menor a mayor grado
# de permanencia domiciliaria en horario laboral.
order_key_candidates = ["home_work_ratio", "home_lab_ratio", "DHD"]
order_key = next((c for c in order_key_candidates if c in df_features.columns), None)
if order_key is None:
    raise ValueError(
        "No existe ninguna variable para ordenar conglomerados "
        "(home_work_ratio / home_lab_ratio / DHD)."
    )

# Columnas opcionales para perfilado (si existen)
profile_optional = [
    "home_work_ratio",
    "home_lab_ratio", "work_lab_ratio", "other_lab_ratio",
    "workday_presence_rate", "home_workday_presence_rate",
    "radius_gyration_km", "entropy_spatial", "unique_locations",
    "n_moves",
    "mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min",
    "DHD", "DPD", "HO", "PO", "TO", "SO", "WDTT", "WETT",
]
profile_cols = [c for c in profile_optional if c in df_features.columns]

agg_dict = {"n_users": (USER_COL, "nunique")}
for c in profile_cols:
    agg_dict[f"med_{c}"] = (c, "median")

cluster_profiles = (
    df_features.groupby("cluster_id", observed=True, sort=True)
    .agg(**agg_dict)
    .reset_index()
)

print(f"[10.1] variable de ordenamiento: {order_key}")
print("[10.1] perfil por conglomerado (medianas):")
print(cluster_profiles.to_string(index=False))

# Artefacto de Nivel 2: agregados por conglomerado derivados de la fuente telco.
# Se escribe en el directorio de artefactos y no forma parte de la entrega pública.
profiles_path = paths["tables_dir"] / f"cluster_profiles_{FEATURE_SET}.parquet"
cluster_profiles.to_parquet(profiles_path, index=False)

log_jsonl(
    paths["logs_dir"] / "final.jsonl",
    {
        "event": "cluster_profiles_saved",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "order_key": order_key,
        "path": str(profiles_path),
        "profiles": cluster_profiles.to_dict(orient="records"),
    },
)

# Gráfico: mediana de la variable de ordenamiento por conglomerado
fig = plt.figure(figsize=(6, 3.5))
plt.bar(cluster_profiles["cluster_id"].astype(str), cluster_profiles[f"med_{order_key}"].to_numpy())
plt.title(f"Cluster profile: median {order_key} — {FEATURE_SET}")
plt.xlabel("cluster_id")
plt.ylabel(f"median({order_key})")
plt.tight_layout()
prof_fig_path = paths["figures_dir"] / f"cluster_profile_med_{order_key}_{FEATURE_SET}.png"
fig.savefig(prof_fig_path, dpi=200)
plt.show()
plt.close(fig)

log_jsonl(
    paths["logs_dir"] / "final.jsonl",
    {"event": "cluster_profile_fig_saved", "ts_utc": now_utc(), "feature_set": FEATURE_SET, "path": str(prof_fig_path)},
)

_ = gc.collect()

In [ ]:
# ------------------------------------------------------------
# 10.2 — Mapeo determinista cluster -> modalidad_3
# ------------------------------------------------------------
# SUPUESTO DE ORDENAMIENTO
# Los conglomerados se ordenan de forma ascendente por la mediana de la
# variable de contraste y se les asignan, en ese orden, las etiquetas
# [ONSITE, HYBRID, REMOTE]. El supuesto es que un mayor valor de la variable
# corresponde a un mayor grado de trabajo remoto.
#
# Este mapeo es una interpretación impuesta sobre la partición, no un resultado
# de ella: el clustering produce grupos sin semántica propia. Su validez
# descansa en que la variable de contraste discrimine efectivamente entre los
# conglomerados, condición que se verifica abajo y se reporta.
#
# REGLA DE DESEMPATE
# El ordenamiento usa mergesort (estable): ante medianas iguales, se preserva
# el orden de cluster_id. La regla es determinista y reproducible, pero cuando
# se activa la asignación entre las etiquetas empatadas NO se apoya en la
# variable de contraste. El diagnóstico posterior lo señala.

order = cluster_profiles.sort_values(
    f"med_{order_key}", ascending=True, kind="mergesort"
)["cluster_id"].to_list()

n_clusters = int(df_features["cluster_id"].nunique())
if len(order) != n_clusters:
    raise RuntimeError("Inconsistencia: clusters en perfiles no calzan con df_features.")

# --- Diagnóstico de discriminación de la variable de ordenamiento ---
_med_vals = cluster_profiles.set_index("cluster_id")[f"med_{order_key}"]
_ordered_meds = [float(_med_vals.loc[c]) for c in order]
_ties = [
    (order[i], order[i + 1])
    for i in range(len(order) - 1)
    if np.isclose(_ordered_meds[i], _ordered_meds[i + 1])
]

print(f"[10.2] orden por med_{order_key} (ascendente): {order}")
print(f"[10.2] medianas en ese orden: {[round(v, 6) for v in _ordered_meds]}")

if _ties:
    print(
        f"[10.2] ADVERTENCIA: la variable de ordenamiento '{order_key}' no "
        f"discrimina entre {len(_ties)} par(es) de conglomerados: {_ties}.\n"
        "       La asignación de etiquetas entre ellos queda determinada por la "
        "regla de desempate (orden de cluster_id) y no por la variable de "
        "contraste. Contrastar con las variables de auditoría del perfilado "
        "(work_lab_ratio, workday_presence_rate, DPD, DHD) antes de interpretar "
        "las etiquetas resultantes."
    )

cluster_to_mod3 = {}
if len(order) == 3:
    # [menor, intermedio, mayor] -> [ONSITE, HYBRID, REMOTE]
    cluster_to_mod3 = {order[0]: "ONSITE", order[1]: "HYBRID", order[2]: "REMOTE"}
else:
    # Con k distinto de 3 la correspondencia semántica con las tres modalidades
    # laborales se pierde. Se conserva un etiquetado determinista sin semántica.
    print(
        f"[10.2] k={len(order)} distinto de 3: no se asignan etiquetas de "
        "modalidad. Se usa un etiquetado ordenado sin semántica."
    )
    for i, cid in enumerate(order):
        cluster_to_mod3[cid] = f"CLUST_{i+1}_ORDERED"

df_features = df_features.copy()
df_features["work_modality_3"] = df_features["cluster_id"].map(cluster_to_mod3).astype("category")

# final_segment: para los clusterizados coincide con work_modality_3.
# Los descartados se incorporan en la Sección 10.3.
df_features["final_segment"] = df_features["work_modality_3"]

log_jsonl(
    paths["logs_dir"] / "final.jsonl",
    {
        "event": "modality3_mapped",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "order_key": order_key,
        "cluster_order": [int(c) for c in order],
        "ordered_medians": _ordered_meds,
        "tie_pairs": [[int(a), int(b)] for a, b in _ties],
        "tie_break_rule": "stable_sort_on_cluster_id",
        "cluster_to_mod3": {int(k_): v for k_, v in cluster_to_mod3.items()},
        "dist_mod3": df_features["work_modality_3"].value_counts(dropna=False).to_dict(),
    },
)

print("[10.2] mapeo aplicado:", {int(k_): v for k_, v in cluster_to_mod3.items()})
print("[10.2] distribución de work_modality_3:")
print(df_features["work_modality_3"].value_counts(dropna=False).sort_index().to_string())

# Gráfico: distribución mod3
counts_mod3 = df_features["work_modality_3"].value_counts()

fig = plt.figure(figsize=(7, 3.5))
plt.bar(counts_mod3.index.astype(str), counts_mod3.values)
plt.title(f"work_modality_3 distribution — {FEATURE_SET}")
plt.xticks(rotation=25, ha="right")
plt.ylabel("n users")
plt.tight_layout()
mod3_fig_path = paths["figures_dir"] / f"work_modality_3_distribution_{FEATURE_SET}.png"
fig.savefig(mod3_fig_path, dpi=200)
plt.show()
plt.close(fig)

log_jsonl(
    paths["logs_dir"] / "final.jsonl",
    {"event": "work_modality_3_fig_saved", "ts_utc": now_utc(), "feature_set": FEATURE_SET, "path": str(mod3_fig_path)},
)

del _med_vals, _ordered_meds, _ties
_ = gc.collect()

In [ ]:
# ------------------------------------------------------------
# 10.3 — Consolidación final por usuario (cinco segmentos)
# ------------------------------------------------------------
# Se unen dos poblaciones con procedencia distinta:
#   - Clusterizados : segmento derivado del agrupamiento de la Sección 9
#   - Descartados   : segmento determinado por regla en la Sección 7
# Los descartados no tienen conglomerado ni modalidad_3; esas columnas se
# rellenan con NA para preservar un esquema común, y final_label conserva la
# procedencia de cada usuario como variable de auditoría.

# Clusterizados
df_workers = df_features.copy()
df_workers["final_label"] = "WORKER"  # audit

# Descartados (móviles / sin evidencia laboral)
df_disc = df_discarded.copy()

df_disc["final_segment"] = np.where(
    df_disc["final_label"].astype(str) == "MOBILE_WORKER",
    "MOBILE_FOR_WORK",
    "NO_WORKER",
)

# Columnas que no aplican a los descartados
df_disc["work_modality_3"] = pd.NA
df_disc["cluster_id"] = pd.NA
df_disc["cluster_k"] = pd.NA
df_disc["cluster_method"] = pd.NA
df_disc["pre_label"] = df_disc["final_label"]

# Selección de columnas para salida (núcleo + opcionales si existen)
base_worker_cols = [
    USER_COL,
    "final_segment",     # cinco segmentos (columna principal)
    "final_label",       # audit: procedencia (WORKER / MOBILE_WORKER / NO_WORKER)
    "work_modality_3",   # tres clases (solo clusterizados)
    "cluster_id", "cluster_k", "cluster_method",
    "pre_label",
    "home_h3", "work_h3",
    "home_conf", "work_conf",
    "home_work_ratio",
]
optional_out = [
    "home_lab_ratio", "work_lab_ratio", "other_lab_ratio",
    "workday_presence_rate", "home_workday_presence_rate",
    "radius_gyration_km", "entropy_spatial", "unique_locations",
    "n_moves",
    "mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min",
    "DHD", "DPD", "HO", "PO", "TO", "SO", "WDTT", "WETT",
]

final_cols_workers = [c for c in base_worker_cols + optional_out if c in df_workers.columns]
final_cols_disc = [
    USER_COL, "final_segment", "final_label", "pre_label",
    "work_modality_3", "cluster_id", "cluster_k", "cluster_method",
]

df_workers_out = df_workers[final_cols_workers].copy()
df_disc_out = df_disc[final_cols_disc].copy()

# ------------------------------------------------------------
# 10.3.X — Armonización de tipos antes del concat
# ------------------------------------------------------------
# Las categorías se declaran ordenadas y con el mismo catálogo en ambos
# dataframes. Sin esto, pd.concat sobre categóricas con catálogos distintos
# degrada la columna a object de forma silenciosa, y el orden semántico de los
# segmentos se pierde en tablas y gráficos.
FINAL_SEG_CATS = ["ONSITE", "HYBRID", "REMOTE", "MOBILE_FOR_WORK", "NO_WORKER"]
MOD3_CATS = ["ONSITE", "HYBRID", "REMOTE"]
PRE_LABEL_CATS = ["WORKER_OFFICE", "WORKER_POTENTIAL_REMOTE", "MOBILE_WORKER", "NO_WORKER"]
FINAL_LABEL_CATS = ["WORKER", "MOBILE_WORKER", "NO_WORKER"]

for _df in (df_workers_out, df_disc_out):
    if "final_segment" in _df.columns:
        _df["final_segment"] = pd.Categorical(
            _df["final_segment"].astype("string[python]"),
            categories=FINAL_SEG_CATS,
            ordered=True,
        )
    if "work_modality_3" in _df.columns:
        _df["work_modality_3"] = pd.Categorical(
            _df["work_modality_3"].astype("string[python]"),
            categories=MOD3_CATS,
            ordered=True,
        )
    if "pre_label" in _df.columns:
        _df["pre_label"] = pd.Categorical(
            _df["pre_label"].astype("string[python]"),
            categories=PRE_LABEL_CATS,
            ordered=True,
        )
    if "final_label" in _df.columns:
        _df["final_label"] = pd.Categorical(
            _df["final_label"].astype("string[python]"),
            categories=FINAL_LABEL_CATS,
            ordered=True,
        )
    # Enteros nullables: permiten NA sin promover a float
    if "cluster_id" in _df.columns:
        _df["cluster_id"] = pd.array(_df["cluster_id"], dtype="Int32")
    if "cluster_k" in _df.columns:
        _df["cluster_k"] = pd.array(_df["cluster_k"], dtype="Int16")
    if "cluster_method" in _df.columns:
        _df["cluster_method"] = _df["cluster_method"].astype("string[python]")

df_final = pd.concat([df_workers_out, df_disc_out], ignore_index=True)

# Controles de integridad
dup = int(df_final.duplicated(subset=[USER_COL]).sum())
assert dup == 0, f"df_final tiene usuarios duplicados: {dup}"

_n_expected = int(len(df_workers_out)) + int(len(df_disc_out))
assert len(df_final) == _n_expected, (
    f"df_final tiene {len(df_final):,} filas; se esperaban {_n_expected:,}."
)

_n_unassigned = int(df_final["final_segment"].isna().sum())
if _n_unassigned:
    raise ValueError(
        f"{_n_unassigned:,} usuarios quedaron sin segmento asignado. "
        "Revisar el mapeo de la Sección 10.2 y las etiquetas de la Sección 7."
    )

print("[10.3] df_final:", df_final.shape)
print("[10.3] distribución de final_segment:")
_counts_fs = df_final["final_segment"].value_counts(dropna=False)
for _seg, _n in _counts_fs.items():
    print(f"       {str(_seg):<18} {int(_n):>6,}  ({_n / max(len(df_final), 1):.2%})")

print(
    f"[10.3] procedencia: {int(len(df_workers_out)):,} desde clustering | "
    f"{int(len(df_disc_out)):,} asignados por regla"
)

log_jsonl(
    paths["logs_dir"] / "final.jsonl",
    {
        "event": "final_consolidated",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "n_total": int(len(df_final)),
        "n_from_clustering": int(len(df_workers_out)),
        "n_from_rules": int(len(df_disc_out)),
        "dist_final_segment": {str(k_): int(v) for k_, v in _counts_fs.items()},
    },
)

del _n_expected, _n_unassigned, _counts_fs
_ = gc.collect()

In [ ]:
# ------------------------------------------------------------
# 10.4 — Persistencia final + tablas resumen
# ------------------------------------------------------------
# Artefacto de Nivel 2 con la restricción más fuerte del cuaderno: combina
# identificador seudonimizado, anclas residencial y laboral inferidas, y una
# condición ocupacional derivada. Deriva de la fuente telco, se escribe en el
# directorio de artefactos y no forma parte de la entrega pública. Su uso queda
# restringido al entorno autorizado; cualquier difusión de resultados debe ser
# agregada y no permitir la reidentificación de casos individuales.
final_path = paths["tables_dir"] / f"final_user_labels_{FEATURE_SET}.parquet"
df_final.to_parquet(final_path, index=False)

# Resumen por conglomerado (solo clusterizados). Agregado, sin identificadores.
cluster_modality = (
    df_workers_out.groupby(["cluster_id", "work_modality_3"], observed=True, sort=True)[USER_COL]
    .nunique()
    .reset_index(name="n_users")
    .sort_values(["cluster_id", "n_users"], ascending=[True, False], kind="mergesort")
)
cluster_modality_path = paths["tables_dir"] / f"cluster_modality_counts_{FEATURE_SET}.parquet"
cluster_modality.to_parquet(cluster_modality_path, index=False)

log_jsonl(
    paths["logs_dir"] / "final.jsonl",
    {
        "event": "final_saved",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "final_path": str(final_path),
        "cluster_modality_path": str(cluster_modality_path),
        "cluster_profiles_path": str(profiles_path),
        "profile_fig_path": str(prof_fig_path),
        "mod3_fig_path": str(mod3_fig_path),
    },
)

print("[10.4] guardado:", final_path.name)
print("[10.4] correspondencia conglomerado → modalidad:")
print(cluster_modality.to_string(index=False))

_ = gc.collect()

In [ ]:
# ------------------------------------------------------------
# 10.5 — Limpieza RAM
# ------------------------------------------------------------
del df_workers, df_workers_out, df_disc, df_disc_out, cluster_modality
_ = gc.collect()

print("[Section 10] OK — dataset final generado (cinco segmentos) para FEATURE_SET =", FEATURE_SET)

## *11. Brazo POIs (Augmented)*

Esta sección construye variables de contexto territorial a partir de
OpenStreetMap y las asocia a cada usuario mediante sus anclas.

El procedimiento tiene tres etapas. Primero se ingesta el extracto nacional
`.pbf`, se filtran las entidades con contenido semántico y se canoniza un
catálogo de puntos de interés indexado en H3. Segundo, se agregan conteos por
grupo temático dentro de anillos H3 en torno a las anclas residencial y laboral
de cada usuario, para dos radios. Tercero, se integran esas variables al
resultado de las secciones anteriores.

**Modo de integración.** El brazo POI opera en modo `audit_only`: las variables
se calculan, se persisten y quedan disponibles para caracterizar los segmentos,
pero **no alimentan el clustering**. La rama alternativa (`train_ablation`) está
implementada pero no se ejecuta en la corrida documentada. El manuscrito declara
esa ablación como no habilitada, por la desproporción entre el número de
variables POI resultantes y el tamaño de la cohorte.

**Nota sobre la discretización de los radios.** Los buffers se aproximan
mediante anillos H3 (`grid_disk`) en lugar de círculos exactos. El número de
anillos se deriva del radio dividido por el paso centro-centro entre celdas
vecinas, redondeado hacia arriba. La región resultante es hexagonal y
sobreestima el área circular nominal; la densidad se calcula sobre el área
circular teórica, de modo que es una aproximación y no una medida exacta.

**Nota sobre las anclas.** Las variables se construyen alrededor de ubicaciones
inferidas. Su incorporación incrementa la especificidad del perfil por usuario,
lo que el manuscrito identifica como una de las fuentes de amplificación del
riesgo de reidentificación. Los artefactos resultantes no forman parte de la
entrega pública.

### **11.0 Setup POIs + control experimental**

In [ ]:
# ============================================================
# 11.0 — Setup POIs + control experimental
# Objetivo:
#   - Parametrizar el brazo POIs (variante augmented)
#   - Definir la unidad espacial de las variables
#   - Definir el modo de filtro sobre el extracto OSM
#   - Crear sub-run POIs con directorios y logging trazables
# ============================================================

# ----------------------------
# 11.0.0 — Guardrails
# ----------------------------
if cfg.variant != "augmented":
    raise ValueError("La Sección 11 requiere cfg.variant='augmented' (brazo POIs).")

if cfg.osm_pbf_path is None:
    raise ValueError("cfg.osm_pbf_path es None. El brazo POIs requiere el extracto OSM.")

# La existencia del archivo ya fue verificada por cfg.validate() en la Sección 0.2.
pbf_path = Path(cfg.osm_pbf_path)

POI_LOG = paths["logs_dir"] / "pois.jsonl"

# ----------------------------
# 11.0.1 — Definiciones experimentales
# ----------------------------
POIUnit = Literal["buffers_home_work", "by_h3", "by_commune"]
POIFilterMode = Literal["SEMANTIC_MIN", "DG_FULL"]

# Unidad espacial de las variables POI.
# 'buffers_home_work' construye el contexto alrededor de las anclas del usuario,
# que es la unidad pertinente para caracterizar modalidad laboral.
POI_UNIT: POIUnit = "buffers_home_work"

# Radios de los buffers, en metros. Se discretizan en anillos H3 (ver 11.2.1).
POI_BUFFERS_M: Tuple[int, ...] = (300, 800)

# CRS métrico de referencia para la región del caso.
POI_METRIC_EPSG = 32719

# Recorte espacial opcional (WGS84): (min_lon, min_lat, max_lon, max_lat).
# None procesa el extracto completo. Acotar la bbox reduce sustantivamente el
# tiempo y la memoria de la ingesta cuando el ámbito territorial es conocido.
POI_BBOX_4326: Optional[Tuple[float, float, float, float]] = None

# Modo de filtro sobre las entidades OSM:
#   SEMANTIC_MIN : retiene entidades con contenido semántico explícito
#                  (amenity, shop, tourism, office, healthcare, transporte, etc.).
#                  Excluye las que solo declaran 'building', que aportan volumen
#                  sin información funcional para caracterizar contexto laboral.
#   DG_FULL      : replica el WHERE del osm_query.yaml de Deep Gravity, que sí
#                  incluye 'building'. Produce un catálogo mayor y menos
#                  específico. Se conserva para comparabilidad entre prototipos.
POI_FILTER_MODE: POIFilterMode = "SEMANTIC_MIN"

# Capas lineales (red vial, ferroviaria). Desactivadas: aportan infraestructura
# de conectividad, no puntos de interés funcionales.
POI_INCLUDE_LINES_LIKE = False

# Saneamiento geométrico para multipolygons (anillos no cerrados en el extracto)
POI_FIX_POLYGONS_GEOM = True

# Reducir RAM: descartar other_tags tras la selección final
POI_DROP_OTHER_TAGS_AFTER_SELECT = True

# H3 para POIs: misma resolución que pings y stays, de modo que el join por
# celda sea directo y sin reproyección de índices.
POI_H3_RES = int(cfg.h3_res_main)
POI_H3_BATCH_SIZE = 400_000

# Claves usadas para construir el filtro y el catálogo
POI_WHERE_KEYS = (
    "amenity", "shop", "tourism", "office", "healthcare", "emergency",
    "public_transport", "railway",
    "landuse", "leisure", "historic", "sport", "aeroway", "building",
)

POI_SELECT_KEYS_POINTS = (
    "name", "amenity", "shop", "tourism", "office", "healthcare", "emergency",
    "public_transport", "railway",
    "landuse", "leisure", "historic", "sport", "aeroway", "building",
    "opening_hours", "operator",
)
POI_SELECT_KEYS_POLYGONS = POI_SELECT_KEYS_POINTS

# Claves adicionales del YAML de Deep Gravity (solo si POI_FILTER_MODE="DG_FULL")
DG_EXTRA_WHERE_KEYS = (
    "highway",
)
DG_SELECT_KEYS_LINES = (
    "name", "highway", "railway", "aeroway", "bridge", "oneway", "surface", "tunnel", "width",
)


def make_poi_run_id(prefix: str = "pois") -> str:
    short = uuid.uuid4().hex[:8]
    return f"{prefix}_{cfg.run_id}_{short}"


POI_RUN_ID = make_poi_run_id()

poi_dirs = {
    "poi_dir": paths["run_dir"] / "pois" / POI_RUN_ID,
    "poi_cache": paths["run_dir"] / "pois" / POI_RUN_ID / "cache",
    "poi_tables": paths["run_dir"] / "pois" / POI_RUN_ID / "tables",
    "poi_figures": paths["run_dir"] / "pois" / POI_RUN_ID / "figures",
}
for p in poi_dirs.values():
    p.mkdir(parents=True, exist_ok=True)

print("[11.0] POI_RUN_ID:", POI_RUN_ID)


# ----------------------------
# 11.0.2 — Config dataclass (trazabilidad)
# ----------------------------
@dataclass(frozen=True)
class POIConfig:
    poi_run_id: str
    pbf_path: Path
    unit: POIUnit
    filter_mode: POIFilterMode
    buffers_m: Tuple[int, ...]
    metric_epsg: int
    bbox_4326: Optional[Tuple[float, float, float, float]]
    include_lines_like: bool
    fix_polygons_geom: bool
    drop_other_tags_after_select: bool
    poi_h3_res: int
    poi_h3_batch_size: int
    where_keys: Tuple[str, ...]
    select_keys_points: Tuple[str, ...]
    select_keys_polygons: Tuple[str, ...]
    dg_extra_where_keys: Tuple[str, ...]
    dg_select_keys_lines: Tuple[str, ...]

    def validate(self) -> None:
        if not self.pbf_path.exists():
            raise FileNotFoundError(f"OSM .pbf no encontrado: {self.pbf_path}")
        if self.unit == "buffers_home_work":
            if not self.buffers_m or any(int(b) <= 0 for b in self.buffers_m):
                raise ValueError("buffers_m debe contener radios positivos (m).")
        if self.metric_epsg <= 0:
            raise ValueError("metric_epsg inválido.")
        if self.bbox_4326 is not None:
            minx, miny, maxx, maxy = self.bbox_4326
            if not (minx < maxx and miny < maxy):
                raise ValueError("bbox_4326 inválida (min < max).")
        if self.poi_h3_res <= 0:
            raise ValueError("poi_h3_res inválido.")
        if self.poi_h3_batch_size <= 0:
            raise ValueError("poi_h3_batch_size inválido.")


poi_cfg = POIConfig(
    poi_run_id=POI_RUN_ID,
    pbf_path=pbf_path,
    unit=POI_UNIT,
    filter_mode=POI_FILTER_MODE,
    buffers_m=tuple(int(b) for b in POI_BUFFERS_M),
    metric_epsg=int(POI_METRIC_EPSG),
    bbox_4326=POI_BBOX_4326,
    include_lines_like=bool(POI_INCLUDE_LINES_LIKE),
    fix_polygons_geom=bool(POI_FIX_POLYGONS_GEOM),
    drop_other_tags_after_select=bool(POI_DROP_OTHER_TAGS_AFTER_SELECT),
    poi_h3_res=int(POI_H3_RES),
    poi_h3_batch_size=int(POI_H3_BATCH_SIZE),
    where_keys=tuple(POI_WHERE_KEYS),
    select_keys_points=tuple(POI_SELECT_KEYS_POINTS),
    select_keys_polygons=tuple(POI_SELECT_KEYS_POLYGONS),
    dg_extra_where_keys=tuple(DG_EXTRA_WHERE_KEYS),
    dg_select_keys_lines=tuple(DG_SELECT_KEYS_LINES),
)
poi_cfg.validate()


# ----------------------------
# 11.0.3 — Logging de configuración
# ----------------------------
def log_poi_jsonl(payload: Dict[str, Any]) -> None:
    log_jsonl(POI_LOG, payload)


log_poi_jsonl(
    {
        "event": "poi_setup",
        "ts_utc": now_utc(),
        "run_id": cfg.run_id,
        "poi_run_id": poi_cfg.poi_run_id,
        "variant": cfg.variant,
        "feature_set": globals().get("FEATURE_SET", None),
        "poi_config": {k: (str(v) if isinstance(v, Path) else v) for k, v in asdict(poi_cfg).items()},
    }
)

print(
    f"[11.0] configuración: unidad={poi_cfg.unit} | filtro={poi_cfg.filter_mode} | "
    f"radios={list(poi_cfg.buffers_m)} m | H3 res={poi_cfg.poi_h3_res}"
)
print("[11.0] POIConfig OK — listo para 11.1 (ingesta y normalización).")

### **11.1 Ingesta OSM y catálogo POI canonizado (H3)**

In [ ]:
# ============================================================
# 11.1 — Ingesta y normalización OSM (PBF)
# Salidas (poi_cache):
#   - pois_catalog.parquet    (sin geometry: poi_lat/poi_lon + h3_id_poi)
#   - pois_catalog_qa.parquet (QA ligero)
# ============================================================

# ----------------------------
# 11.1.0 — Guardrails
# ----------------------------
try:
    _ = poi_cfg
    _ = poi_dirs
except NameError as e:
    raise NameError("Se requiere ejecutar la Sección 11.0 antes (poi_cfg, poi_dirs).") from e

# Silencia el warning de anillos no cerrados: el extracto nacional los contiene
# de forma habitual y el saneamiento geométrico posterior los resuelve.
warnings.filterwarnings("ignore", message="Non closed ring detected.*", category=RuntimeWarning)

print("[11.1] poi_run_id:", poi_cfg.poi_run_id)
print("[11.1] extracto:", poi_cfg.pbf_path.name)
print("[11.1] filter_mode:", poi_cfg.filter_mode)
print("[11.1] include_lines_like:", poi_cfg.include_lines_like)
print("[11.1] bbox_4326:", poi_cfg.bbox_4326)

In [ ]:
# ----------------------------
# 11.1.1 — Helpers robustos
# ----------------------------
def _fieldnames_from_info(info: dict) -> list[str]:
    """
    Extrae nombres de campo desde el resultado de pyogrio.read_info.

    El formato de 'fields' varía entre versiones de pyogrio y GDAL: puede venir
    como lista de dicts, como lista de tuplas (name, type) o como arreglo de
    nombres. Se intentan las tres formas en orden.
    """
    fields = info.get("fields", None)
    if fields is None:
        return []

    # Caso: lista de dicts {"name":..., "type":...}
    try:
        if isinstance(fields, (list, tuple)) and len(fields) and isinstance(fields[0], dict) and "name" in fields[0]:
            return [f["name"] for f in fields]
    except Exception:
        pass

    # Caso: lista de tuplas (name, type)
    try:
        if isinstance(fields, (list, tuple)) and len(fields) and isinstance(fields[0], (list, tuple)) and len(fields[0]) >= 1:
            return [f[0] for f in fields]
    except Exception:
        pass

    # Fallback: conversión directa
    try:
        return list(fields)
    except Exception:
        return []


def _layer_fields(pbf: Path, layer: str) -> list[str]:
    info = read_info(pbf, layer=layer, force_feature_count=False)
    return _fieldnames_from_info(info)


def _read_layer_min(
    pbf: Path,
    layer: str,
    keys_maybe_promoted: list[str],
    *,
    bbox_4326: Optional[Tuple[float, float, float, float]] = None,
    verbose: bool = True,
) -> gpd.GeoDataFrame:
    """
    Lee una capa del extracto con el mínimo de columnas necesario:
    osm_id, other_tags y geometry, más las claves que el driver ya promovió
    a columnas propias en el esquema de esa capa.

    Qué claves vienen promovidas depende de la configuración del driver OSM y
    varía entre capas: por eso se consulta el esquema en lugar de asumirlo.
    """
    fields = set(_layer_fields(pbf, layer))
    base_cols = [c for c in ["osm_id", "other_tags"] if c in fields]
    promoted = [k for k in keys_maybe_promoted if k in fields]
    cols = base_cols + promoted

    if bbox_4326 is None:
        gdf = gpd.read_file(pbf, engine="pyogrio", layer=layer, columns=cols)
    else:
        gdf = gpd.read_file(pbf, engine="pyogrio", layer=layer, columns=cols, bbox=bbox_4326)

    if verbose:
        print(f"[11.1] capa '{layer}': filas={len(gdf):,} cols={len(gdf.columns)} | promovidas={len(promoted)}")
    return gdf


def ensure_tag_col(gdf: gpd.GeoDataFrame, key: str, col_name: Optional[str] = None) -> str:
    """
    Garantiza que una clave OSM esté disponible como columna.

    Si el driver la promovió, la usa directamente. Si no, la extrae desde
    other_tags mediante expresión regular, operando solo sobre las filas donde
    la clave aparece. Los dos puntos de las claves compuestas se sustituyen por
    doble guion bajo, porque no son válidos como nombre de columna en varios
    formatos de persistencia.

    Devuelve el nombre de columna utilizable.
    """
    col = col_name or key.replace(":", "__")
    if key in gdf.columns:
        return key
    if col in gdf.columns:
        return col

    gdf[col] = pd.NA
    if "other_tags" in gdf.columns:
        mask = gdf["other_tags"].notna() & gdf["other_tags"].str.contains(f'"{key}"=>', regex=False)
        if mask.any():
            pat = re.compile(rf'"{re.escape(key)}"=>"([^"]*)"')
            gdf.loc[mask, col] = gdf.loc[mask, "other_tags"].str.extract(pat, expand=False)
    return col


def _make_valid_polygons(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Sanea geometrías poligonales inválidas y descarta las irrecuperables."""
    if gdf.empty:
        return gdf
    if hasattr(gdf.geometry, "make_valid"):
        gdf["geometry"] = gdf.geometry.make_valid()
    else:
        gdf["geometry"] = gdf.buffer(0)
    gdf = gdf[gdf.geometry.notna() & gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
    return gdf


def _drop_other_tags(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if poi_cfg.drop_other_tags_after_select and "other_tags" in gdf.columns:
        return gdf.drop(columns=["other_tags"])
    return gdf


def _get_h3_funcs():
    latlng_to_cell = getattr(h3, "latlng_to_cell", None)
    cell_to_parent = getattr(h3, "cell_to_parent", None)
    if latlng_to_cell is not None and cell_to_parent is not None:
        return latlng_to_cell, cell_to_parent

    geo_to_h3 = getattr(h3, "geo_to_h3", None)
    h3_to_parent = getattr(h3, "h3_to_parent", None)
    if geo_to_h3 is None or h3_to_parent is None:
        raise ImportError("La versión instalada de 'h3' no expone latlng_to_cell/cell_to_parent ni geo_to_h3/h3_to_parent.")
    return geo_to_h3, h3_to_parent


def assign_h3_to_pois(
    lat: np.ndarray,
    lon: np.ndarray,
    res: int,
    batch_size: int,
    verbose: bool = True,
) -> np.ndarray:
    """Asigna índice H3 a cada POI, procesando por lotes."""
    latlng_to_cell, _ = _get_h3_funcs()
    n = int(lat.shape[0])
    out = np.empty(n, dtype=object)

    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        for i in range(start, end):
            out[i] = latlng_to_cell(float(lat[i]), float(lon[i]), int(res))
        if verbose:
            print(f"[11.1] H3 batch {start:,}-{end:,} / {n:,}")
    return out

In [ ]:
# ----------------------------
# 11.1.2 — Verificar layers requeridos
# ----------------------------
layers = list_layers(poi_cfg.pbf_path)
layer_names = set([name for name, _ in layers])

need_layers = {"points", "multipolygons"}
if poi_cfg.include_lines_like:
    need_layers |= {"lines", "multilinestrings"}

missing_layers = need_layers - layer_names
if missing_layers:
    raise FileNotFoundError(f"El extracto OSM no contiene las capas requeridas: {missing_layers}")

log_poi_jsonl(
    {
        "event": "pois_ingest_start",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "pbf_name": poi_cfg.pbf_path.name,
        "need_layers": sorted(list(need_layers)),
        "bbox_4326": poi_cfg.bbox_4326,
        "filter_mode": poi_cfg.filter_mode,
    },
)

In [ ]:
# ----------------------------
# 11.1.3 — Funciones de filtro (SEMANTIC_MIN vs DG_FULL)
# ----------------------------
def _build_mask_semantic_min(df: pd.DataFrame, cols: dict) -> pd.Series:
    """
    Retiene entidades con contenido semántico explícito.

    A diferencia de DG_FULL, NO incluye 'building' como criterio: una entidad
    que solo declara ser un edificio no aporta información funcional sobre el
    contexto de actividad, y su inclusión multiplica el volumen del catálogo
    sin mejorar la caracterización.
    """
    return (
        df[cols["amenity"]].notna()
        | df[cols["shop"]].notna()
        | df[cols["tourism"]].notna()
        | df[cols["office"]].notna()
        | df[cols["healthcare"]].notna()
        | df[cols["emergency"]].notna()
        | df[cols["public_transport"]].notna()
        | df[cols["railway"]].notna()
        | df[cols["landuse"]].notna()
        | df[cols["leisure"]].notna()
        | df[cols["historic"]].notna()
        | df[cols["sport"]].notna()
        | df[cols["aeroway"]].notna()
    )


def _build_mask_dg_full_points(df: pd.DataFrame, cols: dict) -> pd.Series:
    """
    Replica la cláusula WHERE de osm_query.yaml del repositorio de Deep Gravity
    para la capa de puntos. Incluye 'building', por lo que produce un catálogo
    sustancialmente mayor. Se conserva para comparabilidad entre prototipos.
    """
    bld = cols["building"]; shop = cols["shop"]; tour = cols["tourism"]
    amen = cols["amenity"]; off = cols["office"]; emer = cols["emergency"]
    hcare = cols["healthcare"]; luse = cols["landuse"]; leis = cols["leisure"]
    hist = cols["historic"]; spt = cols["sport"]; aero = cols["aeroway"]

    return (
        df[bld].notna()
        | df[shop].notna() | df[tour].notna()
        | df[amen].isin(["marketplace", "restaurant", "fast_food", "cafe", "bar", "pub"]) | df[off].notna()
        | df[amen].isin(["kindergarten", "school", "college", "university", "language_school"]) | (df[off] == "educational_institution")
        | df[emer].notna() | df[amen].isin(["police", "fire_station"])
        | df[hcare].notna() | df[amen].isin(["doctors", "dentist", "clinic", "toilets", "hospital", "pharmacy"]) | df[shop].isin(["herbalist", "nutrition_supplements"])
        | df[luse].notna() | (df[leis] == "park") | (df[amen] == "grave_yard")
        | (df[amen] == "fuel")
        | (df[amen] == "place_of_worship")
        | (df[amen] == "community_centre")
        | (df[amen] == "library")
        | df[hist].notna()
        | df[spt].notna() | df[leis].isin(["stadium", "swimming_pool", "pitch", "sport_centre"])
        | df[aero].notna() | (df[bld] == "aerodrome")
        | (df[amen] == "bus_station")
    )


def _build_mask_dg_full_polygons(df: pd.DataFrame, cols: dict) -> pd.Series:
    """
    Variante poligonal del WHERE de Deep Gravity: añade 'train_station', que
    en la capa de multipolygons aparece como valor de building.
    """
    m = _build_mask_dg_full_points(df, cols)
    bld = cols["building"]; amen = cols["amenity"]
    return m | (df[bld] == "train_station") | (df[amen] == "bus_station")

In [ ]:
# ----------------------------
# 11.1.4 — POINTS: lectura mínima + WHERE + SELECT
# ----------------------------
t0 = now_utc()

points = _read_layer_min(
    poi_cfg.pbf_path,
    layer="points",
    keys_maybe_promoted=list(poi_cfg.where_keys) + list(poi_cfg.select_keys_points),
    bbox_4326=poi_cfg.bbox_4326,
    verbose=True,
)

where_cols_pts = {k: ensure_tag_col(points, k) for k in poi_cfg.where_keys}

if poi_cfg.filter_mode == "SEMANTIC_MIN":
    m_points = _build_mask_semantic_min(points, where_cols_pts)
else:
    m_points = _build_mask_dg_full_points(points, where_cols_pts)

points_filt = points.loc[m_points].copy()
n_in_pts = int(len(points))
n_out_pts = int(len(points_filt))

# Extracción diferida de las columnas del SELECT: se construyen solo sobre el
# subconjunto filtrado, no sobre la capa completa.
sel_map_pts = {k: ensure_tag_col(points_filt, k) for k in poi_cfg.select_keys_points}
cols_pts = ["osm_id"] + list(sel_map_pts.values()) + ["geometry"]
if "other_tags" in points_filt.columns:
    cols_pts += ["other_tags"]
cols_pts = [c for c in cols_pts if c in points_filt.columns]

points_out = points_filt[cols_pts].copy()
points_out["source_layer"] = "points"
points_out = _drop_other_tags(points_out)

points_out["poi_lon"] = points_out.geometry.x.astype(np.float32)
points_out["poi_lat"] = points_out.geometry.y.astype(np.float32)

log_poi_jsonl(
    {
        "event": "pois_points_extracted",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "rows_in": n_in_pts,
        "rows_out": n_out_pts,
        "pct_kept": float(n_out_pts / max(n_in_pts, 1)),
        "elapsed_s": (now_utc() - t0).total_seconds(),
        "mem_mb_points_out": mem_mb_df(points_out),
        "filter_mode": poi_cfg.filter_mode,
    },
)

print(f"[11.1 POINTS] retenidos={n_out_pts:,} de {n_in_pts:,} ({n_out_pts / max(n_in_pts, 1):.2%})")

del points, points_filt, m_points, where_cols_pts, sel_map_pts
_ = gc.collect()

In [ ]:
# ----------------------------
# 11.1.5 — MULTIPOLYGONS: lectura mínima + WHERE + SELECT
# ----------------------------
t1 = now_utc()

multipolygons = _read_layer_min(
    poi_cfg.pbf_path,
    layer="multipolygons",
    keys_maybe_promoted=list(poi_cfg.where_keys) + list(poi_cfg.select_keys_polygons),
    bbox_4326=poi_cfg.bbox_4326,
    verbose=True,
)

where_cols_poly = {k: ensure_tag_col(multipolygons, k) for k in poi_cfg.where_keys}

if poi_cfg.filter_mode == "SEMANTIC_MIN":
    m_poly = _build_mask_semantic_min(multipolygons, where_cols_poly)
else:
    m_poly = _build_mask_dg_full_polygons(multipolygons, where_cols_poly)

poly_filt = multipolygons.loc[m_poly].copy()
n_in_poly = int(len(multipolygons))
n_out_pre = int(len(poly_filt))

if poi_cfg.fix_polygons_geom:
    poly_filt = _make_valid_polygons(poly_filt)

n_out_poly = int(len(poly_filt))

sel_map_poly = {k: ensure_tag_col(poly_filt, k) for k in poi_cfg.select_keys_polygons}
cols_poly = ["osm_id"] + list(sel_map_poly.values()) + ["geometry"]
if "other_tags" in poly_filt.columns:
    cols_poly += ["other_tags"]
cols_poly = [c for c in cols_poly if c in poly_filt.columns]

polygons_out = poly_filt[cols_poly].copy()
polygons_out["source_layer"] = "multipolygons"
polygons_out = _drop_other_tags(polygons_out)

# Conversión a punto representativo. Homogeneiza el catálogo: puntos y
# polígonos pasan a tratarse como entidades puntuales, lo que permite indexar
# todo en H3 con un único criterio. Se usa representative_point y no centroid
# porque garantiza un punto interior al polígono incluso en formas cóncavas.
rep = polygons_out.geometry.representative_point()
polygons_out["poi_lon"] = rep.x.astype(np.float32)
polygons_out["poi_lat"] = rep.y.astype(np.float32)
polygons_out["geometry"] = rep

log_poi_jsonl(
    {
        "event": "pois_polygons_extracted",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "rows_in": n_in_poly,
        "rows_out_pre_fix": n_out_pre,
        "rows_out": n_out_poly,
        "pct_kept": float(n_out_poly / max(n_in_poly, 1)),
        "geom_fix_applied": bool(poi_cfg.fix_polygons_geom),
        "elapsed_s": (now_utc() - t1).total_seconds(),
        "mem_mb_polygons_out": mem_mb_df(polygons_out),
        "filter_mode": poi_cfg.filter_mode,
    },
)

print(
    f"[11.1 POLYGONS] retenidos={n_out_poly:,} de {n_in_poly:,} "
    f"({n_out_poly / max(n_in_poly, 1):.2%}) | descartados por geometría inválida={n_out_pre - n_out_poly:,}"
)

del multipolygons, poly_filt, m_poly, where_cols_poly, sel_map_poly, rep
_ = gc.collect()

In [ ]:
# ----------------------------
# 11.1.6 — LINES-like (punto de extensión, no ejercido)
# ----------------------------
# Las capas lineales (red vial, ferroviaria) no se procesan en esta versión.
# El brazo POI caracteriza contexto funcional mediante entidades puntuales; la
# infraestructura de conectividad es una dimensión distinta que requeriría
# métricas propias (densidad de vía, accesibilidad) y no conteos por grupo.
# La bandera se conserva como punto de extensión declarado.
if poi_cfg.include_lines_like:
    log_poi_jsonl(
        {
            "event": "pois_lines_skipped",
            "ts_utc": now_utc(),
            "note": "include_lines_like=True es un punto de extensión declarado, no implementado en esta versión.",
        }
    )
    print(
        "[11.1 LINES] include_lines_like=True, pero el procesamiento de capas "
        "lineales no está implementado en esta versión. Se continúa sin ellas."
    )

In [ ]:
# ----------------------------
# 11.1.7 — Consolidación del catálogo POI + H3 + persistencia
# ----------------------------
t3 = now_utc()

pois = pd.concat([points_out, polygons_out], ignore_index=True)
pois = gpd.GeoDataFrame(pois, geometry="geometry", crs="EPSG:4326")

lat = pois["poi_lat"].to_numpy(dtype=np.float64, copy=False)
lon = pois["poi_lon"].to_numpy(dtype=np.float64, copy=False)

h3_id_poi = assign_h3_to_pois(
    lat=lat,
    lon=lon,
    res=int(poi_cfg.poi_h3_res),
    batch_size=int(poi_cfg.poi_h3_batch_size),
    verbose=True,
)
pois["h3_id_poi"] = pd.Categorical(h3_id_poi)

# Catálogo sin geometry: la posición queda representada por poi_lat/poi_lon y
# por el índice H3, que es lo que consume la Sección 11.2.
cols_keep = ["osm_id", "source_layer", "poi_lat", "poi_lon", "h3_id_poi"]
for k in poi_cfg.select_keys_points:
    col = k.replace(":", "__")
    if k in pois.columns:
        cols_keep.append(k)
    elif col in pois.columns:
        cols_keep.append(col)

cols_keep = list(dict.fromkeys([c for c in cols_keep if c in pois.columns]))
pois_catalog = pd.DataFrame(pois[cols_keep]).copy()

# Artefacto derivado exclusivamente de OpenStreetMap (fuente pública). No
# contiene información de la fuente telco. Se escribe en el directorio de
# artefactos por volumen y por ser regenerable desde el extracto.
catalog_path = poi_dirs["poi_cache"] / "pois_catalog.parquet"
pois_catalog.to_parquet(catalog_path, index=False)

# QA ligero: cobertura de las claves temáticas en el catálogo resultante
qa_keys = [
    "amenity", "shop", "tourism", "office", "healthcare",
    "public_transport", "railway", "landuse", "leisure",
    "historic", "sport", "aeroway", "building",
]
qa_cols = []
for k in qa_keys:
    if k in pois_catalog.columns:
        qa_cols.append(k)
    else:
        kk = k.replace(":", "__")
        if kk in pois_catalog.columns:
            qa_cols.append(kk)

qa_base = ["osm_id", "source_layer", "h3_id_poi"]
qa_all = list(dict.fromkeys(qa_base + qa_cols))

if len(qa_all) != len(set(qa_all)):
    raise ValueError(f"Las columnas de QA no son únicas: {qa_all}")

pois_qa = pd.DataFrame(pois_catalog[qa_all]).copy()
qa_path = poi_dirs["poi_tables"] / "pois_catalog_qa.parquet"
pois_qa.to_parquet(qa_path, index=False)

log_poi_jsonl(
    {
        "event": "pois_catalog_saved",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "rows_pois": int(len(pois_catalog)),
        "rows_points": int(len(points_out)),
        "rows_polygons": int(len(polygons_out)),
        "n_h3_cells": int(pois_catalog["h3_id_poi"].nunique()),
        "filter_mode": poi_cfg.filter_mode,
        "poi_h3_res": int(poi_cfg.poi_h3_res),
        "elapsed_s": (now_utc() - t3).total_seconds(),
        "mem_mb_catalog": mem_mb_df(pois_catalog),
    },
)

print(
    f"[11.1 DONE] catálogo: {len(pois_catalog):,} POIs "
    f"({len(points_out):,} puntos + {len(polygons_out):,} polígonos) | "
    f"celdas H3 distintas: {pois_catalog['h3_id_poi'].nunique():,}"
)

del pois, pois_catalog, pois_qa, lat, lon, h3_id_poi, points_out, polygons_out
_ = gc.collect()

### **11.2 Features POIs por usuario en H3 k-ring**

In [ ]:
# ============================================================
# 11.2 — Construcción de variables POI por usuario
# Requiere:
#   - Sección 11.0 (poi_cfg, poi_dirs)
#   - Sección 11.1 (poi_cache/pois_catalog.parquet)
#   - homes.parquet, works.parquet (Secciones 4 y 5)
#   - gatekeeper_eligible_users.parquet (Sección 3)
# Salida:
#   - poi_tables/poi_features_{FEATURE_SET}.parquet
# ============================================================
USER_COL = cfg.col_user

# ----------------------------
# 11.2.0 — Guardrails + inputs
# ----------------------------
try:
    _ = poi_cfg
    _ = poi_dirs
except NameError as e:
    raise NameError("Se requiere ejecutar la Sección 11.0 antes (poi_cfg, poi_dirs).") from e

FEATURE_SET = globals().get("FEATURE_SET", "UNKNOWN")

if poi_cfg.unit != "buffers_home_work":
    raise ValueError("Esta implementación cubre unit='buffers_home_work'. Ajuste poi_cfg.unit si corresponde.")

catalog_path = poi_dirs["poi_cache"] / "pois_catalog.parquet"
if not catalog_path.exists():
    raise FileNotFoundError(f"No existe {catalog_path.name}. Ejecute la Sección 11.1 primero.")

# Homes/Works (cargar si no están en RAM)
if "df_homes" not in globals():
    df_homes = pd.read_parquet(paths["tables_dir"] / "homes.parquet")
if "df_works" not in globals():
    df_works = pd.read_parquet(paths["tables_dir"] / "works.parquet")

# Universo objetivo: elegibles del gatekeeper si existe; si no, los de homes
gk_users_path = paths["tables_dir"] / "gatekeeper_eligible_users.parquet"
if gk_users_path.exists():
    df_gk_users = pd.read_parquet(gk_users_path)
    users_target = df_gk_users[USER_COL]
else:
    users_target = df_homes[USER_COL]

users_target = users_target.dropna().drop_duplicates().reset_index(drop=True)
print("[11.2] users_target:", int(users_target.shape[0]))

# Merge de anclas (home/work)
df_anchor = (
    users_target.to_frame(USER_COL)
    .merge(df_homes[[USER_COL, "home_h3", "home_missing"]], on=USER_COL, how="left")
    .merge(df_works[[USER_COL, "work_h3", "work_missing"]], on=USER_COL, how="left")
)

for c in ["home_missing", "work_missing"]:
    df_anchor[c] = pd.to_numeric(df_anchor[c], errors="coerce").fillna(1).astype(np.int8)

home_ok = (df_anchor["home_missing"] == 0) & df_anchor["home_h3"].notna()
work_ok = (df_anchor["work_missing"] == 0) & df_anchor["work_h3"].notna()

print(
    f"[11.2] cobertura de anclas: home={int(home_ok.sum()):,} | work={int(work_ok.sum()):,} "
    f"de {int(len(df_anchor)):,} usuarios"
)

In [ ]:
# ----------------------------
# 11.2.1 — Helpers H3 + conversión de radio a número de anillos
# ----------------------------
def _get_h3_disk_func():
    """Compatibilidad h3-py: grid_disk (nuevo) o k_ring (antiguo)."""
    f = getattr(h3, "grid_disk", None)
    if f is not None:
        return f
    f = getattr(h3, "k_ring", None)
    if f is None:
        raise ImportError("La versión instalada de h3 no expone grid_disk ni k_ring.")
    return f


def _get_h3_edge_len_m(res: int) -> float:
    """Longitud media de arista de una celda H3, en metros."""
    f = getattr(h3, "average_hexagon_edge_length", None)
    if f is not None:
        return float(f(int(res), unit="m"))
    f = getattr(h3, "edge_length", None)
    if f is not None:
        try:
            return float(f(int(res), unit="m"))
        except Exception:
            pass
    # Fallback conservador para resolución 9
    return 180.0


H3_DISK = _get_h3_disk_func()
EDGE_M = _get_h3_edge_len_m(int(poi_cfg.poi_h3_res))
STEP_M = float(EDGE_M * np.sqrt(3.0))  # distancia centro-centro entre vecinos


def meters_to_k(radius_m: int) -> int:
    """
    Convierte un radio en metros al número de anillos H3 que lo cubren.

    La región resultante de grid_disk(cell, k) es hexagonal y circunscribe al
    círculo de radio nominal: al redondear hacia arriba, el área efectivamente
    cubierta excede la del círculo. La densidad de la Sección 11.2.4 se calcula
    sobre el área circular teórica, de modo que subestima el denominador real y
    es por tanto una aproximación, no una medida exacta.
    """
    if radius_m <= 0:
        return 0
    return int(np.ceil(float(radius_m) / max(STEP_M, 1e-6)))


print(f"[11.2] H3 res={poi_cfg.poi_h3_res} | arista≈{EDGE_M:.1f} m | paso centro-centro≈{STEP_M:.1f} m")

buffers_m = [int(b) for b in poi_cfg.buffers_m]
buffers_k = {r: meters_to_k(r) for r in buffers_m}
print("[11.2] radios (m) → anillos (k):", buffers_k)

# Área circular teórica, en km², para el cálculo de densidad
area_km2 = {r: (np.pi * (r ** 2) / 1e6) for r in buffers_m}

In [ ]:
# ----------------------------
# 11.2.2 — Cargar POIs mínimos + asignar poi_group
# ----------------------------
schema_cols = set(pq.read_schema(catalog_path).names)


def pick_col(key: str) -> str | None:
    if key in schema_cols:
        return key
    kk = key.replace(":", "__")
    if kk in schema_cols:
        return kk
    return None


need_cols = ["osm_id", "source_layer", "h3_id_poi"]
tag_keys = [
    "amenity", "shop", "tourism", "office", "healthcare", "emergency",
    "landuse", "leisure", "historic", "sport", "aeroway", "building",
    "public_transport", "railway",
]
for k in tag_keys:
    c = pick_col(k)
    if c is not None:
        need_cols.append(c)

need_cols = list(dict.fromkeys(need_cols))
pois_cat = pd.read_parquet(catalog_path, columns=need_cols)

# Normalizar h3_id_poi a string estable (evita problemas con category)
pois_cat["h3_id_poi"] = pois_cat["h3_id_poi"].astype("string[python]")

# Asegurar columnas de tags con nombre canónico
for k in tag_keys:
    c = pick_col(k)
    out_c = k
    if out_c not in pois_cat.columns:
        if c is not None and c in pois_cat.columns:
            pois_cat[out_c] = pois_cat[c]
        else:
            pois_cat[out_c] = pd.NA

for k in tag_keys:
    pois_cat[k] = pois_cat[k].astype("string[python]")

# ----------------------------
# Taxonomía temática
# ----------------------------
# La asignación es JERÁRQUICA y EXCLUYENTE: np.select evalúa las condiciones en
# orden y asigna la primera que se cumple. Un POI que satisface varias —una
# farmacia dentro de un centro comercial, por ejemplo— recibe solo la primera
# de la lista. El orden refleja especificidad decreciente: las categorías
# funcionales específicas (educación, salud, seguridad) preceden a las
# genéricas (comercio, oficina, turismo). Cambiar el orden cambia los conteos.
amen = pois_cat["amenity"]
shop = pois_cat["shop"]
tour = pois_cat["tourism"]
offc = pois_cat["office"]
hcare = pois_cat["healthcare"]
emer = pois_cat["emergency"]
luse = pois_cat["landuse"]
leis = pois_cat["leisure"]
hist = pois_cat["historic"]
sport = pois_cat["sport"]
aero = pois_cat["aeroway"]
bld = pois_cat["building"]
pt = pois_cat["public_transport"]
rail = pois_cat["railway"]


def _b(x: pd.Series) -> np.ndarray:
    return x.fillna(False).to_numpy(dtype=bool, copy=False)


is_food = _b(amen.isin(["marketplace", "restaurant", "fast_food", "cafe", "bar", "pub"]))
is_edu = _b(amen.isin(["kindergarten", "school", "college", "university", "language_school"]) | (offc == "educational_institution"))
is_safety = _b(emer.notna() | amen.isin(["police", "fire_station"]))
is_health = _b(hcare.notna() | amen.isin(["doctors", "dentist", "clinic", "toilets", "hospital", "pharmacy"]) | shop.isin(["herbalist", "nutrition_supplements"]))
is_transport = _b(amen.isin(["bus_station"]) | (bld == "train_station") | pt.notna() | rail.notna())
is_aeroway = _b(aero.notna() | (bld == "aerodrome"))
is_fuel = _b(amen == "fuel")
is_worship = _b(amen == "place_of_worship")
is_community = _b(amen == "community_centre")
is_library = _b(amen == "library")
is_historic = _b(hist.notna())
is_sport = _b(sport.notna() | leis.isin(["stadium", "swimming_pool", "pitch", "sport_centre"]))
is_landuse = _b(luse.notna() | (leis == "park") | (amen == "grave_yard"))
is_shop = _b(shop.notna())
is_office = _b(offc.notna())
is_tourism = _b(tour.notna())

choices = [
    "education", "health", "public_safety", "transport", "aeroway",
    "fuel", "worship", "community", "library", "historic", "sport",
    "landuse_green", "food", "shop", "office", "tourism",
]
conds = [
    is_edu, is_health, is_safety, is_transport, is_aeroway,
    is_fuel, is_worship, is_community, is_library, is_historic, is_sport,
    is_landuse, is_food, is_shop, is_office, is_tourism,
]

pois_cat["poi_group"] = pd.Categorical(np.select(conds, choices, default="other"))
POI_GROUPS = list(pois_cat["poi_group"].cat.categories)

print(f"[11.2] grupos temáticos ({len(POI_GROUPS)}):", POI_GROUPS)

In [ ]:
# ----------------------------
# 11.2.3 — Pre-agregación: conteos por (h3_id_poi, poi_group)
# ----------------------------
# Se agrega una sola vez por celda y grupo. La Sección 11.2.4 hace joins contra
# esta tabla en lugar de recorrer el catálogo completo por usuario.
counts_h3 = (
    pois_cat.groupby(["h3_id_poi", "poi_group"], observed=True, sort=False)
    .size()
    .unstack("poi_group", fill_value=0)
)

counts_h3 = counts_h3.reindex(columns=POI_GROUPS, fill_value=0)
counts_h3 = counts_h3.astype(np.int32)

log_poi_jsonl(
    {
        "event": "poi_counts_by_h3_ready",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "n_h3_cells": int(counts_h3.shape[0]),
        "n_groups": int(counts_h3.shape[1]),
        "rows_pois": int(len(pois_cat)),
    },
)

print(f"[11.2] conteos por celda: {counts_h3.shape[0]:,} celdas × {counts_h3.shape[1]} grupos")

del pois_cat, amen, shop, tour, offc, hcare, emer, luse, leis, hist, sport, aero, bld, pt, rail
del is_food, is_edu, is_safety, is_health, is_transport, is_aeroway, is_fuel, is_worship
del is_community, is_library, is_historic, is_sport, is_landuse, is_shop, is_office, is_tourism
_ = gc.collect()

In [ ]:
# ----------------------------
# 11.2.4 — Sumar conteos en el anillo H3 alrededor de cada ancla
# ----------------------------
def shannon_entropy_from_counts_mat(cnt: np.ndarray) -> np.ndarray:
    """Entropía de Shannon sobre la distribución de conteos por grupo temático."""
    tot = cnt.sum(axis=1, keepdims=True).astype(np.float32)
    p = cnt.astype(np.float32) / np.clip(tot, 1.0, None)
    p = np.clip(p, 1e-12, 1.0)
    return (-np.sum(p * np.log(p), axis=1)).astype(np.float32)


def build_anchor_block(
    df_anchor: pd.DataFrame,
    anchor_col: str,
    anchor_ok_mask: np.ndarray,
    anchor_name: str,
    radius_m: int,
) -> pd.DataFrame:
    """
    Agrega conteos de POI por grupo temático dentro del anillo H3 que aproxima
    un buffer de radius_m alrededor del ancla indicada.

    Construye tres variables derivadas por usuario:
      - cnt_total   : suma de POIs en el anillo
      - entropy     : diversidad temática de esos POIs
      - density_km2 : cnt_total dividido por el área circular teórica

    La densidad es aproximada: el anillo H3 cubre más superficie que el círculo
    nominal (ver meters_to_k), de modo que el valor sobreestima la densidad real.
    Es comparable entre usuarios y entre radios, no interpretable en términos
    absolutos.

    Los anillos se calculan una vez por celda distinta y se reutilizan: varios
    usuarios pueden compartir ancla.
    """
    k = buffers_k[int(radius_m)]
    area = float(area_km2[int(radius_m)])

    users = df_anchor.loc[anchor_ok_mask, USER_COL].astype("string[python]").to_numpy()
    anchors = df_anchor.loc[anchor_ok_mask, anchor_col].astype("string[python]").to_numpy()

    uniq_cells, inv = np.unique(anchors, return_inverse=True)

    ring_map = {}
    for cell in uniq_cells:
        try:
            ring_map[cell] = list(H3_DISK(cell, int(k)))
        except Exception:
            ring_map[cell] = [cell]

    rows_u = []
    rows_h = []
    for i in range(len(users)):
        cell = anchors[i]
        ring = ring_map.get(cell, [cell])
        rows_u.extend([users[i]] * len(ring))
        rows_h.extend(ring)

    long = pd.DataFrame({USER_COL: rows_u, "h3_id_poi": pd.array(rows_h, dtype="string[python]")})
    joined = long.merge(counts_h3, left_on="h3_id_poi", right_index=True, how="left")

    # Las celdas del anillo sin POI no aparecen en counts_h3: su ausencia es un
    # cero, no un dato faltante.
    for g in POI_GROUPS:
        joined[g] = pd.to_numeric(joined[g], errors="coerce").fillna(0).astype(np.int32)

    agg = joined.groupby(USER_COL, observed=True, sort=False)[POI_GROUPS].sum().astype(np.int32)

    cnt_mat = agg.to_numpy(dtype=np.int32, copy=False)
    total = cnt_mat.sum(axis=1).astype(np.int32)
    ent = shannon_entropy_from_counts_mat(cnt_mat)
    dens = (total.astype(np.float32) / max(area, 1e-6)).astype(np.float32)

    pref = f"poi_{anchor_name}_r{int(radius_m)}m"
    out = agg.reset_index()
    out = out.rename(columns={g: f"{pref}_cnt_{g}" for g in POI_GROUPS})
    out[f"{pref}_cnt_total"] = total
    out[f"{pref}_entropy"] = ent
    out[f"{pref}_density_km2"] = dens

    log_poi_jsonl(
        {
            "event": "poi_anchor_buffer",
            "ts_utc": now_utc(),
            "poi_run_id": poi_cfg.poi_run_id,
            "feature_set": FEATURE_SET,
            "anchor": anchor_name,
            "radius_m": int(radius_m),
            "k_rings": int(k),
            "n_users_anchor": int(len(np.unique(users))),
            "n_unique_cells": int(len(uniq_cells)),
            "n_long_rows": int(len(long)),
            "users_with_any_poi": int(out.shape[0]),
        },
    )

    del long, joined, agg, cnt_mat, total, ent, dens
    gc.collect()

    return out

In [ ]:
# ----------------------------
# 11.2.5 — Variables HOME/WORK por radio + contrastes
# ----------------------------
df_poi_feat = users_target.astype("string[python]").to_frame(name=USER_COL).set_index(USER_COL)

for r in buffers_m:
    # HOME
    if bool(home_ok.any()):
        home_block = build_anchor_block(
            df_anchor=df_anchor,
            anchor_col="home_h3",
            anchor_ok_mask=home_ok.to_numpy(dtype=bool),
            anchor_name="home",
            radius_m=int(r),
        ).set_index(USER_COL)
        df_poi_feat = df_poi_feat.join(home_block, how="left")
        del home_block
        gc.collect()

    # WORK
    if bool(work_ok.any()):
        work_block = build_anchor_block(
            df_anchor=df_anchor,
            anchor_col="work_h3",
            anchor_ok_mask=work_ok.to_numpy(dtype=bool),
            anchor_name="work",
            radius_m=int(r),
        ).set_index(USER_COL)
        df_poi_feat = df_poi_feat.join(work_block, how="left")
        del work_block
        gc.collect()

    # ----------------------------
    # Contrastes trabajo vs. hogar
    # ----------------------------
    # Se usa la diferencia de logaritmos, log1p(work) - log1p(home), y no la
    # razón: es simétrica en torno a 0, está definida cuando alguno de los dos
    # es cero, y comprime el efecto de las diferencias en volumen absoluto.
    # Un valor positivo indica un entorno laboral más denso que el residencial.
    #
    # ADVERTENCIA: los usuarios sin ancla laboral reciben 0 en las variables de
    # trabajo, de modo que su contraste queda determinado por -log1p(home) y no
    # por una comparación real. Al interpretar por segmento, contrastar contra
    # la proporción de anclas laborales resueltas de cada grupo.
    pref_h = f"poi_home_r{int(r)}m"
    pref_w = f"poi_work_r{int(r)}m"
    pref_c = f"poi_contrast_r{int(r)}m"

    new_cols: Dict[str, np.ndarray] = {}

    ch_tot = f"{pref_h}_cnt_total"
    cw_tot = f"{pref_w}_cnt_total"
    if (ch_tot in df_poi_feat.columns) and (cw_tot in df_poi_feat.columns):
        h = df_poi_feat[ch_tot].fillna(0).to_numpy(dtype=np.float32)
        w = df_poi_feat[cw_tot].fillna(0).to_numpy(dtype=np.float32)
        new_cols[f"{pref_c}_cnt_total_logdiff"] = (np.log1p(w) - np.log1p(h)).astype(np.float32)

    ch_ent = f"{pref_h}_entropy"
    cw_ent = f"{pref_w}_entropy"
    if (ch_ent in df_poi_feat.columns) and (cw_ent in df_poi_feat.columns):
        h = df_poi_feat[ch_ent].fillna(0).to_numpy(dtype=np.float32)
        w = df_poi_feat[cw_ent].fillna(0).to_numpy(dtype=np.float32)
        new_cols[f"{pref_c}_entropy_diff"] = (w - h).astype(np.float32)

    for g in POI_GROUPS:
        ch = f"{pref_h}_cnt_{g}"
        cw = f"{pref_w}_cnt_{g}"
        if (ch in df_poi_feat.columns) and (cw in df_poi_feat.columns):
            h = df_poi_feat[ch].fillna(0).to_numpy(dtype=np.float32)
            w = df_poi_feat[cw].fillna(0).to_numpy(dtype=np.float32)
            new_cols[f"{pref_c}_cnt_{g}_logdiff"] = (np.log1p(w) - np.log1p(h)).astype(np.float32)

    # Concatenar en una sola operación evita la fragmentación del frame
    if new_cols:
        df_new = pd.DataFrame(new_cols, index=df_poi_feat.index)
        df_poi_feat = pd.concat([df_poi_feat, df_new], axis=1)
        del df_new, new_cols
        gc.collect()

# NaN -> 0. Un faltante aquí indica ancla no resuelta o ausencia de POIs en el
# anillo; ambos casos corresponden a cero POIs observados.
for c in df_poi_feat.columns:
    df_poi_feat[c] = pd.to_numeric(df_poi_feat[c], errors="coerce").fillna(0)

# Tipos compactos: conteos como entero, el resto como float
for c in df_poi_feat.columns:
    if "_cnt_" in c and (c.endswith("_logdiff") is False):
        df_poi_feat[c] = df_poi_feat[c].astype(np.int32)
    else:
        df_poi_feat[c] = df_poi_feat[c].astype(np.float32)

df_poi_feat = df_poi_feat.copy()  # defragmentar
df_poi_feat = df_poi_feat.reset_index()

print(f"[11.2] df_poi_feat: {df_poi_feat.shape[0]:,} usuarios × {df_poi_feat.shape[1]:,} columnas")

In [ ]:
# ----------------------------
# 11.2.6 — Persistencia
# ----------------------------
# Artefacto de Nivel 2. Aunque los POIs derivan de OpenStreetMap (fuente
# pública), su asociación a anclas inferidas convierte el resultado en un
# descriptor de contexto por usuario: incrementa la especificidad del perfil
# individual. Se escribe en el directorio de artefactos y no forma parte de la
# entrega pública.
out_path = poi_dirs["poi_tables"] / f"poi_features_{FEATURE_SET}.parquet"
df_poi_feat.to_parquet(out_path, index=False)

log_poi_jsonl(
    {
        "event": "poi_features_saved",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "feature_set": FEATURE_SET,
        "unit": poi_cfg.unit,
        "buffers_m": list(map(int, buffers_m)),
        "buffers_k": {int(k_): int(v) for k_, v in buffers_k.items()},
        "users": int(df_poi_feat[USER_COL].nunique()),
        "n_cols": int(df_poi_feat.shape[1]),
        "mem_mb_poi_feat": float(df_poi_feat.memory_usage(deep=True).sum() / (1024**2)),
    },
)

print("[11.2 DONE] guardado:", out_path.name)
_ = gc.collect()

### **11.3 Integración Augmented (POIs como audit features)**

In [ ]:
# ============================================================
# 11.3 — Integración del brazo POI
# Objetivo:
#   - Integrar poi_features_{FEATURE_SET}.parquet a los productos por usuario
#   - Modo por defecto: POIs como variables de auditoría (audit_only)
#   - Modo alternativo (train_ablation): prepara training_cols extendido,
#     sin reejecutar el clustering en esta sección
# Salidas:
#   - poi_tables/features_augmented_{FEATURE_SET}.parquet
#   - poi_tables/final_labels_with_pois_{FEATURE_SET}.parquet
# ============================================================
USER_COL = cfg.col_user
POI_LOG = paths["logs_dir"] / "pois.jsonl"

# ----------------------------
# 11.3.0 — Guardrails + carga de insumos
# ----------------------------
try:
    _ = poi_cfg
    _ = poi_dirs
except NameError as e:
    raise NameError("Se requiere ejecutar la Sección 11.0 antes (poi_cfg, poi_dirs).") from e

FEATURE_SET = globals().get("FEATURE_SET", "UNKNOWN")

if "df_features" not in globals():
    feat_path = paths["tables_dir"] / f"features_with_clusters_{FEATURE_SET}.parquet"
    df_features = pd.read_parquet(feat_path)

poi_feat_path = poi_dirs["poi_tables"] / f"poi_features_{FEATURE_SET}.parquet"
if not poi_feat_path.exists():
    raise FileNotFoundError(f"No existe poi_features para FEATURE_SET={FEATURE_SET}: {poi_feat_path}")

df_poi = pd.read_parquet(poi_feat_path)

# df_final de la Sección 10 (cinco segmentos), si está disponible
df_final_loaded = None
if "df_final" in globals():
    df_final_loaded = df_final
else:
    final_path = paths["tables_dir"] / f"final_user_labels_{FEATURE_SET}.parquet"
    if final_path.exists():
        df_final_loaded = pd.read_parquet(final_path)

# Guardrails de unicidad
for name, df_ in [("df_features", df_features), ("df_poi", df_poi)]:
    if USER_COL not in df_.columns:
        raise ValueError(f"{name} no tiene '{USER_COL}'.")
    if int(df_.duplicated(subset=[USER_COL]).sum()) != 0:
        raise ValueError(f"{name} tiene usuarios duplicados en '{USER_COL}'.")

print("[11.3] df_features:", df_features.shape, "| df_poi:", df_poi.shape)
if df_final_loaded is not None:
    if int(df_final_loaded.duplicated(subset=[USER_COL]).sum()) != 0:
        raise ValueError("df_final tiene usuarios duplicados.")
    print("[11.3] df_final:", df_final_loaded.shape)

In [ ]:
# ----------------------------
# 11.3.1 — Modo de integración
# ----------------------------
# audit_only     : las variables POI se calculan, persisten y quedan
#                  disponibles para caracterizar los segmentos, pero NO
#                  participan del clustering. Es el modo de la corrida
#                  documentada.
# train_ablation : prepara un training_cols extendido con las variables POI.
#                  La rama está implementada pero no se ejecuta: el manuscrito
#                  declara esa ablación como no habilitada, por la desproporción
#                  entre el número de variables POI y el tamaño de la cohorte.
#                  Esta sección NO reejecuta el clustering en ningún caso.
POI_INTEGRATION_MODE = "audit_only"      # "audit_only" | "train_ablation"
POI_TRAINING_MODE = "append_to_training"  # si train_ablation: "append_to_training" | "pois_only"

print("[11.3] POI_INTEGRATION_MODE:", POI_INTEGRATION_MODE)

In [ ]:
# ----------------------------
# 11.3.2 — Merge POIs + casts
# ----------------------------
t0 = now_utc()

poi_cols = [c for c in df_poi.columns if c != USER_COL]

df_aug = df_features.merge(df_poi, on=USER_COL, how="left")

for c in poi_cols:
    df_aug[c] = pd.to_numeric(df_aug[c], errors="coerce").fillna(0)

for c in poi_cols:
    if "_cnt_" in c and (c.endswith("_logdiff") is False):
        df_aug[c] = df_aug[c].astype(np.int32)
    else:
        df_aug[c] = df_aug[c].astype(np.float32)

# Indicador de cobertura: se evalúa sobre los totales por buffer, no sobre la
# suma de todas las columnas, porque los contrastes logarítmicos pueden ser
# negativos y falsear la suma.
total_cols = [c for c in poi_cols if c.endswith("_cnt_total")]
if total_cols:
    has_any = (df_aug[total_cols].sum(axis=1).to_numpy(dtype=np.float32, copy=False) > 0)
else:
    has_any = (df_aug[poi_cols].sum(axis=1).to_numpy(dtype=np.float32, copy=False) > 0)

df_aug = df_aug.copy()  # defragmentar antes de insertar
df_aug["has_poi_features"] = has_any.astype(np.int8)

log_jsonl(
    POI_LOG,
    {
        "event": "poi_merged_into_features",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "feature_set": FEATURE_SET,
        "mode": POI_INTEGRATION_MODE,
        "rows_features": int(len(df_features)),
        "rows_poi": int(len(df_poi)),
        "rows_aug": int(len(df_aug)),
        "n_poi_cols": int(len(poi_cols)),
        "users_with_any_poi": int(df_aug["has_poi_features"].sum()),
        "elapsed_s": (now_utc() - t0).total_seconds(),
        "mem_mb_aug": float(df_aug.memory_usage(deep=True).sum() / (1024**2)),
    },
)

print(
    f"[11.3] df_aug: {df_aug.shape} | variables POI: {len(poi_cols):,} | "
    f"usuarios con algún POI: {int(df_aug['has_poi_features'].sum()):,}"
)

In [ ]:
# ----------------------------
# 11.3.3 — Registrar POIs como auditoría o entrenamiento según el modo
# ----------------------------
training_cols_current = list(globals().get("training_cols", [])) or None
poi_feature_cols = poi_cols
audit_poi_cols = poi_feature_cols
training_cols_new = training_cols_current

if POI_INTEGRATION_MODE == "train_ablation":
    if training_cols_current is None:
        raise NameError("Para train_ablation se requiere que 'training_cols' exista (Sección 8).")

    if POI_TRAINING_MODE == "append_to_training":
        training_cols_new = list(dict.fromkeys(training_cols_current + poi_feature_cols))
    elif POI_TRAINING_MODE == "pois_only":
        training_cols_new = list(dict.fromkeys(poi_feature_cols))
    else:
        raise ValueError("POI_TRAINING_MODE inválido. Usa 'append_to_training' o 'pois_only'.")

    log_jsonl(
        POI_LOG,
        {
            "event": "poi_training_cols_prepared",
            "ts_utc": now_utc(),
            "poi_run_id": poi_cfg.poi_run_id,
            "feature_set": FEATURE_SET,
            "poi_training_mode": POI_TRAINING_MODE,
            "n_training_before": int(len(training_cols_current)),
            "n_training_after": int(len(training_cols_new)),
        },
    )
    print(
        f"[11.3] training_cols extendido preparado: "
        f"{len(training_cols_current)} → {len(training_cols_new)} variables. "
        "El clustering NO se reejecuta en esta sección."
    )
else:
    log_jsonl(
        POI_LOG,
        {
            "event": "poi_audit_only",
            "ts_utc": now_utc(),
            "poi_run_id": poi_cfg.poi_run_id,
            "feature_set": FEATURE_SET,
            "n_audit_poi_cols": int(len(audit_poi_cols)),
        },
    )
    print(f"[11.3] {len(audit_poi_cols):,} variables POI registradas como auditoría.")

In [ ]:
# ----------------------------
# 11.3.4 — Persistencia
# ----------------------------
# Artefacto de Nivel 2: vector conductual y contexto territorial por usuario.
out_path = poi_dirs["poi_tables"] / f"features_augmented_{FEATURE_SET}.parquet"
df_aug.to_parquet(out_path, index=False)

log_jsonl(
    POI_LOG,
    {
        "event": "features_augmented_saved",
        "ts_utc": now_utc(),
        "poi_run_id": poi_cfg.poi_run_id,
        "feature_set": FEATURE_SET,
        "path": str(out_path),
        "n_cols": int(df_aug.shape[1]),
    },
)

print("[11.3] guardado:", out_path.name)

if df_final_loaded is not None:
    df_final_poi = df_final_loaded.merge(df_poi, on=USER_COL, how="left")

    for c in poi_cols:
        df_final_poi[c] = pd.to_numeric(df_final_poi[c], errors="coerce").fillna(0)

    for c in poi_cols:
        if "_cnt_" in c and (c.endswith("_logdiff") is False):
            df_final_poi[c] = df_final_poi[c].astype(np.int32)
        else:
            df_final_poi[c] = df_final_poi[c].astype(np.float32)

    df_final_poi = df_final_poi.copy()  # defragmentar

    total_cols2 = [c for c in poi_cols if c.endswith("_cnt_total")]
    if total_cols2:
        df_final_poi["has_poi_features"] = (df_final_poi[total_cols2].sum(axis=1) > 0).astype(np.int8)
    else:
        df_final_poi["has_poi_features"] = (df_final_poi[poi_cols].sum(axis=1) > 0).astype(np.int8)

    # Artefacto de Nivel 2 con la restricción más fuerte de esta sección:
    # combina identificador, anclas inferidas, condición ocupacional derivada y
    # contexto territorial. Se escribe en el directorio de artefactos y no forma
    # parte de la entrega pública; cualquier difusión debe ser agregada.
    final_poi_path = poi_dirs["poi_tables"] / f"final_labels_with_pois_{FEATURE_SET}.parquet"
    df_final_poi.to_parquet(final_poi_path, index=False)

    log_jsonl(
        POI_LOG,
        {
            "event": "final_labels_with_pois_saved",
            "ts_utc": now_utc(),
            "poi_run_id": poi_cfg.poi_run_id,
            "feature_set": FEATURE_SET,
            "path": str(final_poi_path),
            "n_rows": int(len(df_final_poi)),
            "users_with_any_poi": int(df_final_poi["has_poi_features"].sum()),
        },
    )

    print("[11.3] guardado:", final_poi_path.name)

# Exponer para las Secciones 12–13
df_features_augmented = df_aug

del df_poi
_ = gc.collect()

print("[11.3 DONE] Integración POI completa (modo:", POI_INTEGRATION_MODE + ").")

## *12. Validación interna estructural (consistencia y separación)*

Esta sección caracteriza los segmentos obtenidos y evalúa su **separabilidad
interna**: en qué medida los grupos difieren entre sí respecto de las variables
observadas.

Es validación *interna* en sentido estricto. Mide consistencia y contraste
dentro del propio conjunto de datos, no correspondencia con una realidad
externa. Un segmento puede estar perfectamente separado en el espacio de
variables y no corresponder a ninguna condición laboral real. El manuscrito
declara la validación externa como no habilitada, por incompatibilidad de
universos, escalas y definiciones entre las fuentes disponibles.

El análisis se organiza en cuatro pasos: control de integridad, firmas de
segmento por cuantiles, separabilidad por pares mediante KS y divergencia de
Jensen-Shannon, y un resumen de las variables más discriminantes.

**Sobre las dos variantes.** `baseline` usa los segmentos consolidados de la
Sección 10; `augmented` usa los mismos segmentos enriquecidos con variables POI.
Los segmentos son idénticos entre ambas: el brazo POI operó en modo `audit_only`
y no participó del agrupamiento. La comparación sirve para caracterizar los
segmentos con información territorial, no para contrastar dos segmentaciones
distintas.

**Sobre los segmentos asignados por regla.** `MOBILE_FOR_WORK` y `NO_WORKER` no
pasaron por el clustering y, en consecuencia, no arrastran el vector conductual
completo: sus variables de entrenamiento aparecen como faltantes en las tablas
de firmas. Además, por su tamaño reducido quedan excluidos del cálculo de
separabilidad, que exige un mínimo de casos por grupo.

In [ ]:
# ============================================================
# 12.0 — Setup de evaluación interna
#   - Carga baseline: final_user_labels_{FEATURE_SET}.parquet (Sección 10)
#   - Carga augmented: final_labels_with_pois_{FEATURE_SET}.parquet (Sección 11.3)
#   - Define TRAINING_COLS según FEATURE_SET
#   - Helper attach_final_segment() para evitar colisiones de columna
# ============================================================
USER_COL = cfg.col_user
EVAL_LOG = paths["logs_dir"] / "eval_internal.jsonl"
FEATURE_SET = globals().get("FEATURE_SET", "UNKNOWN")

# --- Variables de entrenamiento según FEATURE_SET ---
BASE5 = ["home_work_ratio", "radius_gyration_km", "entropy_spatial", "n_moves", "unique_locations"]
TELEWORK_PAPER = ["DHD", "DPD", "WDTT", "WETT", "HO", "PO", "TO", "SO"]


def get_training_cols(feature_set: str) -> list[str]:
    """
    Devuelve las variables de entrenamiento efectivas.

    Prefiere la lista construida en la Sección 8 si está en memoria; en su
    ausencia la reconstruye desde FEATURE_SET, de modo que esta sección pueda
    ejecutarse sobre artefactos persistidos sin reejecutar el cuaderno completo.
    """
    if "training_cols" in globals():
        return list(training_cols)
    if feature_set == "BASE5":
        return BASE5
    if feature_set == "TELEWORK_PAPER":
        return TELEWORK_PAPER
    if feature_set == "COMBINED":
        return BASE5 + TELEWORK_PAPER
    return BASE5


TRAINING_COLS = get_training_cols(FEATURE_SET)

# --- baseline: producto consolidado de la Sección 10 ---
baseline_path = paths["tables_dir"] / f"final_user_labels_{FEATURE_SET}.parquet"
if not baseline_path.exists():
    raise FileNotFoundError(f"No existe {baseline_path.name}. Ejecute la Sección 10 primero.")

df_baseline_full = pd.read_parquet(baseline_path)

if USER_COL not in df_baseline_full.columns or "final_segment" not in df_baseline_full.columns:
    raise ValueError("El producto baseline debe contener user + final_segment (Sección 10).")
assert int(df_baseline_full.duplicated(subset=[USER_COL]).sum()) == 0, "baseline tiene usuarios duplicados."

# Etiquetas de referencia: se toman del baseline y se imponen sobre cualquier
# variante, para que la comparación opere siempre sobre la misma partición.
df_final_labels = df_baseline_full[[USER_COL, "final_segment"]].copy()

# --- augmented: producto enriquecido de la Sección 11.3 (opcional) ---
aug_path = None
if "poi_dirs" in globals():
    p = poi_dirs["poi_tables"] / f"final_labels_with_pois_{FEATURE_SET}.parquet"
    if p.exists():
        aug_path = p
if aug_path is None:
    p2 = paths["tables_dir"] / f"final_labels_with_pois_{FEATURE_SET}.parquet"
    if p2.exists():
        aug_path = p2

df_augmented_full = None
if aug_path is not None:
    df_augmented_full = pd.read_parquet(aug_path)
    if USER_COL not in df_augmented_full.columns or "final_segment" not in df_augmented_full.columns:
        raise ValueError("El producto augmented debe contener user + final_segment (Sección 11.3).")
    assert int(df_augmented_full.duplicated(subset=[USER_COL]).sum()) == 0, "augmented tiene usuarios duplicados."

variants = ["baseline"]
if df_augmented_full is not None:
    variants.append("augmented")

print("[12.0] FEATURE_SET:", FEATURE_SET)
print("[12.0] variantes disponibles:", variants)
print(f"[12.0] TRAINING_COLS ({len(TRAINING_COLS)}):", TRAINING_COLS)
print("[12.0] baseline:", df_baseline_full.shape)
if df_augmented_full is not None:
    print("[12.0] augmented:", df_augmented_full.shape)

log_jsonl(
    EVAL_LOG,
    {
        "event": "section12_setup",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "variants": variants,
        "training_cols": TRAINING_COLS,
        "baseline_available": True,
        "augmented_available": bool(df_augmented_full is not None),
    },
)


def load_eval_df(variant: str) -> pd.DataFrame:
    """Devuelve el dataframe de la variante indicada, sin copia profunda."""
    if variant == "baseline":
        return df_baseline_full.copy(deep=False)
    if variant == "augmented":
        if df_augmented_full is None:
            raise FileNotFoundError("La variante augmented no está disponible.")
        return df_augmented_full.copy(deep=False)
    raise ValueError("variant inválido.")


def attach_final_segment(df: pd.DataFrame) -> pd.DataFrame:
    """
    Impone la columna final_segment del baseline sobre cualquier dataframe.

    Se elimina antes de unir para evitar el sufijado automático de pandas
    (final_segment_x / final_segment_y), que rompería silenciosamente los
    agrupamientos posteriores. Garantiza además que ambas variantes usen
    exactamente el mismo catálogo de categorías.
    """
    df0 = df.drop(columns=["final_segment"], errors="ignore")
    out = df0.merge(df_final_labels, on=USER_COL, how="left")
    return out


# --- Variables de auditoría ---
# No participaron del clustering: su comportamiento por segmento es evidencia
# independiente sobre la naturaleza de la partición.
AUDIT_CORE = [
    "home_lab_ratio", "work_lab_ratio", "other_lab_ratio",
    "workday_presence_rate", "home_workday_presence_rate",
    "mean_move_distance_km", "p90_move_distance_km", "mean_move_duration_min",
    "total_lab_wmin", "avg_lab_wmin_per_day", "std_lab_wmin_per_day",
]


def get_poi_core_cols(df: pd.DataFrame) -> list[str]:
    """
    Selecciona un subconjunto representativo de variables POI: totales,
    entropía, densidad y contrastes, para el radio mayor configurado.

    Se acota deliberadamente: el catálogo POI completo supera el centenar de
    columnas y no aporta a la caracterización de segmentos.
    """
    rmax = 800
    if "poi_cfg" in globals() and getattr(poi_cfg, "buffers_m", None):
        rmax = int(max(poi_cfg.buffers_m))

    cand = [
        f"poi_home_r{rmax}m_cnt_total",
        f"poi_home_r{rmax}m_entropy",
        f"poi_home_r{rmax}m_density_km2",
        f"poi_work_r{rmax}m_cnt_total",
        f"poi_work_r{rmax}m_entropy",
        f"poi_work_r{rmax}m_density_km2",
        f"poi_contrast_r{rmax}m_cnt_total_logdiff",
        f"poi_contrast_r{rmax}m_entropy_diff",
    ]
    return [c for c in cand if c in df.columns]

In [ ]:
# ============================================================
# 12.1 — QA de integridad (por variante)
# Salida: eval_internal_integrity_{FEATURE_SET}.parquet
# ============================================================
def qa_integrity(df: pd.DataFrame, variant: str) -> pd.DataFrame:
    """
    Verifica integridad estructural y reporta cobertura de las variables núcleo.

    Las tasas de faltantes son informativas, no diagnósticas: los segmentos
    asignados por regla (MOBILE_FOR_WORK, NO_WORKER) no arrastran el vector
    conductual, de modo que una tasa distinta de cero en las variables de
    entrenamiento es esperable y proporcional al peso de esos segmentos.

    Devuelve una tabla en formato largo con tres bloques: resumen de integridad,
    conteos por segmento y tasas de faltantes por columna.
    """
    t0 = now_utc()
    dfm = attach_final_segment(df)

    dup = int(dfm.duplicated(subset=[USER_COL]).sum())
    if dup != 0:
        raise ValueError(f"[{variant}] usuarios duplicados tras attach_final_segment: {dup}")

    n_total = int(len(dfm))
    n_labeled = int(dfm["final_segment"].notna().sum())
    miss_label = int(n_total - n_labeled)

    seg_counts = (
        dfm["final_segment"]
        .value_counts(dropna=False)
        .rename_axis("final_segment")
        .reset_index(name="n_users")
    )
    seg_counts["variant"] = variant
    seg_counts["feature_set"] = FEATURE_SET
    seg_counts["table"] = "segment_counts"

    core_cols = [c for c in (TRAINING_COLS + AUDIT_CORE + get_poi_core_cols(dfm)) if c in dfm.columns]
    miss_rows = []
    for c in core_cols:
        miss_rows.append(
            {
                "variant": variant,
                "feature_set": FEATURE_SET,
                "table": "missing_rates",
                "col": c,
                "value": float(dfm[c].isna().mean()),
                "key": pd.NA,
            }
        )
    miss_df = pd.DataFrame(miss_rows)

    summary = pd.DataFrame(
        [
            {"variant": variant, "feature_set": FEATURE_SET, "table": "integrity_summary", "key": "n_rows", "value": n_total, "col": pd.NA},
            {"variant": variant, "feature_set": FEATURE_SET, "table": "integrity_summary", "key": "n_labeled_final_segment", "value": n_labeled, "col": pd.NA},
            {"variant": variant, "feature_set": FEATURE_SET, "table": "integrity_summary", "key": "n_missing_final_segment", "value": miss_label, "col": pd.NA},
            {"variant": variant, "feature_set": FEATURE_SET, "table": "integrity_summary", "key": "n_core_cols", "value": int(len(core_cols)), "col": pd.NA},
        ]
    )

    seg_counts_out = seg_counts.rename(columns={"final_segment": "key", "n_users": "value"}).assign(col=pd.NA)
    out = pd.concat([summary, seg_counts_out, miss_df], ignore_index=True)

    log_jsonl(
        EVAL_LOG,
        {
            "event": "eval_integrity_done",
            "ts_utc": now_utc(),
            "variant": variant,
            "feature_set": FEATURE_SET,
            "n_rows": n_total,
            "n_missing_final_segment": miss_label,
            "n_core_cols": int(len(core_cols)),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    return out


integrity_tables = []
for v in variants:
    dfv = load_eval_df(v)
    integrity_tables.append(qa_integrity(dfv, v))
    del dfv
    gc.collect()

df_integrity = pd.concat(integrity_tables, ignore_index=True)

# Artefacto de Nivel 2: agregados por segmento derivados de la fuente telco.
int_path = paths["tables_dir"] / f"eval_internal_integrity_{FEATURE_SET}.parquet"
df_integrity.to_parquet(int_path, index=False)

print("[12.1] guardado:", int_path.name)
print("[12.1] resumen de integridad:")
print(df_integrity[df_integrity["table"].eq("integrity_summary")].to_string(index=False))

del integrity_tables
_ = gc.collect()

In [ ]:
# ============================================================
# 12.2 — Firmas por segmento (cuantiles + IQR) + boxplots
# Salida: segment_signatures_{variant}_{FEATURE_SET}.parquet
# ============================================================
def pick_plot_features(df: pd.DataFrame, max_n: int = 5) -> list[str]:
    """Selecciona un subconjunto acotado de variables para las figuras."""
    prefs = [
        "home_work_ratio", "DHD", "DPD",
        "radius_gyration_km", "entropy_spatial", "n_moves", "unique_locations",
    ]
    out = [c for c in prefs if c in df.columns]
    out += [c for c in get_poi_core_cols(df) if c not in out]
    return out[:max_n]


def segment_signatures(df: pd.DataFrame, variant: str, features: list[str]) -> pd.DataFrame:
    """
    Construye la firma de cada segmento: cuantiles 10, 25, 50, 75 y 90 de cada
    variable, más el rango intercuartílico como medida de dispersión.

    Se emplean cuantiles y no momentos porque los segmentos son de tamaño muy
    desigual y varias variables tienen distribuciones fuertemente asimétricas,
    condiciones bajo las cuales media y desviación estándar son poco
    informativas.

    Los segmentos asignados por regla producen cuantiles faltantes en las
    variables de entrenamiento: no arrastran ese vector desde la Sección 10.
    """
    t0 = now_utc()
    dfm = attach_final_segment(df)
    dfm = dfm[dfm["final_segment"].notna()].copy()

    feats = [c for c in features if c in dfm.columns]
    if not feats:
        raise ValueError(f"[{variant}] no hay variables disponibles para construir firmas.")

    for c in feats:
        dfm[c] = pd.to_numeric(dfm[c], errors="coerce")

    qs = [0.10, 0.25, 0.50, 0.75, 0.90]
    q = (
        dfm.groupby("final_segment", observed=True, sort=False)[feats]
        .quantile(qs)
        .reset_index()
        .rename(columns={"level_1": "q"})
    )
    q["variant"] = variant
    q["feature_set"] = FEATURE_SET

    q25 = q[q["q"] == 0.25].set_index(["final_segment"])[feats]
    q75 = q[q["q"] == 0.75].set_index(["final_segment"])[feats]
    iqr = (q75 - q25).reset_index()

    iqr_long = iqr.melt(id_vars=["final_segment"], var_name="feature", value_name="iqr")
    iqr_long["iqr"] = pd.to_numeric(iqr_long["iqr"], errors="coerce").astype(np.float32)
    iqr_long["variant"] = variant
    iqr_long["feature_set"] = FEATURE_SET

    q_long = q.melt(id_vars=["final_segment", "q", "variant", "feature_set"], var_name="feature", value_name="value")
    q_long["value"] = pd.to_numeric(q_long["value"], errors="coerce").astype(np.float32)

    sig = q_long.merge(iqr_long, on=["variant", "feature_set", "final_segment", "feature"], how="left")

    log_jsonl(
        EVAL_LOG,
        {
            "event": "eval_signatures_done",
            "ts_utc": now_utc(),
            "variant": variant,
            "feature_set": FEATURE_SET,
            "n_segments": int(sig["final_segment"].nunique()),
            "n_features": int(len(feats)),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    return sig


for v in variants:
    dfv = load_eval_df(v)
    core_feats = [c for c in (TRAINING_COLS + AUDIT_CORE + get_poi_core_cols(dfv)) if c in dfv.columns]

    df_sig = segment_signatures(dfv, v, core_feats)
    sig_path = paths["tables_dir"] / f"segment_signatures_{v}_{FEATURE_SET}.parquet"
    df_sig.to_parquet(sig_path, index=False)
    print(f"[12.2] guardado: {sig_path.name} | {len(core_feats)} variables")

    # Figuras: distribución por segmento de un subconjunto de variables
    dfm = attach_final_segment(dfv)
    dfm = dfm[dfm["final_segment"].notna()].copy()

    # Orden estable de segmentos: se respeta el catálogo ordenado definido en
    # la Sección 10.3 si la columna sigue siendo categórica.
    if isinstance(dfm["final_segment"].dtype, CategoricalDtype):
        segs = [str(s) for s in dfm["final_segment"].cat.categories if (dfm["final_segment"] == s).any()]
    else:
        segs = sorted(dfm["final_segment"].astype(str).unique().tolist())

    plot_feats = pick_plot_features(dfm, max_n=5)

    for f in plot_feats:
        data = [
            pd.to_numeric(dfm.loc[dfm["final_segment"].astype(str) == s, f], errors="coerce")
            .dropna()
            .to_numpy()
            for s in segs
        ]
        fig = plt.figure(figsize=(8, 4))
        ax = plt.gca()
        ax.boxplot(data, tick_labels=segs, showfliers=False)
        ax.set_title(f"{f} by final_segment — {v} — {FEATURE_SET}")
        ax.set_xticklabels(segs, rotation=25, ha="right")
        fig.tight_layout()
        fig_path = paths["figures_dir"] / f"sig_box_{f}_{v}_{FEATURE_SET}.png"
        fig.savefig(fig_path, dpi=200)
        plt.close(fig)

    log_jsonl(
        EVAL_LOG,
        {
            "event": "eval_signatures_figs_saved",
            "ts_utc": now_utc(),
            "variant": v,
            "feature_set": FEATURE_SET,
            "plot_features": plot_feats,
        },
    )

    del dfv, df_sig, dfm
    gc.collect()

print("[12.2] OK")

In [ ]:
# ============================================================
# 12.3 — Separabilidad interna (KS + divergencia de Jensen-Shannon)
# Salida: pairwise_separation_{variant}_{FEATURE_SET}.parquet
# ============================================================

# Tamaño mínimo de grupo para calcular separabilidad. Por debajo de este umbral
# los estadísticos carecen de sentido: MOBILE_FOR_WORK, con 3 usuarios en la
# corrida documentada, queda excluido de todos sus pares.
MIN_GROUP_SIZE = 5

# Tope de variables analizadas. El número de pares crece con el producto de
# variables por combinaciones de segmentos, de modo que el tope acota el costo.
# ADVERTENCIA: la lista de entrada se ordena TRAINING_COLS + AUDIT_CORE +
# variables POI. Bajo COMBINED las dos primeras suman 24, por lo que el tope
# corta antes de alcanzar las POI y estas NO entran al análisis. La identidad
# entre las tablas baseline y augmented es consecuencia de esto, además de que
# ambas variantes comparten la misma partición.
MAX_FEATURES_SEPARATION = 20


def jsd_hist(a: np.ndarray, b: np.ndarray, bins: int = 30) -> float:
    """
    Divergencia de Jensen-Shannon entre dos muestras, estimada por histograma.

    Ambas se discretizan sobre un rango común para que las distribuciones sean
    comparables. Devuelve NaN si alguna muestra está vacía, y 0 si el rango es
    degenerado (todos los valores iguales), caso en que las distribuciones son
    idénticas por construcción.

    A diferencia del estadístico KS, la JSD está acotada y es simétrica, lo que
    la hace apta para promediar entre pares de segmentos.
    """
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if a.size == 0 or b.size == 0:
        return float("nan")

    lo = float(min(a.min(), b.min()))
    hi = float(max(a.max(), b.max()))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return 0.0

    ha, _ = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hb, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)

    pa = ha.astype(np.float64)
    pb = hb.astype(np.float64)
    pa = pa / max(pa.sum(), 1e-12)
    pb = pb / max(pb.sum(), 1e-12)

    return float(jensenshannon(pa, pb, base=np.e))


def pairwise_separation(df: pd.DataFrame, variant: str, features: list[str]) -> pd.DataFrame:
    """
    Calcula separabilidad entre cada par de segmentos para cada variable.

    Reporta dos estadísticos complementarios:
      - ks_stat / ks_pvalue : prueba de Kolmogorov-Smirnov de dos muestras.
        Detecta cualquier diferencia distribucional, pero su p-valor depende
        fuertemente del tamaño muestral.
      - jsd : divergencia de Jensen-Shannon. Acotada y simétrica, medida de
        magnitud del contraste independiente del tamaño.

    Los pares en que alguno de los grupos tiene menos de MIN_GROUP_SIZE casos
    se reportan con NaN en los tres estadísticos, sin excluirlos de la tabla:
    su ausencia de resultado es en sí información.
    """
    t0 = now_utc()
    dfm = attach_final_segment(df)
    dfm = dfm[dfm["final_segment"].notna()].copy()

    segs = sorted(dfm["final_segment"].astype(str).unique().tolist())

    feats_all = [c for c in features if c in dfm.columns]
    feats = feats_all[:MAX_FEATURES_SEPARATION]
    feats_dropped = feats_all[MAX_FEATURES_SEPARATION:]

    if feats_dropped:
        print(
            f"[12.3][{variant}] el tope de {MAX_FEATURES_SEPARATION} variables dejó fuera "
            f"{len(feats_dropped)}: {feats_dropped}"
        )

    rows = []
    seg_arr = dfm["final_segment"].astype(str).to_numpy()

    for f in feats:
        x = pd.to_numeric(dfm[f], errors="coerce").to_numpy(np.float64, copy=False)
        for a, b in combinations(segs, 2):
            xa = x[seg_arr == a]
            xb = x[seg_arr == b]
            xa = xa[np.isfinite(xa)]
            xb = xb[np.isfinite(xb)]

            if xa.size < MIN_GROUP_SIZE or xb.size < MIN_GROUP_SIZE:
                ks_stat, ks_p, jsd = np.nan, np.nan, np.nan
            else:
                try:
                    ks = ks_2samp(xa, xb, alternative="two-sided", mode="auto")
                    ks_stat, ks_p = float(ks.statistic), float(ks.pvalue)
                except Exception:
                    ks_stat, ks_p = np.nan, np.nan
                jsd = jsd_hist(xa, xb, bins=30)

            rows.append(
                {
                    "variant": variant,
                    "feature_set": FEATURE_SET,
                    "feature": f,
                    "seg_a": a,
                    "seg_b": b,
                    "n_a": int(xa.size),
                    "n_b": int(xb.size),
                    "ks_stat": ks_stat,
                    "ks_pvalue": ks_p,
                    "jsd": jsd,
                }
            )

    out = pd.DataFrame(rows)

    log_jsonl(
        EVAL_LOG,
        {
            "event": "eval_pairwise_separation_done",
            "ts_utc": now_utc(),
            "variant": variant,
            "feature_set": FEATURE_SET,
            "n_features_in": int(len(feats_all)),
            "n_features_used": int(len(feats)),
            "features_dropped_by_cap": feats_dropped,
            "max_features_cap": int(MAX_FEATURES_SEPARATION),
            "min_group_size": int(MIN_GROUP_SIZE),
            "n_pairs": int(len(list(combinations(segs, 2)))),
            "rows": int(len(out)),
            "elapsed_s": (now_utc() - t0).total_seconds(),
        },
    )

    return out


for v in variants:
    dfv = load_eval_df(v)
    core_feats = [c for c in (TRAINING_COLS + AUDIT_CORE + get_poi_core_cols(dfv)) if c in dfv.columns]

    df_sep = pairwise_separation(dfv, v, core_feats)
    sep_path = paths["tables_dir"] / f"pairwise_separation_{v}_{FEATURE_SET}.parquet"
    df_sep.to_parquet(sep_path, index=False)

    _n_nan = int(df_sep["jsd"].isna().sum())
    print(
        f"[12.3] guardado: {sep_path.name} | {len(df_sep):,} filas | "
        f"{_n_nan:,} sin resultado por tamaño de grupo"
    )

    del dfv, df_sep, _n_nan
    gc.collect()

print("[12.3] OK")

In [ ]:
# ============================================================
# 12.4 — Resumen interno: variables más discriminantes
# Salida: internal_summary_{variant}_{FEATURE_SET}.parquet
# ============================================================
OUT_COLS = [
    "variant", "feature_set", "table",
    "rank", "feature", "mean_jsd", "max_ks", "min_ks_p",
    "final_segment", "n_users",
]


def internal_summary(variant: str) -> pd.DataFrame:
    """
    Consolida el ranking de variables discriminantes y la prevalencia por
    segmento en una tabla en formato largo.

    El ranking ordena por JSD media entre pares —magnitud de contraste
    independiente del tamaño muestral— y desempata por el estadístico KS
    máximo. Los pares sin resultado por tamaño de grupo se omiten del promedio.

    La construcción declara explícitamente el esquema y los tipos de ambos
    bloques antes de concatenarlos: sin eso, pandas infiere tipos por bloque y
    las columnas que resultan enteramente nulas en uno de ellos degradan el tipo
    común de forma silenciosa.
    """
    sep_path = paths["tables_dir"] / f"pairwise_separation_{variant}_{FEATURE_SET}.parquet"
    if not sep_path.exists():
        raise FileNotFoundError(f"No existe {sep_path.name}. Ejecute la Sección 12.3 primero.")

    df_sep = pd.read_parquet(sep_path)

    agg = (
        df_sep.groupby(["variant", "feature_set", "feature"], observed=True, sort=False)
        .agg(
            mean_jsd=("jsd", "mean"),
            max_ks=("ks_stat", "max"),
            min_ks_p=("ks_pvalue", "min"),
        )
        .reset_index()
        .sort_values(["mean_jsd", "max_ks"], ascending=[False, False], kind="mergesort")
    )

    top = agg.head(10).copy()
    top["rank"] = np.arange(1, len(top) + 1, dtype=np.int32)
    top["table"] = "top_features"

    # Prevalencia por segmento
    dfv = load_eval_df(variant)
    dfm = attach_final_segment(dfv)
    prev = (
        dfm["final_segment"]
        .value_counts(dropna=False)
        .rename_axis("final_segment")
        .reset_index(name="n_users")
    )
    prev["variant"] = variant
    prev["feature_set"] = FEATURE_SET
    prev["table"] = "prevalence"

    # Armonización de esquema y tipos antes del concat.
    # Nota: los faltantes de las columnas float usan np.nan y no pd.NA, porque
    # float32 de numpy no admite el escalar nulo de pandas.
    top_out = top.copy()
    top_out["final_segment"] = pd.array([pd.NA] * len(top_out), dtype="string[python]")
    top_out["n_users"] = pd.array([pd.NA] * len(top_out), dtype="Int32")
    top_out["rank"] = pd.array(top_out["rank"], dtype="Int32")
    top_out["feature"] = top_out["feature"].astype("string[python]")
    top_out["mean_jsd"] = pd.to_numeric(top_out["mean_jsd"], errors="coerce").astype("float32")
    top_out["max_ks"] = pd.to_numeric(top_out["max_ks"], errors="coerce").astype("float32")
    top_out["min_ks_p"] = pd.to_numeric(top_out["min_ks_p"], errors="coerce").astype("float32")

    prev_out = prev.copy()
    prev_out["rank"] = pd.array([pd.NA] * len(prev_out), dtype="Int32")
    prev_out["feature"] = pd.array([pd.NA] * len(prev_out), dtype="string[python]")
    prev_out["mean_jsd"] = np.full(len(prev_out), np.nan, dtype=np.float32)
    prev_out["max_ks"] = np.full(len(prev_out), np.nan, dtype=np.float32)
    prev_out["min_ks_p"] = np.full(len(prev_out), np.nan, dtype=np.float32)
    prev_out["final_segment"] = prev_out["final_segment"].astype("string[python]")
    prev_out["n_users"] = pd.array(prev_out["n_users"], dtype="Int32")

    out = pd.concat([top_out[OUT_COLS], prev_out[OUT_COLS]], ignore_index=True)

    del df_sep, dfv, dfm, prev, top, agg, top_out, prev_out
    gc.collect()

    return out


for v in variants:
    df_sum = internal_summary(v)
    sum_path = paths["tables_dir"] / f"internal_summary_{v}_{FEATURE_SET}.parquet"
    df_sum.to_parquet(sum_path, index=False)
    print(f"[12.4] guardado: {sum_path.name}")

    log_jsonl(
        EVAL_LOG,
        {
            "event": "eval_internal_summary_saved",
            "ts_utc": now_utc(),
            "variant": v,
            "feature_set": FEATURE_SET,
            "path": str(sum_path),
        },
    )

    del df_sum
    gc.collect()

print("[12.4] OK")

In [ ]:
# ============================================================
# 12.5 — Snapshot interpretativo
#   - Prevalencia por segmento
#   - Medianas de variables clave por segmento
#   - Ranking de variables discriminantes
# ============================================================
def _load_internal_summary(variant: str) -> pd.DataFrame:
    p = paths["tables_dir"] / f"internal_summary_{variant}_{FEATURE_SET}.parquet"
    if not p.exists():
        raise FileNotFoundError(f"No existe {p.name}. Ejecute la Sección 12.4.")
    return pd.read_parquet(p)


def _load_signatures(variant: str) -> pd.DataFrame:
    p = paths["tables_dir"] / f"segment_signatures_{variant}_{FEATURE_SET}.parquet"
    if not p.exists():
        raise FileNotFoundError(f"No existe {p.name}. Ejecute la Sección 12.2.")
    return pd.read_parquet(p)


def pick_demo_features(df: pd.DataFrame) -> list[str]:
    """Selecciona hasta diez variables representativas para el snapshot."""
    base = [
        "home_work_ratio",
        "radius_gyration_km", "entropy_spatial", "n_moves", "unique_locations",
        "DHD", "DPD", "WDTT", "WETT", "HO", "PO", "TO", "SO",
    ]
    base = [c for c in base if c in df.columns]
    poi_core = get_poi_core_cols(df)
    poi_demo = [c for c in poi_core if ("cnt_total" in c or "density" in c or "logdiff" in c)][:3]
    out = base[:7] + [c for c in poi_demo if c not in base]
    return out[:10]


for v in variants:
    print("\n" + "=" * 72)
    print(f"[12.5] VARIANTE={v} | FEATURE_SET={FEATURE_SET}")

    # --- 1) Prevalencia ---
    dfv = load_eval_df(v)
    dfm = attach_final_segment(dfv)

    prev = (
        dfm["final_segment"]
        .astype("string[python]")
        .value_counts(dropna=False)
        .rename_axis("final_segment")
        .reset_index(name="n_users")
    )
    prev["pct"] = (prev["n_users"] / max(prev["n_users"].sum(), 1)) * 100.0

    print("\n[12.5] prevalencia por segmento:")
    print(prev.to_string(index=False))

    labels = prev["final_segment"].astype(str).to_numpy()
    x = np.arange(len(labels), dtype=np.int32)

    fig = plt.figure(figsize=(7, 3.5))
    ax = plt.gca()
    ax.bar(x, prev["pct"].to_numpy(dtype=np.float32, copy=False))
    ax.set_title(f"Prevalencia por segmento (%) — {v} — {FEATURE_SET}")
    ax.set_ylabel("% users")
    ax.set_xlabel("final_segment")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    fig.tight_layout()
    fig_path = paths["figures_dir"] / f"prevalence_{v}_{FEATURE_SET}.png"
    fig.savefig(fig_path, dpi=200)
    plt.close(fig)

    # --- 2) Medianas por segmento ---
    # Los segmentos asignados por regla presentan medianas faltantes en las
    # variables de entrenamiento: no arrastran ese vector desde la Sección 10.
    # El faltante indica ausencia de la columna para ese grupo, no ausencia de
    # actividad observada.
    sig = _load_signatures(v)
    sig50 = sig.loc[sig["q"] == 0.50].copy()

    demo_feats = pick_demo_features(dfm)
    sig50 = sig50[sig50["feature"].isin(demo_feats)].copy()

    sig50["final_segment"] = sig50["final_segment"].astype("string[python]")
    sig50["feature"] = sig50["feature"].astype("string[python]")

    med = sig50.pivot_table(
        index="final_segment",
        columns="feature",
        values="value",
        aggfunc="first",
        observed=False,
    )

    seg_order = prev["final_segment"].astype("string[python]").tolist()
    med = med.reindex(seg_order).reset_index()

    med_path = paths["tables_dir"] / f"snapshot_medians_{v}_{FEATURE_SET}.parquet"
    med.to_parquet(med_path, index=False)

    print("\n[12.5] medianas por segmento (variables clave):")
    print(med.to_string(index=False))
    print("[12.5] guardado:", med_path.name)

    # --- 3) Variables más discriminantes ---
    summ = _load_internal_summary(v)
    top = summ[summ["table"] == "top_features"].copy()
    top["mean_jsd"] = pd.to_numeric(top["mean_jsd"], errors="coerce")
    top["max_ks"] = pd.to_numeric(top["max_ks"], errors="coerce")
    top = top.sort_values(["mean_jsd", "max_ks"], ascending=[False, False], kind="mergesort")

    print("\n[12.5] variables más discriminantes (JSD media, KS máximo):")
    print(top[["rank", "feature", "mean_jsd", "max_ks", "min_ks_p"]].to_string(index=False))

    log_jsonl(
        EVAL_LOG,
        {
            "event": "eval_snapshot_12_5",
            "ts_utc": now_utc(),
            "variant": v,
            "feature_set": FEATURE_SET,
            "prevalence_fig": str(fig_path),
            "medians_path": str(med_path),
            "demo_features": demo_feats,
            "top_features": top["feature"].astype(str).head(10).tolist(),
        },
    )

    del dfv, dfm, prev, sig, sig50, med, summ, top, labels, x
    gc.collect()

print("\n[12.5] OK — snapshot listo.")

## *13. Robustez y sensibilidad (estabilidad + sensibilidad a hiperparámetros)*

Esta sección evalúa **cuánto dependen los resultados de decisiones que podrían
haber sido otras**. Es la contrapartida empírica del principio de trazabilidad:
cada umbral del pipeline es una decisión metodológica, y su efecto sobre el
producto debe ser medible y no supuesto.

Se examinan dos dimensiones.

**Estabilidad del agrupamiento.** Se reejecuta el clustering sobre submuestras
aleatorias de los candidatos y se compara la partición resultante con la de
referencia. Dos métricas complementarias: el índice de Rand ajustado, que mide
concordancia global corrigiendo por azar, y el índice de Jaccard tras
emparejamiento óptimo de conglomerados, que mide solapamiento grupo a grupo.

**Sensibilidad a hiperparámetros.** Se varía un parámetro a la vez en torno a la
configuración documentada y se mide el efecto sobre el producto: el umbral de
permanencia domiciliaria, la regla de trabajo móvil, los cuatro criterios del
gatekeeper, y la ponderación del anclaje residencial.

**Qué se compara y qué no.** Las comparaciones son de conjuntos de usuarios y de
particiones, no de métricas de desempeño: sin fuente externa de contraste no hay
desempeño que medir. La pregunta que responde esta sección es si el producto
sería sustancialmente distinto bajo otras decisiones razonables, no si es
correcto.

**Advertencia sobre la lectura del gatekeeper.** El barrido de sus umbrales es el
resultado más consecuente de esta sección: muestra que el tamaño del universo
elegible depende fuertemente de una decisión metodológica, mientras que la
insuficiencia longitudinal de la fuente es estructural y no depende de ella.

In [ ]:
# ============================================================
# 13.0 — Setup de robustez
#   - Define réplicas y tamaño de submuestra
#   - Carga los insumos de estabilidad y sensibilidad
#   - Define helpers de comparación de particiones (ARI + Jaccard emparejado)
# ============================================================
USER_COL = cfg.col_user
ROB_LOG = paths["logs_dir"] / "robustness.jsonl"
FEATURE_SET = globals().get("FEATURE_SET", "UNKNOWN")

# ---------
# Configuración de robustez (declarada en ExperimentConfig, Sección 0.2)
# ---------
ROB_SEEDS = list(range(int(cfg.rob_n_seeds)))
SUBSAMPLE_FRAC = float(cfg.rob_subsample_frac)
K_BASELINE = int(cfg.k_baseline)

# ---------
# Rejillas de sensibilidad
# ---------
# Cada rejilla incluye el valor de la configuración documentada más dos
# alternativas que lo flanquean. El diseño es de una variable a la vez: se
# aísla el efecto de cada decisión sin confundirlo con interacciones.
TAU_REMOTE_GRID = [0.50, float(cfg.tau_remote), 0.70]
MOBILE_PCT_GRID = [0.70, float(cfg.mobile_percentile_p) / 100.0, 0.80]
MOBILE_TAU_UNIQ_GRID = [4, 10, int(cfg.mobile_tau_unique)]
ALPHA_GRID = [0.50, float(cfg.home_alpha), 0.90]

# ---------
# Rutas de insumos
# ---------
workers_path = paths["tables_dir"] / f"features_with_clusters_{FEATURE_SET}.parquet"
if not workers_path.exists():
    raise FileNotFoundError(f"No existe {workers_path.name}. Ejecute las Secciones 9 y 10 primero.")

universe_path = paths["tables_dir"] / "universe_full.parquet"
if not universe_path.exists():
    raise FileNotFoundError("No existe universe_full.parquet (Sección 7).")

gk_stats_path = paths["tables_dir"] / "gatekeeper_user_stats.parquet"
gk_elig_path = paths["tables_dir"] / "gatekeeper_eligible_users.parquet"
if not gk_stats_path.exists() or not gk_elig_path.exists():
    raise FileNotFoundError("Faltan gatekeeper_user_stats.parquet o gatekeeper_eligible_users.parquet (Sección 3).")

stays_gk_path = paths["cache_dir"] / "stays_gk.parquet"
if not stays_gk_path.exists():
    raise FileNotFoundError("No existe stays_gk.parquet. Ejecute el gatekeeper con persist=True.")

homes_path = paths["tables_dir"] / "homes.parquet"
works_path = paths["tables_dir"] / "works.parquet"
mob_path = paths["tables_dir"] / "mobility_features.parquet"
flags_path = paths["tables_dir"] / "mobile_flags.parquet"
for p in [homes_path, works_path, mob_path, flags_path]:
    if not p.exists():
        raise FileNotFoundError(f"Falta {p.name}. Ejecute las Secciones 4 a 6.")

final_labels_path = paths["tables_dir"] / f"final_user_labels_{FEATURE_SET}.parquet"
if not final_labels_path.exists():
    raise FileNotFoundError(f"Falta {final_labels_path.name}. Ejecute la Sección 10.")

# Producto enriquecido con POIs (opcional)
aug_final_path = None
if "poi_dirs" in globals():
    p = poi_dirs["poi_tables"] / f"final_labels_with_pois_{FEATURE_SET}.parquet"
    if p.exists():
        aug_final_path = p
if aug_final_path is None:
    p = paths["tables_dir"] / f"final_labels_with_pois_{FEATURE_SET}.parquet"
    if p.exists():
        aug_final_path = p

print("[13.0] FEATURE_SET:", FEATURE_SET)
print(f"[13.0] réplicas: {len(ROB_SEEDS)} | submuestreo: {SUBSAMPLE_FRAC:.0%} | k={K_BASELINE}")
print("[13.0] rejilla tau_remote:", TAU_REMOTE_GRID)
print("[13.0] rejilla percentil Rg:", MOBILE_PCT_GRID, "| tau_unique:", MOBILE_TAU_UNIQ_GRID)
print("[13.0] rejilla alpha:", ALPHA_GRID)
print("[13.0] producto con POIs disponible:", bool(aug_final_path is not None))

log_jsonl(
    ROB_LOG,
    {
        "event": "section13_setup",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "n_seeds": len(ROB_SEEDS),
        "subsample_frac": float(SUBSAMPLE_FRAC),
        "k": int(K_BASELINE),
        "tau_remote_grid": TAU_REMOTE_GRID,
        "mobile_pct_grid": MOBILE_PCT_GRID,
        "mobile_tau_uniq_grid": MOBILE_TAU_UNIQ_GRID,
        "alpha_grid": ALPHA_GRID,
        "augmented_available": bool(aug_final_path is not None),
    },
)


# ---------
# Helpers de comparación de particiones
# ---------
def _contingency(a: np.ndarray, b: np.ndarray, k: int) -> np.ndarray:
    """
    Matriz de contingencia k×k entre dos particiones con etiquetas en 1..k.

    La celda (i, j) cuenta los elementos asignados al conglomerado i en la
    partición a y al j en la partición b.
    """
    C = np.zeros((k, k), dtype=np.int32)
    for i in range(1, k + 1):
        ai = (a == i)
        if not ai.any():
            continue
        for j in range(1, k + 1):
            C[i - 1, j - 1] = int(np.sum(ai & (b == j)))
    return C


def jaccard_matched(a: np.ndarray, b: np.ndarray, k: int) -> dict:
    """
    Índice de Jaccard entre particiones, tras emparejamiento óptimo de
    conglomerados.

    Las etiquetas de conglomerado carecen de significado propio: la misma
    partición puede numerarse de k! formas distintas. El emparejamiento húngaro
    resuelve la correspondencia maximizando la suma de intersecciones, de modo
    que la comparación no dependa de la numeración.

    Devuelve el Jaccard medio ponderado por el tamaño de los conglomerados de la
    partición de referencia, y el detalle por conglomerado.
    """
    C = _contingency(a, b, k)

    # linear_sum_assignment minimiza; se niega para maximizar intersecciones
    r, c = linear_sum_assignment(-C)

    sizes_a = np.array([(a == i).sum() for i in range(1, k + 1)], dtype=np.int32)
    sizes_b = np.array([(b == j).sum() for j in range(1, k + 1)], dtype=np.int32)

    per = []
    weights = []
    jvals = []
    for i0, j0 in zip(r, c):
        inter = int(C[i0, j0])
        union = int(sizes_a[i0] + sizes_b[j0] - inter)
        jac = float(inter / union) if union > 0 else float("nan")
        per.append({"cluster_a": int(i0 + 1), "cluster_b": int(j0 + 1), "jaccard": jac, "inter": inter, "union": union})
        w = float(sizes_a[i0])
        if np.isfinite(jac):
            weights.append(w)
            jvals.append(jac)

    mean_w = float(np.average(jvals, weights=weights)) if weights else float("nan")
    return {"mean_jaccard_weighted": mean_w, "per_cluster": per}


_ = gc.collect()

In [ ]:
# ============================================================
# 13.1 — Estabilidad del agrupamiento (submuestreo)
#   - Reejecuta Ward sobre submuestras y compara con la partición de referencia
#   - ARI: concordancia global corregida por azar
#   - Jaccard emparejado: solapamiento grupo a grupo
# Salidas:
#   - clustering_stability_{FEATURE_SET}.parquet
#   - figuras de distribución de ambas métricas
# ============================================================
dfW = pd.read_parquet(workers_path)

req = {USER_COL, "cluster_id"}
miss = req - set(dfW.columns)
if miss:
    raise ValueError(f"El parquet de candidatos no tiene las columnas requeridas: {miss}")

# Variables de entrenamiento: se prefieren las de la Sección 12 si están en
# memoria; en su ausencia se usa el conjunto BASE5 como respaldo mínimo.
if "TRAINING_COLS" in globals():
    TRAIN_COLS = [c for c in TRAINING_COLS if c in dfW.columns]
else:
    TRAIN_COLS = [c for c in ["home_work_ratio", "radius_gyration_km", "entropy_spatial", "n_moves", "unique_locations"] if c in dfW.columns]

if len(TRAIN_COLS) < 3:
    raise ValueError(f"Insuficientes variables de entrenamiento para evaluar estabilidad: {TRAIN_COLS}")

dfX = dfW[[USER_COL, "cluster_id"] + TRAIN_COLS].copy()
for c in TRAIN_COLS:
    dfX[c] = pd.to_numeric(dfX[c], errors="coerce").fillna(0.0).astype(np.float32)

users = dfX[USER_COL].astype("string[python]").to_numpy()
y_base = pd.to_numeric(dfX["cluster_id"], errors="coerce").fillna(0).astype(np.int32).to_numpy()
X = dfX[TRAIN_COLS].to_numpy(dtype=np.float32, copy=True)

# El escalador se reajusta sobre el universo completo, replicando la Sección 9.
# Las submuestras se toman del espacio ya estandarizado y no se reescalan: así
# la variación medida proviene del submuestreo y no de un cambio de escala.
scaler = StandardScaler(with_mean=True, with_std=True)
Xz_full = scaler.fit_transform(X).astype(np.float32, copy=False)

n = Xz_full.shape[0]
n_s = int(max(10, np.floor(SUBSAMPLE_FRAC * n)))

print(f"[13.1] candidatos n={n:,} | submuestra n_s={n_s:,} | variables={len(TRAIN_COLS)}")


def cluster_ward(Xz: np.ndarray, k: int) -> np.ndarray:
    """Reejecuta el enlace de Ward y corta a k conglomerados."""
    Z = linkage(Xz, method="ward")
    return fcluster(Z, t=int(k), criterion="maxclust").astype(np.int32)


rows = []
for seed in ROB_SEEDS:
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(n, size=n_s, replace=False)

    Xs = Xz_full[idx]
    yb = y_base[idx]           # asignación de referencia, restringida a la submuestra
    y_new = cluster_ward(Xs, K_BASELINE)   # asignación reejecutada

    ari = float(adjusted_rand_score(yb, y_new))
    jac = jaccard_matched(yb, y_new, K_BASELINE)

    rows.append(
        {
            "seed": int(seed),
            "n_s": int(n_s),
            "ari": ari,
            "jaccard_mean_weighted": float(jac["mean_jaccard_weighted"]),
        }
    )

df_stab = pd.DataFrame(rows)

# Artefacto de Nivel 2: agregados derivados de la fuente telco.
stab_path = paths["tables_dir"] / f"clustering_stability_{FEATURE_SET}.parquet"
df_stab.to_parquet(stab_path, index=False)

print("[13.1] guardado:", stab_path.name)
print(df_stab[["ari", "jaccard_mean_weighted"]].describe().to_string())

_ari_med = float(df_stab["ari"].median())
print(
    f"[13.1] ARI mediano={_ari_med:.4f} "
    f"[p25={float(df_stab['ari'].quantile(0.25)):.4f}, "
    f"p75={float(df_stab['ari'].quantile(0.75)):.4f}] | "
    f"Jaccard mediano={float(df_stab['jaccard_mean_weighted'].median()):.4f}"
)
print(
    "[13.1] Lectura: un ARI alto indica que la estructura de conglomerados se "
    "reproduce al variar la composición de la muestra. La dispersión entre "
    "réplicas es en sí informativa: refleja cuánto depende la partición de "
    "casos particulares."
)

# Figuras
fig = plt.figure(figsize=(6.5, 3.5))
ax = plt.gca()
ax.hist(df_stab["ari"].dropna().to_numpy(), bins=10)
ax.set_title(f"Clustering stability — ARI (subsampling) — {FEATURE_SET}")
ax.set_xlabel("ARI vs partición de referencia")
ax.set_ylabel("réplicas")
fig.tight_layout()
fig_path_ari = paths["figures_dir"] / f"stability_ari_hist_{FEATURE_SET}.png"
fig.savefig(fig_path_ari, dpi=200)
plt.close(fig)

fig = plt.figure(figsize=(6.5, 3.5))
ax = plt.gca()
ax.hist(df_stab["jaccard_mean_weighted"].dropna().to_numpy(), bins=10)
ax.set_title(f"Clustering stability — Jaccard (matched) — {FEATURE_SET}")
ax.set_xlabel("Jaccard medio ponderado")
ax.set_ylabel("réplicas")
fig.tight_layout()
fig_path_j = paths["figures_dir"] / f"stability_jaccard_hist_{FEATURE_SET}.png"
fig.savefig(fig_path_j, dpi=200)
plt.close(fig)

log_jsonl(
    ROB_LOG,
    {
        "event": "clustering_stability_done",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "n_workers": int(n),
        "n_subsample": int(n_s),
        "n_seeds": int(len(ROB_SEEDS)),
        "train_cols": TRAIN_COLS,
        "ari_median": _ari_med,
        "ari_p25": float(df_stab["ari"].quantile(0.25)),
        "ari_p75": float(df_stab["ari"].quantile(0.75)),
        "jaccard_median": float(df_stab["jaccard_mean_weighted"].median()),
        "table_path": str(stab_path),
    },
)

del dfW, dfX, X, Xz_full, _ari_med
_ = gc.collect()
print("[13.1] OK")

In [ ]:
# ============================================================
# 13.2 — Sensibilidad a hiperparámetros
#   13.2.1 tau_remote      : efecto sobre las etiquetas previas y los candidatos
#   13.2.2 regla móvil     : efecto sobre el conjunto marcado
#   13.2.3 gatekeeper      : efecto sobre el universo elegible
#   13.2.4 alpha           : efecto sobre las anclas residencial y laboral
# ============================================================

# ----------------------------
# 13.2.1 Sensibilidad a tau_remote
# ----------------------------
# Se recalculan las etiquetas previas de la Sección 7.2 bajo cada valor de la
# rejilla y se compara el conjunto de candidatos al clustering con el de la
# configuración documentada. El Jaccard mide solapamiento de conjuntos de
# usuarios, no concordancia de etiquetas: un valor alto indica que el umbral
# apenas redefine quién entra al agrupamiento.
dfU = pd.read_parquet(universe_path)

reqU = {USER_COL, "work_missing", "is_mobile_worker", "home_work_ratio"}
missU = reqU - set(dfU.columns)
if missU:
    raise ValueError(f"universe_full.parquet no tiene las columnas: {missU}")

dfU[USER_COL] = dfU[USER_COL].astype("string[python]")
dfU["work_missing"] = pd.to_numeric(dfU["work_missing"], errors="coerce").fillna(1).astype(np.int8)
dfU["is_mobile_worker"] = pd.to_numeric(dfU["is_mobile_worker"], errors="coerce").fillna(0).astype(np.int8)
dfU["home_work_ratio"] = pd.to_numeric(dfU["home_work_ratio"], errors="coerce").fillna(0).astype(np.float32)

baseline_tau = float(cfg.tau_remote)


def prelabel_from_tau(df: pd.DataFrame, tau: float) -> pd.Series:
    """Replica el etiquetado previo de la Sección 7.2 bajo un tau alternativo."""
    cond_mobile_raw = (df["is_mobile_worker"] == 1)
    cond_mobile = cond_mobile_raw & (df["home_work_ratio"] < tau)
    cond_office = (~cond_mobile) & (df["work_missing"] == 0)
    cond_remote = (~cond_mobile) & (df["work_missing"] == 1) & (df["home_work_ratio"] >= tau)

    pre = np.select(
        [cond_mobile, cond_office, cond_remote],
        ["MOBILE_WORKER", "WORKER_OFFICE", "WORKER_POTENTIAL_REMOTE"],
        default="NO_WORKER",
    )
    return pd.Series(pre, index=df.index, dtype="string[python]")


rows = []
cand_base = None

for tau in TAU_REMOTE_GRID:
    pre = prelabel_from_tau(dfU, float(tau))
    cand = dfU.loc[pre.isin(["WORKER_OFFICE", "WORKER_POTENTIAL_REMOTE"]), USER_COL]
    cand_set = set(cand.tolist())

    if np.isclose(float(tau), baseline_tau):
        cand_base = cand_set

    rows.append(
        {
            "tau_remote": float(tau),
            "n_total": int(len(dfU)),
            "n_candidates": int(len(cand_set)),
            "n_office": int((pre == "WORKER_OFFICE").sum()),
            "n_potential_remote": int((pre == "WORKER_POTENTIAL_REMOTE").sum()),
            "n_mobile": int((pre == "MOBILE_WORKER").sum()),
            "n_no_worker": int((pre == "NO_WORKER").sum()),
        }
    )

df_tau = pd.DataFrame(rows)

if cand_base is not None:
    jacc = []
    for tau in df_tau["tau_remote"].tolist():
        pre = prelabel_from_tau(dfU, float(tau))
        cand = set(dfU.loc[pre.isin(["WORKER_OFFICE", "WORKER_POTENTIAL_REMOTE"]), USER_COL].tolist())
        inter = len(cand & cand_base)
        union = len(cand | cand_base)
        jacc.append(float(inter / union) if union else float("nan"))
    df_tau["candidates_jaccard_vs_baseline"] = jacc

tau_path = paths["tables_dir"] / f"sensitivity_tau_remote_{FEATURE_SET}.parquet"
df_tau.to_parquet(tau_path, index=False)

fig = plt.figure(figsize=(7, 3.5))
ax = plt.gca()
x = np.arange(len(df_tau), dtype=np.int32)
ax.plot(x, df_tau["n_candidates"].to_numpy(), marker="o")
ax.set_title(f"Sensibilidad tau_remote — candidatos — {FEATURE_SET}")
ax.set_xlabel("tau_remote")
ax.set_ylabel("n candidatos")
ax.set_xticks(x)
ax.set_xticklabels([str(t) for t in df_tau["tau_remote"].tolist()])
fig.tight_layout()
tau_fig = paths["figures_dir"] / f"sens_tau_remote_candidates_{FEATURE_SET}.png"
fig.savefig(tau_fig, dpi=200)
plt.close(fig)

log_jsonl(
    ROB_LOG,
    {
        "event": "sens_tau_remote_done",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "baseline_tau": baseline_tau,
        "results": df_tau.to_dict(orient="records"),
        "table_path": str(tau_path),
    },
)

print("[13.2.1] sensibilidad a tau_remote:")
print(df_tau.to_string(index=False))

# ----------------------------
# 13.2.2 Sensibilidad de la regla Mobile-for-Work
# ----------------------------
# Se recorre la rejilla completa percentil × tau_unique y se compara el conjunto
# marcado contra el de la configuración documentada.
#
# Esta tabla es la que sustenta la desviación instrumental declarada en la
# Sección 6.2: si para tau_unique >= 10 la regla no identifica ningún caso en
# ningún percentil, el valor de marco resulta inaplicable sobre la densidad
# observacional de la fuente, y el valor efectivo no es una elección de
# conveniencia sino la única configuración operativa.
df_works = pd.read_parquet(works_path)
df_mob = pd.read_parquet(mob_path)
df_flags_base = pd.read_parquet(flags_path)

for df_ in (df_works, df_mob, df_flags_base):
    df_[USER_COL] = df_[USER_COL].astype("string[python]")

df_works["work_missing"] = pd.to_numeric(df_works["work_missing"], errors="coerce").fillna(1).astype(np.int8)
df_mob["radius_gyration_km"] = pd.to_numeric(df_mob["radius_gyration_km"], errors="coerce").fillna(0).astype(np.float32)
df_mob["unique_locations"] = pd.to_numeric(df_mob["unique_locations"], errors="coerce").fillna(0).astype(np.int32)

base_mobile_set = set(df_flags_base.loc[df_flags_base["is_mobile_worker"] == 1, USER_COL].tolist())


def compute_mobile_set(pct: float, tau_uniq: int) -> tuple[float, set]:
    """Recalcula el umbral de Rg y el conjunto marcado bajo parámetros alternativos."""
    df = df_works[[USER_COL, "work_missing"]].merge(
        df_mob[[USER_COL, "radius_gyration_km", "unique_locations"]],
        on=USER_COL, how="left"
    )
    no_work_rg = df.loc[df["work_missing"] == 1, "radius_gyration_km"].dropna()
    thr = float(no_work_rg.quantile(float(pct))) if len(no_work_rg) else float("inf")

    is_mobile = (
        (df["work_missing"] == 1) &
        (df["radius_gyration_km"] >= thr) &
        (df["unique_locations"] >= int(tau_uniq))
    )
    return thr, set(df.loc[is_mobile, USER_COL].tolist())


rows = []
for pct in MOBILE_PCT_GRID:
    for tau_uniq in MOBILE_TAU_UNIQ_GRID:
        thr, s = compute_mobile_set(float(pct), int(tau_uniq))
        inter = len(s & base_mobile_set)
        union = len(s | base_mobile_set)
        jac = float(inter / union) if union else float("nan")
        rows.append(
            {
                "rg_percentile": float(pct),
                "tau_unique": int(tau_uniq),
                "rg_threshold": float(thr),
                "n_mobile": int(len(s)),
                "jaccard_vs_baseline": jac,
            }
        )

df_mob_sens = pd.DataFrame(rows)
mob_sens_path = paths["tables_dir"] / f"sensitivity_mobile_{FEATURE_SET}.parquet"
df_mob_sens.to_parquet(mob_sens_path, index=False)

pcts = sorted(df_mob_sens["rg_percentile"].unique().tolist())
taus = sorted(df_mob_sens["tau_unique"].unique().tolist())
mat = np.full((len(pcts), len(taus)), np.nan, dtype=np.float32)
for i, p_ in enumerate(pcts):
    for j, t_ in enumerate(taus):
        v = df_mob_sens.loc[(df_mob_sens["rg_percentile"] == p_) & (df_mob_sens["tau_unique"] == t_), "jaccard_vs_baseline"]
        if len(v):
            mat[i, j] = float(v.iloc[0])

fig = plt.figure(figsize=(7, 3.5))
ax = plt.gca()
im = ax.imshow(mat, aspect="auto")
ax.set_title(f"Mobile sensitivity — Jaccard vs baseline — {FEATURE_SET}")
ax.set_xlabel("tau_unique")
ax.set_ylabel("rg_percentile")
ax.set_xticks(np.arange(len(taus), dtype=np.int32))
ax.set_xticklabels([str(t_) for t_ in taus])
ax.set_yticks(np.arange(len(pcts), dtype=np.int32))
ax.set_yticklabels([str(p_) for p_ in pcts])
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
mob_fig = paths["figures_dir"] / f"sens_mobile_jaccard_{FEATURE_SET}.png"
fig.savefig(mob_fig, dpi=200)
plt.close(fig)

log_jsonl(
    ROB_LOG,
    {
        "event": "sens_mobile_done",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "baseline_n_mobile": int(len(base_mobile_set)),
        "results": df_mob_sens.to_dict(orient="records"),
        "table_path": str(mob_sens_path),
    },
)

print("[13.2.2] sensibilidad de la regla móvil:")
print(df_mob_sens.sort_values(["rg_percentile", "tau_unique"]).to_string(index=False))

# Diagnóstico: valores de tau_unique bajo los cuales la regla no identifica caso alguno
_null_taus = sorted(
    df_mob_sens.groupby("tau_unique")["n_mobile"].max().loc[lambda s: s == 0].index.tolist()
)
if _null_taus:
    print(
        f"[13.2.2] Con tau_unique en {_null_taus} la regla no identifica ningún "
        "caso en ninguno de los percentiles evaluados. Es la evidencia que "
        "sustenta la desviación instrumental declarada en la Sección 6.2."
    )
del _null_taus

# ----------------------------
# 13.2.3 Sensibilidad del gatekeeper (variación de un umbral a la vez)
# ----------------------------
# Se varía cada uno de los cuatro umbrales manteniendo los otros tres en su
# valor documentado, y se mide el efecto sobre el universo elegible.
#
# Es el resultado más consecuente de la sección: si un umbral desplaza el
# universo en un orden de magnitud mientras los otros apenas lo mueven, la
# elegibilidad depende críticamente de esa decisión, y el manuscrito debe
# declararla como tal.
gk_stats = pd.read_parquet(gk_stats_path)
gk_stats[USER_COL] = gk_stats[USER_COL].astype("string[python]")

D_min = int(cfg.gk_min_days_with_stays)
D_wdmin = int(cfg.gk_min_weekdays_with_stays)
H_min = float(cfg.gk_min_total_hours)
S_min = int(cfg.gk_min_span_days)

base_mask = (
    (gk_stats["n_days_with_stays"] >= D_min) &
    (gk_stats["n_weekdays_with_stays"] >= D_wdmin) &
    (gk_stats["total_stay_hours"] >= H_min) &
    (gk_stats["span_days"] >= S_min)
)
base_elig = set(gk_stats.loc[base_mask, USER_COL].tolist())
n_users_gk = int(len(gk_stats))

tests = []
for d in [max(1, D_min - 2), D_min, D_min + 2]:
    tests.append(("D_min", {"D_min": int(d), "D_wdmin": D_wdmin, "H_min": H_min, "S_min": S_min}))
for wd in [max(1, D_wdmin - 1), D_wdmin, D_wdmin + 1]:
    tests.append(("D_wdmin", {"D_min": D_min, "D_wdmin": int(wd), "H_min": H_min, "S_min": S_min}))
for h in [max(1.0, H_min - 5.0), H_min, H_min + 5.0]:
    tests.append(("H_min", {"D_min": D_min, "D_wdmin": D_wdmin, "H_min": float(h), "S_min": S_min}))
for s in [max(1, S_min - 2), S_min, S_min + 2]:
    tests.append(("S_min", {"D_min": D_min, "D_wdmin": D_wdmin, "H_min": H_min, "S_min": int(s)}))

rows = []
for name, params in tests:
    mask = (
        (gk_stats["n_days_with_stays"] >= params["D_min"]) &
        (gk_stats["n_weekdays_with_stays"] >= params["D_wdmin"]) &
        (gk_stats["total_stay_hours"] >= params["H_min"]) &
        (gk_stats["span_days"] >= params["S_min"])
    )
    elig = set(gk_stats.loc[mask, USER_COL].tolist())
    inter = len(elig & base_elig)
    union = len(elig | base_elig)
    jac = float(inter / union) if union else float("nan")

    rows.append(
        {
            "vary": name,
            **params,
            "n_eligible": int(len(elig)),
            "retention": float(len(elig) / max(n_users_gk, 1)),
            "jaccard_vs_baseline": jac,
        }
    )

df_gk_sens = pd.DataFrame(rows)
gk_sens_path = paths["tables_dir"] / f"sensitivity_gatekeeper_{FEATURE_SET}.parquet"
df_gk_sens.to_parquet(gk_sens_path, index=False)

fig = plt.figure(figsize=(9, 3.5))
ax = plt.gca()
x = np.arange(len(df_gk_sens), dtype=np.int32)
ax.bar(x, df_gk_sens["retention"].to_numpy(dtype=np.float32, copy=False))
ax.set_title(f"Gatekeeper sensitivity — retención — {FEATURE_SET}")
ax.set_ylabel("retención")
ax.set_xticks(x)
ax.set_xticklabels(df_gk_sens["vary"].astype(str).to_numpy(), rotation=30, ha="right")
fig.tight_layout()
gk_fig = paths["figures_dir"] / f"sens_gatekeeper_retention_{FEATURE_SET}.png"
fig.savefig(gk_fig, dpi=200)
plt.close(fig)

log_jsonl(
    ROB_LOG,
    {
        "event": "sens_gatekeeper_done",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "baseline_params": {"D_min": D_min, "D_wdmin": D_wdmin, "H_min": H_min, "S_min": S_min},
        "baseline_n_eligible": int(len(base_elig)),
        "results": df_gk_sens.to_dict(orient="records"),
        "table_path": str(gk_sens_path),
    },
)

print("[13.2.3] sensibilidad del gatekeeper:")
print(df_gk_sens.sort_values(["vary", "D_min", "D_wdmin", "H_min", "S_min"]).to_string(index=False))

# Diagnóstico: qué umbral domina el tamaño del universo
_span = (
    df_gk_sens.groupby("vary")["n_eligible"]
    .agg(lambda s: int(s.max() - s.min()))
    .sort_values(ascending=False)
)
print("\n[13.2.3] amplitud del universo elegible por umbral variado:")
for _var, _amp in _span.items():
    print(f"       {_var:<10} {int(_amp):>6,} usuarios entre el mínimo y el máximo")

_dominant = str(_span.index[0])
print(
    f"[13.2.3] El umbral '{_dominant}' domina el tamaño del universo. La "
    "insuficiencia longitudinal de la fuente es estructural; el tamaño exacto "
    "de la cohorte depende de esta decisión metodológica y debe declararse "
    "como tal."
)
del _span, _dominant

# ----------------------------
# 13.2.4 Sensibilidad a alpha (ponderación del anclaje residencial)
# ----------------------------
# alpha pondera la ventana nocturna frente a la matinal en la Sección 4. Se
# recalculan ambas anclas bajo cada valor de la rejilla y se mide la proporción
# de usuarios cuya ancla coincide con la de la configuración documentada.
#
# La comparación entre ambas tasas es el punto: si el anclaje residencial es
# robusto a alpha y el laboral no, la fragilidad del prototipo no está en la
# ponderación sino en la resolución del lugar de trabajo. Esa asimetría es la
# que el manuscrito declara en la Sección 5.
stays = pd.read_parquet(stays_gk_path)
stays[USER_COL] = stays[USER_COL].astype("string[python]")

homes_base = pd.read_parquet(homes_path)[[USER_COL, "home_h3", "home_missing"]].copy()
works_base = pd.read_parquet(works_path)[[USER_COL, "work_h3", "work_missing"]].copy()
homes_base[USER_COL] = homes_base[USER_COL].astype("string[python]")
works_base[USER_COL] = works_base[USER_COL].astype("string[python]")


def identify_home_alpha(stays: pd.DataFrame, alpha: float) -> pd.DataFrame:
    """Replica el anclaje residencial de la Sección 4 bajo un alpha alternativo."""
    user_col = USER_COL
    h3_col = "h3_id"
    start_col, end_col = "start_ts", "end_ts"

    (ev_start, ev_end) = cfg.home_win_primary
    (mo_start, mo_end) = cfg.home_win_secondary
    min_days = int(cfg.home_min_days)
    min_ratio = float(cfg.home_min_ratio)

    s = stays[[user_col, h3_col, start_col, end_col]].copy()

    start_ts = s[start_col].to_numpy(dtype="datetime64[ns]", copy=False)
    end_ts = s[end_col].to_numpy(dtype="datetime64[ns]", copy=False)
    dt_start = pd.DatetimeIndex(start_ts)
    dt_end = pd.DatetimeIndex(end_ts)

    start_sec = (dt_start.hour.astype(np.int32) * 3600 + dt_start.minute.astype(np.int32) * 60 + dt_start.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = (dt_end.hour.astype(np.int32) * 3600 + dt_end.minute.astype(np.int32) * 60 + dt_end.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = np.maximum(end_sec, start_sec)

    ev0 = ev_start.hour * 3600 + ev_start.minute * 60 + ev_start.second
    ev1 = ev_end.hour * 3600 + ev_end.minute * 60 + ev_end.second
    mo0 = mo_start.hour * 3600 + mo_start.minute * 60 + mo_start.second
    mo1 = mo_end.hour * 3600 + mo_end.minute * 60 + mo_end.second

    ev_overlap = np.maximum(0, np.minimum(end_sec, ev1) - np.maximum(start_sec, ev0)).astype(np.float32)
    mo_overlap = np.maximum(0, np.minimum(end_sec, mo1) - np.maximum(start_sec, mo0)).astype(np.float32)

    wsec = (float(alpha) * ev_overlap + (1.0 - float(alpha)) * mo_overlap).astype(np.float32)
    mask = wsec > 0

    s = s.loc[mask, [user_col, h3_col, start_col]].copy()
    s["home_wsec"] = wsec[mask]
    s["date_int"] = s[start_col].to_numpy(dtype="datetime64[ns]", copy=False).astype("datetime64[D]").astype(np.int32, copy=False)

    cand = (
        s.groupby([user_col, h3_col], observed=True, sort=False)
        .agg(total=("home_wsec", "sum"), days=("date_int", pd.Series.nunique))
        .reset_index()
    )
    cand["u_total"] = cand.groupby(user_col, observed=True, sort=False)["total"].transform("sum")
    cand["dom"] = (cand["total"] / cand["u_total"]).astype(np.float32)
    cand.sort_values([user_col, "total"], ascending=[True, False], kind="mergesort", inplace=True)

    best = cand.drop_duplicates(subset=[user_col], keep="first").copy()
    best["valid"] = (best["days"] >= min_days) & (best["dom"] >= min_ratio)

    out = best[[user_col, h3_col, "valid"]].copy()
    out.rename(columns={h3_col: "home_h3"}, inplace=True)
    out["home_missing"] = np.where(out["valid"] == True, 0, 1).astype(np.int8)
    out.loc[out["home_missing"] == 1, "home_h3"] = pd.NA
    return out[[user_col, "home_h3", "home_missing"]]


def identify_work_from_homes(stays: pd.DataFrame, homes_df: pd.DataFrame) -> pd.DataFrame:
    """Replica el anclaje laboral de la Sección 5 sobre un conjunto de hogares dado."""
    user_col = USER_COL
    h3_col = "h3_id"
    start_col, end_col = "start_ts", "end_ts"

    (wk_start, wk_end) = cfg.work_win
    wk0 = wk_start.hour * 3600 + wk_start.minute * 60 + wk_start.second
    wk1 = wk_end.hour * 3600 + wk_end.minute * 60 + wk_end.second
    min_days = int(cfg.work_min_days)
    min_ratio = float(cfg.work_min_ratio)

    valid_h = homes_df.loc[homes_df["home_missing"] == 0, [user_col, "home_h3"]].copy()
    if valid_h.empty:
        out = homes_df[[user_col]].copy()
        out["work_h3"] = pd.NA
        out["work_missing"] = np.int8(1)
        return out[[user_col, "work_h3", "work_missing"]]

    s = stays[[user_col, h3_col, start_col, end_col]].merge(valid_h, on=user_col, how="inner")

    start_ts = s[start_col].to_numpy(dtype="datetime64[ns]", copy=False)
    end_ts = s[end_col].to_numpy(dtype="datetime64[ns]", copy=False)
    dt_start = pd.DatetimeIndex(start_ts)
    dt_end = pd.DatetimeIndex(end_ts)

    dow = dt_start.dayofweek.astype(np.int8, copy=False)
    is_wd = (dow < 5)

    start_sec = (dt_start.hour.astype(np.int32) * 3600 + dt_start.minute.astype(np.int32) * 60 + dt_start.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = (dt_end.hour.astype(np.int32) * 3600 + dt_end.minute.astype(np.int32) * 60 + dt_end.second.astype(np.int32)).astype(np.int32, copy=False)
    end_sec = np.maximum(end_sec, start_sec)

    overlap = np.maximum(0, np.minimum(end_sec, wk1) - np.maximum(start_sec, wk0)).astype(np.float32)
    mask = is_wd & (overlap > 0) & (s[h3_col] != s["home_h3"])

    if not mask.any():
        out = homes_df[[user_col]].copy()
        out["work_h3"] = pd.NA
        out["work_missing"] = np.int8(1)
        return out[[user_col, "work_h3", "work_missing"]]

    sc = s.loc[mask, [user_col, h3_col, start_col]].copy()
    sc["work_wsec"] = overlap[mask]
    sc["date_int"] = sc[start_col].to_numpy(dtype="datetime64[ns]", copy=False).astype("datetime64[D]").astype(np.int32, copy=False)

    cand = (
        sc.groupby([user_col, h3_col], observed=True, sort=False)
        .agg(total=("work_wsec", "sum"), days=("date_int", pd.Series.nunique))
        .reset_index()
    )
    cand["u_total"] = cand.groupby(user_col, observed=True, sort=False)["total"].transform("sum")
    cand["dom"] = (cand["total"] / cand["u_total"]).astype(np.float32)
    cand.sort_values([user_col, "total"], ascending=[True, False], kind="mergesort", inplace=True)

    best = cand.drop_duplicates(subset=[user_col], keep="first").copy()
    best["valid"] = (best["days"] >= min_days) & (best["dom"] >= min_ratio)

    out = homes_df[[user_col]].copy()
    out = out.merge(best[[user_col, h3_col, "valid"]], on=user_col, how="left")
    out.rename(columns={h3_col: "work_h3"}, inplace=True)
    out["work_missing"] = np.where(out["valid"] == True, 0, 1).astype(np.int8)
    out.loc[out["work_missing"] == 1, "work_h3"] = pd.NA
    return out[[user_col, "work_h3", "work_missing"]]


rows = []
hb = homes_base.set_index(USER_COL)["home_h3"]
wb = works_base.set_index(USER_COL)["work_h3"]

for a in ALPHA_GRID:
    h = identify_home_alpha(stays, float(a))
    w = identify_work_from_homes(stays, h)

    h_map = h.set_index(USER_COL)["home_h3"]
    w_map = w.set_index(USER_COL)["work_h3"]

    common = hb.index.intersection(h_map.index)
    h_eq = (hb.loc[common].astype("string[python]") == h_map.loc[common].astype("string[python]")).fillna(False)
    home_same_rate = float(h_eq.mean()) if len(h_eq) else float("nan")

    common2 = wb.index.intersection(w_map.index)
    w_eq = (wb.loc[common2].astype("string[python]") == w_map.loc[common2].astype("string[python]")).fillna(False)
    work_same_rate = float(w_eq.mean()) if len(w_eq) else float("nan")

    rows.append(
        {
            "alpha": float(a),
            "home_missing_rate": float(h["home_missing"].mean()) if len(h) else float("nan"),
            "work_missing_rate": float(w["work_missing"].mean()) if len(w) else float("nan"),
            "home_same_rate_vs_baseline": home_same_rate,
            "work_same_rate_vs_baseline": work_same_rate,
        }
    )

df_alpha = pd.DataFrame(rows)
alpha_path = paths["tables_dir"] / f"sensitivity_alpha_{FEATURE_SET}.parquet"
df_alpha.to_parquet(alpha_path, index=False)

fig = plt.figure(figsize=(7, 3.5))
ax = plt.gca()
x = np.arange(len(df_alpha), dtype=np.int32)
ax.plot(x, df_alpha["home_same_rate_vs_baseline"].to_numpy(), marker="o", label="ancla residencial")
ax.plot(x, df_alpha["work_same_rate_vs_baseline"].to_numpy(), marker="o", label="ancla laboral")
ax.set_title(f"Sensibilidad alpha — coincidencia de anclas — {FEATURE_SET}")
ax.set_ylabel("tasa de coincidencia")
ax.set_xlabel("alpha")
ax.set_xticks(x)
ax.set_xticklabels([str(a) for a in df_alpha["alpha"].tolist()])
ax.legend()
fig.tight_layout()
alpha_fig = paths["figures_dir"] / f"sens_alpha_same_rate_{FEATURE_SET}.png"
fig.savefig(alpha_fig, dpi=200)
plt.close(fig)

log_jsonl(
    ROB_LOG,
    {
        "event": "sens_alpha_done",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "alpha_grid": ALPHA_GRID,
        "results": df_alpha.to_dict(orient="records"),
        "table_path": str(alpha_path),
    },
)

print("[13.2.4] sensibilidad a alpha:")
print(df_alpha.to_string(index=False))

_h_min = float(df_alpha["home_same_rate_vs_baseline"].min())
_w_min = float(df_alpha["work_same_rate_vs_baseline"].min())
if np.isfinite(_h_min) and np.isfinite(_w_min) and _h_min > _w_min:
    print(
        f"[13.2.4] El anclaje residencial es robusto a alpha (coincidencia "
        f"mínima {_h_min:.1%}); el laboral no ({_w_min:.1%}). La fragilidad "
        "reside en la resolución del lugar de trabajo, no en la ponderación de "
        "las ventanas."
    )
del _h_min, _w_min

del dfU, df_works, df_mob, df_flags_base, gk_stats, stays, homes_base, works_base
_ = gc.collect()
print("[13.2] OK")

In [ ]:
# ============================================================
# 13.3 — Sensibilidad al radio de los buffers POI
# ============================================================
# Se compara el contexto territorial medido a 300 m y a 800 m. Una correlación
# alta indica que el radio no altera el ordenamiento relativo de los usuarios:
# quien tiene más POIs cerca los tiene también a mayor distancia. No implica que
# las magnitudes sean equivalentes ni que el radio sea indiferente para el
# significado sustantivo de la variable.
#
# Advertencia: los usuarios sin ancla laboral tienen cero en las variables de
# trabajo para ambos radios, lo que infla artificialmente la correlación de ese
# par. La cobertura de anclas laborales reportada en la Sección 5 acota su
# interpretación.
if aug_final_path is None:
    print("[13.3] producto con POIs no disponible: se omite la sensibilidad de radio.")
else:
    dfP = pd.read_parquet(aug_final_path)
    dfP[USER_COL] = dfP[USER_COL].astype("string[python]")
    dfP["final_segment"] = dfP["final_segment"].astype("string[python]")

    pairs = [
        ("home_cnt_total", "poi_home_r300m_cnt_total", "poi_home_r800m_cnt_total"),
        ("work_cnt_total", "poi_work_r300m_cnt_total", "poi_work_r800m_cnt_total"),
        ("home_density", "poi_home_r300m_density_km2", "poi_home_r800m_density_km2"),
        ("work_density", "poi_work_r300m_density_km2", "poi_work_r800m_density_km2"),
    ]
    pairs = [(nm, c1, c2) for nm, c1, c2 in pairs if c1 in dfP.columns and c2 in dfP.columns]

    if not pairs:
        print("[13.3] no se encontraron columnas POI de r300/r800: se omite.")
    else:
        rows = []
        for name, c1, c2 in pairs:
            x = pd.to_numeric(dfP[c1], errors="coerce").fillna(0).to_numpy(np.float64, copy=False)
            y = pd.to_numeric(dfP[c2], errors="coerce").fillna(0).to_numpy(np.float64, copy=False)
            corr = float(np.corrcoef(x, y)[0, 1]) if (np.std(x) > 0 and np.std(y) > 0) else float("nan")
            rows.append({"metric": name, "col_300": c1, "col_800": c2, "corr": corr})

        df_corr = pd.DataFrame(rows)

        cols_demo = [c for c in [
            "poi_home_r300m_cnt_total", "poi_home_r800m_cnt_total",
            "poi_work_r300m_cnt_total", "poi_work_r800m_cnt_total",
        ] if c in dfP.columns]

        med = (
            dfP.groupby("final_segment", observed=False)[cols_demo]
            .median(numeric_only=True)
            .reset_index()
        )

        # Persistencia en formato largo, con esquema y tipos declarados
        OUT_COLS = ["table", "final_segment", "key", "value"]

        corr_out = df_corr.assign(table="corr").rename(columns={"metric": "key", "corr": "value"}).copy()
        corr_out["table"] = corr_out["table"].astype("string[python]")
        corr_out["key"] = corr_out["key"].astype("string[python]")
        corr_out["value"] = pd.to_numeric(corr_out["value"], errors="coerce").astype(np.float32)
        corr_out["final_segment"] = pd.array([pd.NA] * len(corr_out), dtype="string[python]")
        corr_out = corr_out[OUT_COLS]

        med_long = med.melt(id_vars=["final_segment"], var_name="key", value_name="value").assign(table="medians")
        med_out = med_long.copy()
        med_out["table"] = med_out["table"].astype("string[python]")
        med_out["final_segment"] = med_out["final_segment"].astype("string[python]")
        med_out["key"] = med_out["key"].astype("string[python]")
        med_out["value"] = pd.to_numeric(med_out["value"], errors="coerce").astype(np.float32)
        med_out = med_out[OUT_COLS]

        out_long = pd.concat([corr_out, med_out], ignore_index=True)
        poi_sens_path = paths["tables_dir"] / f"poi_sensitivity_{FEATURE_SET}.parquet"
        out_long.to_parquet(poi_sens_path, index=False)

        if "poi_home_r300m_cnt_total" in dfP.columns and "poi_home_r800m_cnt_total" in dfP.columns:
            x = pd.to_numeric(dfP["poi_home_r300m_cnt_total"], errors="coerce").fillna(0).to_numpy()
            y = pd.to_numeric(dfP["poi_home_r800m_cnt_total"], errors="coerce").fillna(0).to_numpy()

            fig = plt.figure(figsize=(6.5, 4))
            ax = plt.gca()
            ax.scatter(x, y, s=8, alpha=0.6)
            ax.set_title(f"POI home cnt_total: r300 vs r800 — {FEATURE_SET}")
            ax.set_xlabel("poi_home_r300m_cnt_total")
            ax.set_ylabel("poi_home_r800m_cnt_total")
            fig.tight_layout()
            poi_fig = paths["figures_dir"] / f"poi_buffer_scatter_home_{FEATURE_SET}.png"
            fig.savefig(poi_fig, dpi=200)
            plt.close(fig)
        else:
            poi_fig = None

        log_jsonl(
            ROB_LOG,
            {
                "event": "sens_poi_done",
                "ts_utc": now_utc(),
                "feature_set": FEATURE_SET,
                "correlations": df_corr.to_dict(orient="records"),
                "table_path": str(poi_sens_path),
            },
        )

        print("[13.3] guardado:", poi_sens_path.name)
        print("[13.3] correlación entre radios:")
        print(df_corr.to_string(index=False))
        print("[13.3] medianas por segmento:")
        print(med.to_string(index=False))

    del dfP
    gc.collect()

print("[13.3] OK")

In [ ]:
# ============================================================
# 13.4 — Resumen de robustez
#   Consolida los resultados de 13.1 a 13.3 en una tabla compacta, para que la
#   auditoría de la corrida quede en un único artefacto legible.
# ============================================================
stab_path = paths["tables_dir"] / f"clustering_stability_{FEATURE_SET}.parquet"
tau_path = paths["tables_dir"] / f"sensitivity_tau_remote_{FEATURE_SET}.parquet"
mob_path2 = paths["tables_dir"] / f"sensitivity_mobile_{FEATURE_SET}.parquet"
gk_path2 = paths["tables_dir"] / f"sensitivity_gatekeeper_{FEATURE_SET}.parquet"
alpha_path2 = paths["tables_dir"] / f"sensitivity_alpha_{FEATURE_SET}.parquet"
poi_path2 = paths["tables_dir"] / f"poi_sensitivity_{FEATURE_SET}.parquet"

dfs = {}
for name, p in [
    ("stability", stab_path),
    ("tau_remote", tau_path),
    ("mobile", mob_path2),
    ("gatekeeper", gk_path2),
    ("alpha", alpha_path2),
]:
    if p.exists():
        dfs[name] = pd.read_parquet(p)

rows = []

if "stability" in dfs:
    d = dfs["stability"]
    rows.append({"table": "clustering_stability", "key": "ari_median", "value": float(d["ari"].median())})
    rows.append({"table": "clustering_stability", "key": "ari_p25", "value": float(d["ari"].quantile(0.25))})
    rows.append({"table": "clustering_stability", "key": "ari_p75", "value": float(d["ari"].quantile(0.75))})
    rows.append({"table": "clustering_stability", "key": "jaccard_median", "value": float(d["jaccard_mean_weighted"].median())})

if "tau_remote" in dfs:
    d = dfs["tau_remote"]
    base_tau = float(cfg.tau_remote)
    b = d.loc[np.isclose(d["tau_remote"], base_tau)]
    if len(b):
        rows.append({"table": "tau_remote", "key": "baseline_tau", "value": base_tau})
        rows.append({"table": "tau_remote", "key": "baseline_n_candidates", "value": float(b["n_candidates"].iloc[0])})

if "mobile" in dfs:
    d = dfs["mobile"]
    d2 = d.sort_values("jaccard_vs_baseline", ascending=False, kind="mergesort").head(1)
    if len(d2):
        rows.append({"table": "mobile", "key": "best_jaccard_vs_baseline", "value": float(d2["jaccard_vs_baseline"].iloc[0])})
        rows.append({"table": "mobile", "key": "best_n_mobile", "value": float(d2["n_mobile"].iloc[0])})

if "gatekeeper" in dfs:
    d = dfs["gatekeeper"]
    D_min = int(cfg.gk_min_days_with_stays)
    D_wdmin = int(cfg.gk_min_weekdays_with_stays)
    H_min = float(cfg.gk_min_total_hours)
    S_min = int(cfg.gk_min_span_days)
    b = d.loc[
        (d["D_min"] == D_min) & (d["D_wdmin"] == D_wdmin)
        & (np.isclose(d["H_min"], H_min)) & (d["S_min"] == S_min)
    ]
    if len(b):
        rows.append({"table": "gatekeeper", "key": "baseline_retention", "value": float(b["retention"].iloc[0])})
        rows.append({"table": "gatekeeper", "key": "baseline_n_eligible", "value": float(b["n_eligible"].iloc[0])})

if "alpha" in dfs:
    d = dfs["alpha"]
    for a in d["alpha"].tolist():
        r = d.loc[np.isclose(d["alpha"], float(a))].iloc[0]
        rows.append({"table": "alpha", "key": f"home_same_rate@{a}", "value": float(r["home_same_rate_vs_baseline"])})
        rows.append({"table": "alpha", "key": f"work_same_rate@{a}", "value": float(r["work_same_rate_vs_baseline"])})

if poi_path2.exists():
    df_poi_s = pd.read_parquet(poi_path2)
    corr = df_poi_s[df_poi_s["table"] == "corr"].copy()
    for _, r in corr.iterrows():
        rows.append({"table": "poi", "key": f"corr_{r['key']}", "value": float(r["value"])})

df_snap = pd.DataFrame(rows)
snap_path = paths["tables_dir"] / f"robustness_snapshot_{FEATURE_SET}.parquet"
df_snap.to_parquet(snap_path, index=False)

log_jsonl(
    ROB_LOG,
    {
        "event": "robustness_snapshot_saved",
        "ts_utc": now_utc(),
        "feature_set": FEATURE_SET,
        "path": str(snap_path),
        "n_rows": int(len(df_snap)),
        "snapshot": df_snap.to_dict(orient="records"),
    },
)

print("[13.4] guardado:", snap_path.name)
print("[13.4] resumen de robustez:")
for _tbl in df_snap["table"].unique():
    print(f"\n  {_tbl}")
    _sub = df_snap[df_snap["table"] == _tbl]
    for _, _r in _sub.iterrows():
        print(f"       {str(_r['key']):<32} {float(_r['value']):>12.4f}")

del _tbl, _sub
_ = gc.collect()

print("\n[Section 13] OK — auditoría de robustez completa.")